<a href="https://colab.research.google.com/github/ProjWashuRyoko-pixel/Ryoko-Seven-v2/blob/main/Ryoko_Seven_v2_GI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""

RyokoSeven — pip package setup

================================

pip install ryoko

ryoko --domain biology --therapy sotorasib

ryoko --domain os --target /path/to/game

ryoko --domain physics

ryoko --demo

"""


from setuptools import setup, find_packages


setup(

    name         = "ryokoseven",

    version      = "0.1.0",

    description  = "Pattern-optimized convergence engine. K = φ·π·e = 13.8176.",

    long_description = open("RYOKOSEVEN_OS_README.md").read()

        if __import__("pathlib").Path("RYOKOSEVEN_OS_README.md").exists()

        else "RyokoSeven convergence engine.",

    long_description_content_type = "text/markdown",

    author       = "WHUCM Collaboration",

    license      = "MIT",

    python_requires = ">=3.9",


    # Package layout

    py_modules = [

        "ryoko",               # unified entry point

        "ryoko_scanner",       # OS compatibility scanner

        "ryoko_flake_gen",     # NixOS flake generator

        "ryoko_auto_interface",# hardware discovery

        "adapter_approval_api",# LLM adapter security gate

        "shared_types",        # canonical shared types + sandbox

        "treatment_loop",      # biological treatment loop

        "kras_pipeline",       # KRAS G12C P-layer scoring

        "ryoko_integration_roadmap", # KRAS integration roadmap

        "ryoko_seven_functional",    # core NP-P-K solver

        "photonic_np",         # photonic NP layer

        "dual_track_ryoko",    # dual-track hardware blueprint

        "adapter",             # Eu³⁺ Foundry real-time adapter

    ],


    install_requires = [

        "numpy>=1.24",

        "scipy>=1.10",

        "requests>=2.28",

        "fastapi>=0.100",

        "uvicorn>=0.22",

        "pydantic>=2.0",

    ],


    extras_require = {

        # Real biochemistry (replaces heuristics)

        "bio": [

            "rdkit-pypi>=2023.3",

        ],

        # Hardware discovery

        "hardware": [

            "pyusb>=1.2",

            "pyserial>=3.5",

            "zeroconf>=0.84",

        ],

        # Full scientific stack

        "science": [

            "matplotlib>=3.7",

            "pandas>=2.0",

        ],

        # Everything

        "full": [

            "rdkit-pypi>=2023.3",

            "pyusb>=1.2",

            "pyserial>=3.5",

            "zeroconf>=0.84",

            "matplotlib>=3.7",

            "pandas>=2.0",

        ],

    },


    entry_points = {

        "console_scripts": [

            # Primary entry point

            "ryoko = ryoko:main_cli",

            # Domain shortcuts

            "ryoko-scan    = ryoko_scanner:main_cli",

            "ryoko-flake   = ryoko_flake_gen:main_cli",

            "ryoko-treat   = treatment_loop:main_cli",

        ],

    },


    classifiers = [

        "Development Status :: 3 - Alpha",

        "Intended Audience :: Science/Research",

        "Intended Audience :: Developers",

        "License :: OSI Approved :: MIT License",

        "Programming Language :: Python :: 3",

        "Programming Language :: Python :: 3.9",

        "Programming Language :: Python :: 3.10",

        "Programming Language :: Python :: 3.11",

        "Topic :: Scientific/Engineering :: Bio-Informatics",

        "Topic :: Scientific/Engineering :: Physics",

        "Topic :: System :: Operating System",

        "Topic :: Software Development :: Libraries :: Python Modules",

    ],


    keywords = (

        "convergence biology cancer kras nixos linux "

        "pattern-optimization phi-pi-e k-resonance "

        "drug-discovery treatment-loop oscillation-detection"

    ),


    project_urls = {

        "Source":       "https://github.com/ryokoseven/os",

        "Bug Tracker":  "https://github.com/ryokoseven/os/issues",

    },

)

In [ ]:
import { useState, useEffect, useRef, useCallback } from "react";

import {

  LineChart, Line, XAxis, YAxis, CartesianGrid,

  Tooltip, Legend, ResponsiveContainer, ReferenceLine,

} from "recharts";


// ============================================================

// HYPERION SPIN-ORBIT PARAMETERS

// Wisdom, Peale & Mignard (1984) Icarus 58, 137-152

// ============================================================

const HYP = {

  e:       0.1042,    // orbital eccentricity (Hyperion-Saturn)

  eps:     0.26,      // shape asymmetry ε = (B-A)/C

  T_days:  21.277,    // orbital period (days)

  omega0:  2.0,       // initial spin rate (units of n) — chaotic zone

  theta0:  0.0,       // initial rotation angle (rad)

  M0:      0.0,       // initial mean anomaly (rad)

};


// Literature Lyapunov time for Hyperion: ~36-84 days

// In normalized units (period=2π, n=1): τ_L ≈ T_days/period_norm * lit_days


// ============================================================

// KEPLER'S EQUATION  M = E − e·sin E

// Newton-Raphson solver

// ============================================================

function solveKepler(M, e) {

  M = ((M % (2*Math.PI)) + 2*Math.PI) % (2*Math.PI);

  let E = M;

  for(let i=0; i<40; i++) {

    const dE = (M - E + e*Math.sin(E)) / (1 - e*Math.cos(E));

    E += dE;

    if(Math.abs(dE) < 1e-12) break;

  }

  return E;

}


// Orbital state (r/a, true anomaly f, eccentric anomaly E) at mean anomaly M

function orbState(M, e) {

  const E    = solveKepler(M, e);

  const cosE = Math.cos(E), sinE = Math.sin(E);

  const r    = 1 - e*cosE;                         // r/a (normalized)

  const cosF = (cosE - e) / r;

  const sinF = Math.sqrt(1 - e*e) * sinE / r;

  const f    = Math.atan2(sinF, cosF);              // true anomaly

  return { r, f, E, sinE };

}


// ============================================================

// EQUATIONS OF MOTION  (time in units where n=1, period = 2π)

//

//  dθ/dt = ω

//  dω/dt = −(3/2)ε·(a/r)³·sin 2(θ−f)

//  dM/dt = 1

//

// Liouville: trace(J) = 0  →  Σλᵢ = 0  (Hamiltonian, conservative)

// This is the KEY difference from Lorenz where Σλᵢ = −13.667

// ============================================================

function hyperionF(state, e = HYP.e, eps = HYP.eps) {

  const [theta, omega, M] = state;

  const { r, f } = orbState(M, e);

  return [

    omega,

    -(3/2) * eps * Math.pow(r, -3) * Math.sin(2*(theta - f)),

    1.0

  ];

}


// Analytical Jacobian J = ∂F/∂state

// Derived from spin-orbit equations (see README Appendix)

function hyperionJ(state, e = HYP.e, eps = HYP.eps) {

  const [theta, , M] = state;

  const { r, f, sinE } = orbState(M, e);

  const r3inv = Math.pow(r, -3);

  const r5inv = Math.pow(r, -5);

  const phi   = 2*(theta - f);

  const sqe   = Math.sqrt(1 - e*e);


  // J[1][0] = ∂(dω/dt)/∂θ

  const J10 = -3 * eps * r3inv * Math.cos(phi);


  // J[1][2] = ∂(dω/dt)/∂M  (involves dr/dM and df/dM via chain rule)

  // dr/dM = e·sinE / r,  df/dM = √(1−e²) / r²

  const J12 = -(3/2) * eps * r5inv * (

    -3 * e * sinE * Math.sin(phi) + 2 * sqe * Math.cos(phi)

  );


  return [

    [0,    1,   0  ],   // ∂(dθ/dt)/∂(θ,ω,M)

    [J10,  0,   J12],   // ∂(dω/dt)/∂(θ,ω,M)

    [0,    0,   0  ]    // ∂(dM/dt)/∂(θ,ω,M) — trace = 0 always

  ];

}


// ============================================================

// LINEAR ALGEBRA UTILITIES

// ============================================================

const dot3   = (a,b) => a[0]*b[0]+a[1]*b[1]+a[2]*b[2];

const norm3  = v => Math.sqrt(dot3(v,v));

const scale3 = (v,s) => v.map(x=>x*s);

const sub3   = (a,b) => a.map((x,i)=>x-b[i]);

const mv3    = (M,v) => M.map(row=>dot3(row,v));


function gramSchmidt(vecs) {

  const Q=[], R=[];

  for(let i=0; i<vecs.length; i++) {

    let v=[...vecs[i]];

    for(let j=0; j<Q.length; j++) v=sub3(v, scale3(Q[j],dot3(v,Q[j])));

    const n=norm3(v); const safe=n<1e-14;

    R.push(safe?1e-14:n);

    Q.push(safe?vecs[i].map((_,k)=>k===i?1:0):scale3(v,1/n));

  }

  return {Q,R};

}


function rk4(state, f, dt) {

  const k1=f(state), k2=f(state.map((v,i)=>v+dt/2*k1[i]));

  const k3=f(state.map((v,i)=>v+dt/2*k2[i])), k4=f(state.map((v,i)=>v+dt*k3[i]));

  return state.map((v,i)=>v+(dt/6)*(k1[i]+2*k2[i]+2*k3[i]+k4[i]));

}


// ============================================================

// SIMULATION ENGINE

// 12D RK4: primary state [θ,ω,M] + 3 tangent vectors (9D)

// Same architecture as WHUCM Lorenz engine — equations swapped

// ============================================================

class HyperionEngine {

  constructor() { this.reset(); }


  reset() {

    this.t     = 0;

    this.dt    = 0.01;   // time step (n=1 units)

    this.cycle = 0;

    // Primary [θ,ω,M] + 3 identity tangent vectors

    this.state  = [HYP.theta0, HYP.omega0, HYP.M0, 1,0,0, 0,1,0, 0,0,1];

    this.sumL   = [0,0,0];

    this.steps  = 0;

    this.lambda = [0,0,0];

    this.M_orb  = 0;    // orbit counter (floor(M/2π))


    this.lyapHist  = [];

    this.spinHist  = [];

    this.poincare  = [];   // Poincaré section: sampled at each periapsis

  }


  step() {

    this.cycle++;

    const { e, eps } = HYP;


    // Combined F for 12D state [primary(3), q1(3), q2(3), q3(3)]

    const combinedF = s => {

      const prim = s.slice(0,3);

      const J    = hyperionJ(prim, e, eps);

      const res  = hyperionF(prim, e, eps);

      for(let i=0; i<3; i++) res.push(...mv3(J, s.slice(3+i*3, 6+i*3)));

      return res;

    };


    const next = rk4(this.state, combinedF, this.dt);


    // Gram-Schmidt reorthonormalization of tangent vectors

    const { Q, R } = gramSchmidt([next.slice(3,6), next.slice(6,9), next.slice(9,12)]);

    const logR = R.map(r => r>0&&isFinite(r) ? Math.log(r) : 0);

    for(let i=0; i<3; i++) this.sumL[i] += logR[i];

    this.steps++;

    this.t += this.dt;

    if(this.steps > 0) this.lambda = this.sumL.map(s => s/this.t);


    this.state = [...next.slice(0,3), ...Q[0], ...Q[1], ...Q[2]];

    const [theta, omega, M] = this.state;


    // Lyapunov spectrum

    const [l1,l2,l3] = this.lambda;

    const lSum = l1+l2+l3;

    const tauNorm = l1>0.001 ? Math.log(0.1/1e-6)/l1 : Infinity;

    // Convert τ* to days: t_norm / (2π) * T_days

    const tauDays = isFinite(tauNorm)

      ? Math.min(tauNorm * HYP.T_days / (2*Math.PI), 9999) : 9999;


    // Physical time conversions

    const t_orbits = this.t / (2*Math.PI);

    const t_days   = t_orbits * HYP.T_days;


    // Poincaré section: record (θ mod π, ω) at each periapsis (M crossing 2π·k)

    const newOrb = Math.floor(M / (2*Math.PI));

    if(newOrb > this.M_orb) {

      this.M_orb = newOrb;

      const th_mod = ((theta % Math.PI) + Math.PI) % Math.PI;

      this.poincare.push({ th: +th_mod.toFixed(4), om: +omega.toFixed(4) });

      if(this.poincare.length > 1000) this.poincare.shift();

    }


    // History

    this.lyapHist.push({

      t:  +t_days.toFixed(1),

      l1: +l1.toFixed(5), l2: +l2.toFixed(5), l3: +l3.toFixed(5),

      sum:+lSum.toFixed(5)

    });

    if(this.lyapHist.length > 200) this.lyapHist.shift();


    this.spinHist.push({ t:+t_days.toFixed(1), omega:+omega.toFixed(4) });

    if(this.spinHist.length > 300) this.spinHist.shift();


    return {

      t_days:  +t_days.toFixed(2),

      t_orbits:+t_orbits.toFixed(3),

      cycle:   this.cycle,

      theta:   +theta.toFixed(5),

      omega:   +omega.toFixed(5),

      M:       +(M % (2*Math.PI)).toFixed(5),

      l1:      +l1.toFixed(5), l2:+l2.toFixed(5), l3:+l3.toFixed(5),

      lSum:    +lSum.toFixed(5),

      tauDays: +tauDays.toFixed(1),

      poincare_n: this.poincare.length,

    };

  }

}


// ============================================================

// POINCARÉ SECTION CANVAS

// Incremental drawing — only renders new points each update

// ============================================================

function PoincareCanvas({ poincare, resetKey }) {

  const ref      = useRef(null);

  const lastN    = useRef(0);

  const prevKey  = useRef(resetKey);


  const W = 360, H = 300;

  const OM_MIN = 0.0, OM_MAX = 4.0;

  const TH_MIN = 0.0, TH_MAX = Math.PI;


  const thToX = th => ((th - TH_MIN) / (TH_MAX - TH_MIN)) * W;

  const omToY = om => H - ((om - OM_MIN) / (OM_MAX - OM_MIN)) * H;


  function drawBackground(ctx) {

    ctx.fillStyle = "#ffffff";

    ctx.fillRect(0,0,W,H);


    // Grid lines at spin-orbit resonances

    ctx.strokeStyle = "#e0e0e0"; ctx.lineWidth = 0.5;

    [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5].forEach(om => {

      const y = omToY(om);

      ctx.beginPath(); ctx.moveTo(0,y); ctx.lineTo(W,y); ctx.stroke();

    });

    // θ gridlines at π/4 intervals

    [0, Math.PI/4, Math.PI/2, 3*Math.PI/4, Math.PI].forEach(th => {

      const x = thToX(th);

      ctx.beginPath(); ctx.moveTo(x,0); ctx.lineTo(x,H); ctx.stroke();

    });


    // Resonance labels

    ctx.fillStyle = "#bdbdbd"; ctx.font = "8px monospace";

    [[0.5,"1:2"],[1.0,"1:1"],[1.5,"3:2"],[2.0,"2:1"],[2.5,"5:2"],[3.0,"3:1"]].forEach(([om,lbl])=>{

      ctx.fillText(lbl, 3, omToY(om)-2);

    });


    // Border

    ctx.strokeStyle = "#dee2e6"; ctx.lineWidth = 1;

    ctx.strokeRect(0,0,W,H);

  }


  // Clear and redraw on reset

  useEffect(() => {

    if(resetKey !== prevKey.current) {

      prevKey.current = resetKey;

      lastN.current   = 0;

      const ctx = ref.current?.getContext("2d");

      if(ctx) drawBackground(ctx);

    }

  }, [resetKey]);


  // Initial background

  useEffect(() => {

    const ctx = ref.current?.getContext("2d");

    if(ctx) drawBackground(ctx);

  }, []);


  // Incremental draw of new points only

  useEffect(() => {

    const canvas = ref.current;

    if(!canvas || poincare.length === 0) return;

    const ctx = canvas.getContext("2d");

    const start = lastN.current;

    const end   = poincare.length;


    for(let i = start; i < end; i++) {

      const p  = poincare[i];

      const px = thToX(p.th);

      const py = omToY(p.om);

      // Color cycles from deep blue (early) toward teal (later) for visual density

      const progress = Math.min(i / 500, 1);

      const hue = 220 - progress * 40;

      ctx.fillStyle = `hsla(${hue},75%,42%,0.7)`;

      ctx.beginPath(); ctx.arc(px, py, 1.8, 0, Math.PI*2); ctx.fill();

    }

    lastN.current = end;

  }, [poincare.length]);


  return (

    <div style={{position:"relative", display:"inline-block"}}>

      <canvas ref={ref} width={W} height={H}

        style={{display:"block", border:"1px solid #dee2e6", borderRadius:4}}/>

      {/* y-axis label */}

      <div style={{position:"absolute",top:0,bottom:0,left:-18,display:"flex",

        alignItems:"center",justifyContent:"center"}}>

        <span style={{fontFamily:"monospace",fontSize:"0.6rem",color:"#6c757d",

          writingMode:"vertical-rl",transform:"rotate(180deg)"}}>

          ω  (units of n)

        </span>

      </div>

      {/* x-axis labels */}

      <div style={{display:"flex",justifyContent:"space-between",

        padding:"2px 0",fontSize:"0.6rem",color:"#6c757d",fontFamily:"monospace"}}>

        <span>0</span><span>π/4</span><span>π/2</span><span>3π/4</span><span>π</span>

      </div>

      <div style={{textAlign:"center",fontSize:"0.6rem",color:"#6c757d",

        fontFamily:"monospace",marginTop:1}}>

        θ  (mod π)

      </div>

    </div>

  );

}


// ============================================================

// UI ATOMS

// ============================================================

const C = {

  bg:'#f8f9fa', panel:'#ffffff', border:'#dee2e6',

  text:'#212529', muted:'#6c757d', grid:'#f0f0f0',

  l1:'#1565c0', l2:'#2e7d32', l3:'#c62828', lsum:'#6a1b9a',

  omega:'#0277bd', theta:'#e65100',

  good:'#2e7d32', warn:'#f57f17', info:'#0277bd',

};


function DataRow({label,value,unit="",note="",color=C.text}) {

  return(

    <div style={{display:"grid",gridTemplateColumns:"155px 1fr auto",

      gap:6,alignItems:"baseline",padding:"3px 0",

      borderBottom:`1px solid ${C.grid}`}}>

      <span style={{fontSize:"0.71rem",color:C.muted}}>{label}</span>

      <span style={{fontSize:"0.76rem",color,fontFamily:"monospace",fontWeight:500}}>{value}</span>

      <span style={{fontSize:"0.63rem",color:C.muted}}>{unit}{note?` · ${note}`:""}</span>

    </div>

  );

}


function SectionHead({children, sub}) {

  return(

    <div style={{borderBottom:`2px solid ${C.border}`,paddingBottom:5,marginBottom:10}}>

      <div style={{fontSize:"0.75rem",fontWeight:700,letterSpacing:2,

        textTransform:"uppercase",color:C.muted}}>{children}</div>

      {sub&&<div style={{fontSize:"0.62rem",color:C.muted,marginTop:1}}>{sub}</div>}

    </div>

  );

}


function StatusPill({ok,label,info}) {

  const bg  = info?"#e3f2fd" : ok?"#e8f5e9":"#fff3e0";

  const col = info?C.info   : ok?C.good   :C.warn;

  const bdr = info?"#90caf9": ok?"#a5d6a7":"#ffcc02";

  return(

    <span style={{display:"inline-block",padding:"2px 8px",borderRadius:2,

      fontSize:"0.65rem",fontWeight:700,letterSpacing:1,

      background:bg,color:col,border:`1px solid ${bdr}`}}>

      {label}

    </span>

  );

}


function Btn({children,onClick,variant="default"}) {

  const s = {

    default:{background:"#fff",border:`1px solid ${C.border}`,color:C.text},

    primary:{background:C.l1,border:`1px solid ${C.l1}`,color:"#fff"},

    danger: {background:C.l3,border:`1px solid ${C.l3}`,color:"#fff"},

  };

  return(

    <button onClick={onClick} style={{...s[variant],padding:"4px 12px",

      borderRadius:3,cursor:"pointer",fontSize:"0.7rem",fontFamily:"inherit",

      fontWeight:600,letterSpacing:1,textTransform:"uppercase"}}>{children}</button>

  );

}


const axStyle = {

  tick:{fill:C.muted,fontSize:9,fontFamily:"monospace"},

  axisLine:false, tickLine:false

};


const ttStyle = {

  contentStyle:{background:"#fff",border:`1px solid ${C.border}`,

    borderRadius:3,fontSize:"0.7rem",fontFamily:"monospace"},

  labelStyle:{color:C.text,fontWeight:600},

  itemStyle:{fontSize:"0.68rem"},

};


// ============================================================

// MAIN DASHBOARD

// ============================================================

export default function HyperionDashboard() {

  const [latest,  setLatest]  = useState(null);

  const [lyapH,   setLyapH]   = useState([]);

  const [spinH,   setSpinH]   = useState([]);

  const [poinc,   setPoinc]   = useState([]);

  const [running, setRunning] = useState(false);

  const [speed,   setSpeed]   = useState(60);

  const [tab,     setTab]     = useState("dynamics");

  const [resetKey,setResetKey]= useState(0);


  const engRef = useRef(new HyperionEngine());

  const tmrRef = useRef(null);


  // Run 4 steps per tick — fills Poincaré section faster

  const tick = useCallback(() => {

    let e = null;

    for(let i=0; i<4; i++) e = engRef.current.step();

    setLatest(e);

    setLyapH([...engRef.current.lyapHist]);

    setSpinH([...engRef.current.spinHist]);

    setPoinc([...engRef.current.poincare]);

  }, []);


  useEffect(() => {

    if(running) tmrRef.current = setInterval(tick, speed);

    else clearInterval(tmrRef.current);

    return () => clearInterval(tmrRef.current);

  }, [running, speed, tick]);


  const reset = () => {

    clearInterval(tmrRef.current);

    engRef.current.reset();

    setLatest(null); setLyapH([]); setSpinH([]); setPoinc([]);

    setRunning(false); setResetKey(k=>k+1);

  };


  const liouOk = Math.abs(latest?.lSum??9) < 0.05;   // Hamiltonian → Σλ ≈ 0

  const isChaotic = (latest?.l1??0) > 0.05;


  return(

    <div style={{background:C.bg, minHeight:"100vh", color:C.text,

      fontFamily:'"IBM Plex Sans","Helvetica Neue",sans-serif', fontSize:"0.85rem"}}>


      <style>{`

        @import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:wght@400;500;600;700&family=IBM+Plex+Mono:wght@400;500&display=swap');

        * { box-sizing: border-box; }

        button:hover { opacity: 0.85; }

      `}</style>


      {/* ── HEADER ── */}

      <div style={{background:"#fff", borderBottom:`1px solid ${C.border}`,

        padding:"10px 20px", display:"flex", justifyContent:"space-between",

        alignItems:"center"}}>

        <div>

          <div style={{fontWeight:700, fontSize:"1rem", letterSpacing:0.5}}>

            Hyperion Spin-Orbit Chaos  ·  Saturn Moon Tumbling Dynamics

          </div>

          <div style={{fontSize:"0.68rem",color:C.muted,marginTop:1,fontFamily:"monospace"}}>

            Triaxial ellipsoid model · ε={HYP.eps} · e={HYP.e} · T={HYP.T_days} days ·

            12D RK4 + Gram-Schmidt · Analytical Jacobian · Hamiltonian system (Σλ=0)

          </div>

        </div>

        <div style={{display:"flex",gap:8,alignItems:"center",flexWrap:"wrap"}}>

          {latest&&<StatusPill info label={`${latest.t_orbits} orbits · ${latest.t_days} days`}/>}

          {latest&&<StatusPill ok={liouOk}

            label={`Liouville (Hamiltonian)  Σλ = ${latest.lSum.toFixed(4)}`}/>}

          {latest&&<StatusPill ok={isChaotic}

            label={isChaotic?`CHAOTIC  λ₁=${latest.l1.toFixed(4)}`:`regular  λ₁=${latest.l1.toFixed(4)}`}/>}

        </div>

      </div>


      <div style={{display:"grid",gridTemplateColumns:"265px 1fr",

        minHeight:"calc(100vh - 53px)"}}>


        {/* ── SIDEBAR ── */}

        <div style={{background:"#fff",borderRight:`1px solid ${C.border}`,

          padding:"16px",overflowY:"auto"}}>


          <SectionHead>Simulation Control</SectionHead>

          <div style={{display:"flex",gap:6,marginBottom:12,flexWrap:"wrap"}}>

            <Btn variant={running?"danger":"primary"} onClick={()=>setRunning(r=>!r)}>

              {running?"■ Stop":"▶ Run"}

            </Btn>

            <Btn onClick={reset}>Reset</Btn>

          </div>

          <div style={{marginBottom:14}}>

            <div style={{display:"flex",justifyContent:"space-between",marginBottom:3}}>

              <label style={{fontSize:"0.7rem",color:C.muted}}>Tick (ms)</label>

              <span style={{fontFamily:"monospace",fontSize:"0.7rem"}}>{speed}</span>

            </div>

            <input type="range" min={30} max={300} value={speed}

              onChange={e=>setSpeed(+e.target.value)}

              style={{width:"100%",accentColor:C.l1}}/>

            <div style={{fontSize:"0.61rem",color:C.muted,marginTop:2,lineHeight:1.6}}>

              4 RK4 steps per tick. Let it run ~200 orbits for a

              well-populated Poincaré section.

            </div>

          </div>


          <SectionHead>Current State</SectionHead>

          <div style={{marginBottom:14}}>

            <DataRow label="Physical time"     value={latest?.t_days??   "—"} unit="days"/>

            <DataRow label="Orbital periods"   value={latest?.t_orbits?? "—"} unit="orbits"/>

            <DataRow label="Rotation angle θ"  value={latest?.theta??    "—"} unit="rad" color={C.theta}/>

            <DataRow label="Spin rate ω"       value={latest?.omega??    "—"} unit="n"

              note="1=sync, 1.5=3:2 res" color={C.omega}/>

            <DataRow label="Mean anomaly M"    value={latest?.M??        "—"} unit="rad"/>

            <DataRow label="Poincaré points"   value={poinc.length}/>

          </div>


          <SectionHead>Lyapunov Spectrum</SectionHead>

          <div style={{marginBottom:10}}>

            <DataRow label="λ₁  (expanding)"  value={latest?.l1??  "—"} unit="n⁻¹"

              color={isChaotic?C.l3:C.l2}/>

            <DataRow label="λ₂  (neutral)"    value={latest?.l2??  "—"} unit="n⁻¹"

              note="expect ≈ 0"/>

            <DataRow label="λ₃  (contracting)" value={latest?.l3?? "—"} unit="n⁻¹"

              note="expect ≈ −λ₁"/>

            <DataRow label="Σλᵢ"              value={latest?.lSum??"—"}

              color={liouOk?C.good:C.warn} note="Hamiltonian → 0"/>

            <DataRow label="τ*  (predict.)"   value={latest?.tauDays??"—"} unit="days"

              note="ln(Δ_max/Δ₀)/λ₁"/>

          </div>

          <div style={{fontSize:"0.61rem",color:C.muted,background:"#f8f9fa",

            border:`1px solid ${C.border}`,borderRadius:3,padding:"7px 9px",

            lineHeight:1.75,marginBottom:14}}>

            <strong style={{color:C.text}}>Hamiltonian spectrum:</strong><br/>

            (+λ₁,  0,  −λ₁)<br/>

            Phase-space volume preserved.<br/>

            No strange attractor — trajectories<br/>

            fill chaotic regions area-densely.<br/>

            <br/>

            <strong style={{color:C.text}}>Contrast with Lorenz:</strong><br/>

            Lorenz: Σλᵢ = −13.667 (dissipative)<br/>

            Strange attractor, fractal structure<br/>

            D_KY ≈ 2.062, contracts to set.

          </div>


          <SectionHead>Physical Parameters</SectionHead>

          <div style={{marginBottom:14}}>

            {[

              ["ε = (B-A)/C",    HYP.eps,     "shape asymmetry"],

              ["e",              HYP.e,        "orbital eccentricity"],

              ["T",              `${HYP.T_days} d`, "orbital period"],

              ["ω₀",            HYP.omega0,   "initial spin rate (n)"],

              ["dt",             "0.01",       "step (n=1 units)"],

              ["4 steps/tick",  "×4",          "for speed"],

              ["Kepler solver",  "Newton-Raphson","30 iterations"],

              ["Jacobian",       "Analytical", "exact, no finite diff"],

            ].map(([l,v,n])=>(<DataRow key={l} label={l} value={v} note={n}/>))}

          </div>


          <div style={{background:"#e8f5e9",border:"1px solid #a5d6a7",

            borderRadius:3,padding:"7px 9px",fontSize:"0.61rem",

            lineHeight:1.8,color:"#1b5e20"}}>

            <strong>Other systems you can swap in:</strong><br/>

            · Pluto-Charon obliquity chaos<br/>

            · Asteroid belt 3:1 Kirkwood gap<br/>

            · Double pendulum (another Hamiltonian)<br/>

            · Mercury spin-orbit capture<br/>

            Same engine — replace hyperionF() and hyperionJ().

          </div>

        </div>


        {/* ── MAIN CONTENT ── */}

        <div style={{padding:"16px",overflowY:"auto"}}>


          {/* Tabs */}

          <div style={{display:"flex",gap:0,marginBottom:16,

            borderBottom:`2px solid ${C.border}`}}>

            {[

              ["dynamics",  "Spin Dynamics"],

              ["poincare",  "Poincaré Section"],

              ["lyapunov",  "Lyapunov Spectrum"],

              ["reference", "Physics & Reference"],

            ].map(([id,lbl])=>(

              <button key={id} onClick={()=>setTab(id)} style={{

                padding:"7px 18px",border:"none",

                borderBottom:tab===id?`2px solid ${C.l1}`:"2px solid transparent",

                marginBottom:-2,background:"transparent",

                color:tab===id?C.l1:C.muted,fontFamily:"inherit",

                fontSize:"0.72rem",fontWeight:tab===id?700:400,

                letterSpacing:0.5,cursor:"pointer",textTransform:"uppercase",

              }}>{lbl}</button>

            ))}

          </div>


          {/* ── DYNAMICS TAB ── */}

          {tab==="dynamics"&&(

            <div>

              <div style={{background:"#fff",border:`1px solid ${C.border}`,

                borderRadius:4,padding:"12px",marginBottom:12}}>

                <div style={{fontSize:"0.7rem",fontWeight:600,

                  color:C.muted,textTransform:"uppercase",letterSpacing:1,marginBottom:3}}>

                  Spin Rate ω(t) — Chaotic Tumbling Signature

                </div>

                <div style={{fontSize:"0.62rem",color:C.muted,marginBottom:10,lineHeight:1.7}}>

                  Horizontal lines mark spin-orbit resonances (ω = n/2, n, 3/2n, 2n...).

                  Regular rotation produces smooth periodic oscillation around a resonance.

                  Chaotic rotation jumps irregularly between regions — Hyperion's observed behavior.

                </div>

                <ResponsiveContainer width="100%" height={220}>

                  <LineChart data={spinH} margin={{top:4,right:12,bottom:0,left:0}}>

                    <CartesianGrid stroke={C.grid} strokeDasharray="2 4"/>

                    <XAxis dataKey="t" {...axStyle}

                      label={{value:"t (days)",fill:C.muted,fontSize:9,position:"insideRight"}}/>

                    <YAxis {...axStyle}

                      label={{value:"ω (n)",fill:C.muted,fontSize:9,angle:-90,position:"insideLeft"}}/>

                    <Tooltip {...ttStyle} labelFormatter={v=>`t = ${v} days`}/>

                    {[0.5,1.0,1.5,2.0,2.5,3.0].map(om=>(

                      <ReferenceLine key={om} y={om} stroke="#e0e0e0" strokeDasharray="2 4"

                        label={{value:[" ½","1:1","3:2","2:1","5:2","3:1"][[0.5,1,1.5,2,2.5,3].indexOf(om)],

                          fill:"#bdbdbd",fontSize:8,position:"right"}}/>

                    ))}

                    <Line dataKey="omega" name="ω (spin rate)" stroke={C.omega}

                      strokeWidth={1.5} dot={false}/>

                    <Legend wrapperStyle={{fontSize:"0.68rem",fontFamily:"monospace"}}/>

                  </LineChart>

                </ResponsiveContainer>

              </div>


              <div style={{display:"grid",gridTemplateColumns:"1fr 1fr",gap:12}}>

                <div style={{background:"#fff",border:`1px solid ${C.border}`,

                  borderRadius:4,padding:"12px"}}>

                  <div style={{fontSize:"0.7rem",fontWeight:600,

                    color:C.muted,textTransform:"uppercase",letterSpacing:1,marginBottom:3}}>

                    State Variables

                  </div>

                  {[

                    ["θ (rotation angle)", latest?.theta??"—", "rad", C.theta],

                    ["ω (spin rate)",      latest?.omega??"—", "n",   C.omega],

                    ["M (mean anomaly)",   latest?.M??"—",     "rad", C.muted],

                    ["λ₁",                latest?.l1??"—",     "n⁻¹", C.l1],

                    ["λ₃",                latest?.l3??"—",     "n⁻¹", C.l3],

                    ["Σλᵢ",               latest?.lSum??"—",   "",    liouOk?C.good:C.warn],

                    ["τ*",                latest?.tauDays??"—","days", C.info],

                  ].map(([l,v,u,c])=>(

                    <DataRow key={l} label={l} value={v} unit={u} color={c}/>

                  ))}

                </div>


                <div style={{background:"#fffde7",border:"1px solid #fff176",

                  borderRadius:4,padding:"12px",fontSize:"0.68rem",lineHeight:1.85}}>

                  <div style={{fontWeight:700,marginBottom:6,fontSize:"0.7rem",

                    color:C.muted,textTransform:"uppercase",letterSpacing:1}}>

                    Observational Context

                  </div>

                  <p style={{margin:"0 0 8px 0",color:C.text}}>

                    Hyperion's chaotic tumbling was <strong>confirmed observationally</strong> by

                    Klavetter (1989) using 13 months of photometry — the light curve was

                    non-repeating and unpredictable, exactly as this model predicts.

                  </p>

                  <p style={{margin:"0 0 8px 0",color:C.text}}>

                    Voyager 2 fly-by (1981) captured Hyperion's irregular shape:

                    ~360×280×225 km, heavily cratered, low density (≈0.54 g/cm³ —

                    largely empty space). The irregular shape drives ε=0.26.

                  </p>

                  <p style={{margin:0,color:C.text}}>

                    Predictability horizon τ* ≈ {latest?.tauDays??'?'} days means

                    any rotation prediction beyond ~{Math.round((latest?.tauDays??36)/7)} weeks

                    diverges exponentially from reality, regardless of measurement precision.

                    This is not noise — it is deterministic chaos.

                  </p>

                </div>

              </div>

            </div>

          )}


          {/* ── POINCARÉ TAB ── */}

          {tab==="poincare"&&(

            <div>

              <div style={{display:"grid",gridTemplateColumns:"auto 1fr",gap:20,

                alignItems:"start"}}>

                <div>

                  <div style={{background:"#fff",border:`1px solid ${C.border}`,

                    borderRadius:4,padding:"12px",marginBottom:8}}>

                    <div style={{fontSize:"0.7rem",fontWeight:600,

                      color:C.muted,textTransform:"uppercase",letterSpacing:1,marginBottom:3}}>

                      Poincaré Section

                    </div>

                    <div style={{fontSize:"0.62rem",color:C.muted,marginBottom:8,lineHeight:1.6}}>

                      Sampled at periapsis (M = 0 mod 2π).

                      Points: <strong style={{color:C.l1}}>{poinc.length}</strong> orbits.

                      Run for 200+ orbits to see full structure.

                    </div>

                    <PoincareCanvas poincare={poinc} resetKey={resetKey}/>

                  </div>

                  <div style={{background:"#f8f9fa",border:`1px solid ${C.border}`,

                    borderRadius:4,padding:"8px 10px",fontSize:"0.62rem",

                    color:C.muted,lineHeight:1.75,maxWidth:400}}>

                    <strong style={{color:C.text}}>Color:</strong> Early points (blue) →

                    later points (teal). Dense clusters indicate preferred

                    rotation states. Scattered fill = chaotic region.

                    KAM islands (if visible) appear as closed loops near

                    ω = 1.0 (1:1) and ω = 1.5 (3:2).

                  </div>

                </div>


                <div style={{display:"flex",flexDirection:"column",gap:12}}>

                  <div style={{background:"#fff",border:`1px solid ${C.border}`,

                    borderRadius:4,padding:"12px",fontSize:"0.68rem",lineHeight:1.9}}>

                    <div style={{fontWeight:700,marginBottom:8,fontSize:"0.7rem",

                      color:C.muted,textTransform:"uppercase",letterSpacing:1}}>

                      Reading the Poincaré Section

                    </div>

                    {[

                      ["Closed loop","Regular spin-orbit resonance (KAM torus)"],

                      ["Scattered fill","Chaotic tumbling — the Hyperion regime"],

                      ["Island chains","Higher-order resonances inside the chaotic sea"],

                      ["Empty regions","Stable KAM barriers (forbidden zones)"],

                    ].map(([pattern,meaning])=>(

                      <div key={pattern} style={{display:"grid",

                        gridTemplateColumns:"110px 1fr",gap:8,

                        borderBottom:`1px solid ${C.grid}`,padding:"3px 0"}}>

                        <span style={{fontWeight:600,color:C.text}}>{pattern}</span>

                        <span style={{color:C.muted}}>{meaning}</span>

                      </div>

                    ))}

                  </div>


                  <div style={{background:"#fff",border:`1px solid ${C.border}`,

                    borderRadius:4,padding:"12px",fontSize:"0.68rem",lineHeight:1.8}}>

                    <div style={{fontWeight:700,marginBottom:6,fontSize:"0.7rem",

                      color:C.muted,textTransform:"uppercase",letterSpacing:1}}>

                      Historical Context

                    </div>

                    <p style={{margin:"0 0 8px"}}>

                      Wisdom, Peale & Mignard (1984) computed the first Poincaré section

                      for Hyperion parameters. They found that for ε = 0.26, e = 0.1,

                      the chaotic sea covers nearly the entire phase space between

                      ω ∈ [0.5, 3.0] — meaning chaotic tumbling is overwhelmingly

                      probable for any initial condition.

                    </p>

                    <p style={{margin:0}}>

                      This was a prediction of the <em>inevitable</em> chaos of Hyperion

                      before it was directly observed. The Poincaré section told the

                      whole story.

                    </p>

                  </div>


                  <div style={{background:"#fff",border:`1px solid ${C.border}`,

                    borderRadius:4,padding:"12px",fontSize:"0.68rem",lineHeight:1.8}}>

                    <div style={{fontWeight:700,marginBottom:6,fontSize:"0.7rem",

                      color:C.muted,textTransform:"uppercase",letterSpacing:1}}>

                      KAM Theory

                    </div>

                    <p style={{margin:"0 0 8px"}}>

                      Kolmogorov-Arnold-Moser (KAM) theory predicts that for small

                      perturbations, most regular orbits survive as deformed tori.

                      The islands you see in the Poincaré section are surviving KAM tori.

                    </p>

                    <p style={{margin:0}}>

                      For Hyperion (large ε = 0.26), the perturbation is strong enough

                      to destroy most tori, leaving a predominantly chaotic sea with only

                      small stable islands near the lowest-order resonances.

                    </p>

                  </div>

                </div>

              </div>

            </div>

          )}


          {/* ── LYAPUNOV TAB ── */}

          {tab==="lyapunov"&&(

            <div>

              <div style={{display:"grid",gridTemplateColumns:"repeat(4,1fr)",

                gap:12,marginBottom:16}}>

                {[

                  ["λ₁  (largest)", latest?.l1, "n⁻¹", isChaotic?"chaos":"regular", isChaotic?C.l3:C.l2],

                  ["λ₂  (neutral)", latest?.l2, "n⁻¹", "expect ≈ 0", C.l2],

                  ["λ₃  (contract.)",latest?.l3,"n⁻¹", "expect ≈ −λ₁", C.l3],

                  ["Σλᵢ",           latest?.lSum,"",   "Hamiltonian → 0", liouOk?C.good:C.warn],

                ].map(([lbl,val,unit,note,color])=>(

                  <div key={lbl} style={{background:"#fff",border:`1px solid ${C.border}`,

                    borderRadius:4,padding:"10px 12px"}}>

                    <div style={{fontSize:"0.65rem",color:C.muted,marginBottom:4,

                      textTransform:"uppercase",letterSpacing:1}}>{lbl}</div>

                    <div style={{fontFamily:"monospace",fontSize:"1.1rem",fontWeight:500,

                      color,marginBottom:2}}>{val!=null?val.toFixed(5):"—"}</div>

                    <div style={{fontSize:"0.62rem",color:C.muted}}>{unit} · {note}</div>

                  </div>

                ))}

              </div>


              <div style={{background:"#fff",border:`1px solid ${C.border}`,

                borderRadius:4,padding:"12px",marginBottom:12}}>

                <div style={{fontSize:"0.7rem",fontWeight:600,marginBottom:2,

                  color:C.muted,textTransform:"uppercase",letterSpacing:1}}>

                  Lyapunov Spectrum Convergence  λ₁, λ₂, λ₃, Σλᵢ  vs. Time

                </div>

                <div style={{fontSize:"0.62rem",color:C.muted,marginBottom:10,lineHeight:1.6}}>

                  Hamiltonian: Σλᵢ → 0 (not −13.667 like Lorenz). Spectrum is antisymmetric:

                  λ₃ converges to −λ₁. Convergence is slower than Lorenz — no dissipation

                  to pull trajectories toward an attractor.

                </div>

                <ResponsiveContainer width="100%" height={200}>

                  <LineChart data={lyapH} margin={{top:4,right:12,bottom:0,left:0}}>

                    <CartesianGrid stroke={C.grid} strokeDasharray="2 4"/>

                    <XAxis dataKey="t" {...axStyle}

                      label={{value:"t (days)",fill:C.muted,fontSize:9,position:"insideRight"}}/>

                    <YAxis {...axStyle}/>

                    <Tooltip {...ttStyle} labelFormatter={v=>`t = ${v} days`}/>

                    <ReferenceLine y={0} stroke={C.lsum} strokeWidth={1.5} strokeDasharray="4 2"

                      label={{value:"Σλᵢ = 0 (Hamiltonian)",fill:C.lsum,fontSize:8}}/>

                    <Line dataKey="l1"  name="λ₁" stroke={C.l1}  strokeWidth={2} dot={false}/>

                    <Line dataKey="l2"  name="λ₂" stroke={C.l2}  strokeWidth={1.5} dot={false}/>

                    <Line dataKey="l3"  name="λ₃" stroke={C.l3}  strokeWidth={1.5} dot={false}/>

                    <Line dataKey="sum" name="Σλᵢ" stroke={C.lsum} strokeWidth={1.2}

                      dot={false} strokeDasharray="4 2"/>

                    <Legend wrapperStyle={{fontSize:"0.68rem",fontFamily:"monospace"}}/>

                  </LineChart>

                </ResponsiveContainer>

              </div>


              <div style={{display:"grid",gridTemplateColumns:"1fr 1fr",gap:12}}>

                <div style={{background:"#fff",border:`1px solid ${C.border}`,

                  borderRadius:4,padding:"12px",fontSize:"0.68rem",lineHeight:1.9}}>

                  <div style={{fontWeight:700,marginBottom:8,fontSize:"0.7rem",

                    color:C.muted,textTransform:"uppercase",letterSpacing:1}}>

                    Hyperion  vs  Lorenz  — Two Chaos Types

                  </div>

                  <div style={{display:"grid",gridTemplateColumns:"110px 1fr 1fr",

                    gap:4,fontSize:"0.67rem"}}>

                    {[

                      ["","Hyperion","Lorenz"],

                      ["System type","Conservative","Dissipative"],

                      ["Σλᵢ","= 0","= −13.667"],

                      ["Liouville","Vol preserved","Vol contracts"],

                      ["Attractor","None","Strange attractor"],

                      ["Spectrum","(+λ, 0, −λ)","(+, 0, −−)"],

                      ["D_KY","3 or 2","2.062"],

                      ["Phase space","Area-filling","Fractal set"],

                      ["Origin","Hamiltonian","SDE forced"],

                      ["Predictor","τ* ~ weeks","τ* ~ 11 tu"],

                    ].map(([l,h,lor],i)=>(

                      <div key={i} style={{display:"contents"}}>

                        <span style={{fontWeight:i===0?700:400,

                          color:i===0?C.text:C.muted,borderBottom:`1px solid ${C.grid}`,

                          padding:"2px 0"}}>{l}</span>

                        <span style={{fontFamily:"monospace",

                          color:i===0?C.text:C.l1,borderBottom:`1px solid ${C.grid}`,

                          padding:"2px 0",fontWeight:i===0?700:400}}>{h}</span>

                        <span style={{fontFamily:"monospace",

                          color:i===0?C.text:C.l3,borderBottom:`1px solid ${C.grid}`,

                          padding:"2px 0",fontWeight:i===0?700:400}}>{lor}</span>

                      </div>

                    ))}

                  </div>

                </div>


                <div style={{background:"#fff",border:`1px solid ${C.border}`,

                  borderRadius:4,padding:"12px",fontSize:"0.68rem",lineHeight:1.8}}>

                  <div style={{fontWeight:700,marginBottom:6,fontSize:"0.7rem",

                    color:C.muted,textTransform:"uppercase",letterSpacing:1}}>

                    Algorithm  (same engine as WHUCM Lorenz)

                  </div>

                  <div style={{fontFamily:"monospace",fontSize:"0.67rem",lineHeight:2,

                    background:"#f8f9fa",padding:"8px",borderRadius:3,marginBottom:8}}>

                    State dim:    12  (3 primary + 9 tangent)<br/>

                    Integrator:   RK4  dt = 0.01  (n=1 units)<br/>

                    Reorthog.:    Gram-Schmidt at every step<br/>

                    Jacobian:     Analytical (exact)<br/>

                    Kepler:       Newton-Raphson, tol=10⁻¹²<br/>

                    Lyapunov:     Benettin et al. (1980)<br/>

                    Liouville:    trace(J) = 0  (verified live)<br/>

                    Poincaré:     sampled at M = 0 mod 2π

                  </div>

                  <div style={{fontSize:"0

In [ ]:
#!/usr/bin/env python3

"""

Ryoko — Unified Entry Point

============================

The front door. Not a wrapper — the NP/P/K architecture

expressed as a single coherent interface.


Every domain Ryoko operates in follows the same four verbs:


  ryoko.observe(system)   → NP: scan, generate hypotheses

  ryoko.propose()         → P:  construct environment / therapy / config

  ryoko.apply()           → K:  measure alignment, detect regime

  ryoko.explain()         → ∂:  surface conflict, suggest next step


The domains share structure, not code:


  Domain          observe()          propose()         apply()

  ─────────────────────────────────────────────────────────────

  biology         scan tumor state   design therapy    run cycle

  os              scan executable    generate flake    nix rebuild

  hardware        discover devices   generate adapter  register

  physics         measure spectrum   fit model         update params


Usage:

  python3 ryoko.py --domain biology  --target KRAS_G12C --therapy sotorasib

  python3 ryoko.py --domain os       --target /path/to/game

  python3 ryoko.py --domain hardware --scan

  python3 ryoko.py --status          (show all active loops)

  python3 ryoko.py --demo            (run all domains)


Or as a library:

  from ryoko import Ryoko

  r = Ryoko()

  r.observe("biology", target="KRAS_G12C")

  r.propose(therapy="sotorasib")

  result = r.apply()

  r.explain(result)

"""


import json

import sys

import time

import logging

import argparse

import math

from pathlib import Path

from dataclasses import dataclass, field, asdict

from typing import Optional, Dict, Any, List

from enum import Enum


log = logging.getLogger("ryoko")

logging.basicConfig(level=logging.INFO,

                    format="%(asctime)s %(levelname)-7s %(message)s",

                    datefmt="%H:%M:%S")


PHI   = (1 + math.sqrt(5)) / 2

K_MAX = PHI * math.pi * math.e   # 13.8176


# ── K MODEL PROTOCOL ─────────────────────────────────────────

# K-resonance is a normalized convergence score.

# Same ceiling (K_MAX), different semantics per domain.

# Explicitly declared — no false equivalence between domains.


from typing import Protocol, runtime_checkable


@runtime_checkable

class KModel(Protocol):

    """

    Domain-specific K-resonance computation.

    Every domain measures distance-to-goal differently.

    All normalize to [0, K_MAX] — that's the only shared thing.

    """

    domain: str


    def compute(self, state: "RyokoState") -> float:

        """Return K-resonance in [0, K_MAX]."""

        ...


    def describe(self) -> dict:

        """

        Machine-readable semantic tag for this domain's K.

        Keys:

          meaning        - what K measures in this domain

          inputs         - what data feeds into K computation

          confidence_scaled - whether K is scaled by data quality

          comparable     - False = don't compare to other domains

          clinical       - whether output carries clinical meaning

          ceiling        - K_MAX (shared structural ceiling)

        """

        ...



class BiologyKModel:

    """

    Biology: K measures distance from healthy tumor equilibrium.

    Inputs: pathway activity vector from TumorModel.

    High K = oncogenes suppressed, suppressors active, apoptosis triggered.

    """

    domain = "biology"


    def compute(self, state: "RyokoState") -> float:

        k = state.result.get("final_k", 0.0)

        return float(k)


    def describe(self) -> dict:

        return {

            "meaning":           "weighted pathway distance to healthy equilibrium",

            "inputs":            ["pathway_activity_vector", "treatment_cycle"],

            "confidence_scaled": False,

            "comparable":        False,

            "clinical":          False,

            "ceiling":           K_MAX,

            "note":              ("Pathway pressure signals only. "

                                  "Not treatment recommendations. "

                                  "Consult oncology for clinical translation."),

        }



class PhysicsKModel:

    """

    Physics: K measures spectroscopic coherence near phase transition.

    Inputs: R-ratio, β (KWW stretch), curvature d²logI/dt².

    High K = near Tc, coherent dynamics, phase transition in range.

    """

    domain = "physics"


    def compute(self, state: "RyokoState") -> float:

        p    = state.proposal

        R    = float(p.get("R")    or 2.1)

        beta = float(p.get("beta") or 0.88)

        curv = float(p.get("curvature") or -0.003)

        R_n   = max(0, min(1, (R - 0.3) / 7.7))

        b_div = 1.0 - max(0.3, min(1.0, beta))

        c_coh = max(0, min(1, -curv / 0.1))

        sat   = (1-R_n)*0.4 + (1-b_div)*0.3 + c_coh*0.3

        conf  = p.get("confidence", 1.0)

        return float(K_MAX * sat * conf)


    def describe(self) -> dict:

        return {

            "meaning":           "spectroscopic coherence near phase transition",

            "inputs":            ["R_ratio", "beta_KWW", "curvature_d2logI"],

            "confidence_scaled": True,

            "comparable":        False,

            "clinical":          False,

            "ceiling":           K_MAX,

            "note":              ("Offline Foundry sets confidence=0.3, "

                                  "scaling K down proportionally. "

                                  "Do not compare to biology or OS K."),

        }



class OSKModel:

    """

    OS: K measures dependency satisfaction for a target executable.

    Inputs: manifest satisfaction_score from ryoko_scanner.

    High K = all required libraries present, compat mode viable.

    """

    domain = "os"


    def compute(self, state: "RyokoState") -> float:

        manifest = state.proposal.get("manifest", {})

        sat      = manifest.get("satisfaction_score",

                   state.proposal.get("initial_k", 0) / K_MAX)

        return float(K_MAX * sat)


    def describe(self) -> dict:

        return {

            "meaning":           "fraction of required dependencies satisfied",

            "inputs":            ["nix_packages", "manifest_satisfaction_score"],

            "confidence_scaled": False,

            "comparable":        False,

            "clinical":          False,

            "ceiling":           K_MAX,

            "note":              ("K=K_MAX means all deps present, system ready. "

                                  "Different manifold from biology/physics K."),

        }



class HardwareKModel:

    """

    Hardware: K measures NP-layer source coverage.

    Inputs: number of active adapters feeding the 12-channel array.

    High K = rich, diverse sensor input to the NP layer.

    """

    domain = "hardware"


    # 12 channels = full photonic array = K_MAX

    FULL_CHANNELS = 12


    def compute(self, state: "RyokoState") -> float:

        n_adapters = len(state.result.get("adapters", []))

        sat        = min(1.0, n_adapters / self.FULL_CHANNELS)

        return float(K_MAX * sat)


    def describe(self) -> dict:

        return {

            "meaning":           "NP-layer sensor source coverage",

            "inputs":            ["active_adapter_count"],

            "confidence_scaled": False,

            "comparable":        False,

            "clinical":          False,

            "ceiling":           K_MAX,

            "note":              ("12 adapters = full 12-channel photonic array = K_MAX. "

                                  "Sensor coverage only — not comparable to other domains."),

        }



# Registry: domain → KModel instance

K_MODELS: dict = {

    "biology":  BiologyKModel(),

    "physics":  PhysicsKModel(),

    "os":       OSKModel(),

    "hardware": HardwareKModel(),

}


# ── AUDIT LAYER ───────────────────────────────────────────────

# Every refusal, comparison, and confidence drop is logged.

# Turns "the system said no" into "here is exactly why."


# Audit log: append-only JSONL with SHA-256 hash chain.

# Each entry includes hash of (prev_hash + current_entry).

# Deleting or modifying any entry breaks the chain — detectable.

# Append-only by design: clear_audit_log() never touches disk.


_AUDIT_FILE       = Path("./audit_logs/ryoko_audit.jsonl")

_audit_log:  list = []   # in-memory cache (fast reads)

_audit_file_handle = None  # opened lazily


# Initialize _last_hash from last entry on disk at import time.

# This ensures the chain continues correctly across sessions.

def _load_last_hash() -> str:

    try:

        if _AUDIT_FILE.exists() and _AUDIT_FILE.stat().st_size > 0:

            with open(_AUDIT_FILE) as _f:

                _last = ""

                for _line in _f:

                    if _line.strip():

                        _last = _line.strip()

            if _last:

                return json.loads(_last).get("hash", "genesis")

    except Exception:

        pass

    return "genesis"


_last_hash: str = _load_last_hash()  # chain seed — continues from disk



def _get_audit_file():

    """Lazily open audit file in append mode."""

    global _audit_file_handle

    if _audit_file_handle is None:

        _AUDIT_FILE.parent.mkdir(parents=True, exist_ok=True)

        _audit_file_handle = open(_AUDIT_FILE, "a", buffering=1)  # line-buffered

    return _audit_file_handle



def _audit(event: str, domain: str, detail: dict):

    """

    Append to both in-memory cache and on-disk JSONL.

    Each entry is hash-chained: sha256(prev_hash + entry_json).

    Modifying any entry breaks the chain — detectable by verify_audit().

    Disk write is line-buffered — survives crashes.

    """

    global _last_hash

    import hashlib as _hl


    entry = {

        "ts":     time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),

        "event":  event,

        "domain": domain,

        **detail,

    }


    # Hash chain: every entry depends on the previous one

    payload      = json.dumps(entry, sort_keys=True)

    chain_input  = (_last_hash + payload).encode()

    entry_hash   = _hl.sha256(chain_input).hexdigest()


    entry["prev"] = _last_hash

    entry["hash"] = entry_hash

    _last_hash    = entry_hash


    _audit_log.append(entry)

    try:

        _get_audit_file().write(json.dumps(entry) + "\n")

    except Exception:

        pass  # disk failure doesn't break operation — log still in memory



def get_audit_log(event_filter: str = None) -> list:

    """

    Return audit trail from in-memory cache.

    Optional event_filter: "comparison_refused", "scope_boundary",

                           "confidence_drop", "unknown_domain"

    """

    if event_filter:

        return [e for e in _audit_log if e["event"] == event_filter]

    return list(_audit_log)



def load_audit_file(path: str = None) -> list:

    """

    Load full audit history from disk.

    Reads the persistent JSONL file — includes entries from previous runs.

    Use this to audit across sessions, not just the current run.

    """

    target = Path(path) if path else _AUDIT_FILE

    if not target.exists():

        return []

    entries = []

    with open(target) as f:

        for line in f:

            line = line.strip()

            if line:

                try:

                    entries.append(json.loads(line))

                except json.JSONDecodeError:

                    pass

    return entries



def clear_audit_log():

    """Clear in-memory cache only. Disk log is never cleared — append-only."""

    _audit_log.clear()

    # Intentionally does NOT clear the disk file.

    # The persistent record survives even if in-memory state is reset.



def verify_audit(entries: list = None) -> dict:

    """

    Verify the hash chain of an audit log.

    Returns a report: valid, chain length, and first broken link if any.


    A broken chain means an entry was deleted, modified, or inserted

    after the fact. Chain integrity is necessary but not sufficient

    for full tamper-evidence (file replacement is not detected here).

    """

    import hashlib as _hl

    log = entries if entries is not None else _audit_log

    if not log:

        return {"valid": True, "length": 0, "note": "empty log"}


    prev_hash = "genesis"

    for i, entry in enumerate(log):

        stored_hash = entry.get("hash", "")

        stored_prev = entry.get("prev", "")


        if stored_prev != prev_hash:

            return {

                "valid":        False,

                "length":       len(log),

                "broken_at":    i,

                "ts":           entry.get("ts","?"),

                "event":        entry.get("event","?"),

                "reason":       "prev hash mismatch — entry deleted or reordered",

                "expected_prev":prev_hash,

                "found_prev":   stored_prev,

            }


        # Recompute hash without chain fields to verify content integrity

        check_entry = {k:v for k,v in entry.items() if k not in ("hash","prev")}

        payload     = json.dumps(check_entry, sort_keys=True)

        expected    = _hl.sha256((prev_hash + payload).encode()).hexdigest()


        if expected != stored_hash:

            return {

                "valid":     False,

                "length":    len(log),

                "broken_at": i,

                "ts":        entry.get("ts","?"),

                "event":     entry.get("event","?"),

                "reason":    "content hash mismatch — entry was modified",

            }


        prev_hash = stored_hash


    return {

        "valid":      True,

        "length":     len(log),

        "final_hash": prev_hash,

        "note":       "chain intact — no modifications detected",

    }



def audit_summary(entries: list = None) -> dict:

    """

    Summarize audit events by type.

    Works on in-memory log or any loaded list of entries.

    """

    log = entries if entries is not None else _audit_log

    by_event = {}

    for e in log:

        ev = e["event"]

        by_event.setdefault(ev, []).append(e)

    return {

        "total":              len(log),

        "by_event":           {k: len(v) for k, v in by_event.items()},

        "refusals":           by_event.get("comparison_refused", []),

        "scope_boundaries":   by_event.get("scope_boundary", []),

        "confidence_drops":   by_event.get("confidence_drop", []),

        "unknown_domains":    by_event.get("unknown_domain", []),

        "audit_file":         str(_AUDIT_FILE),

    }



def compute_k(domain: str, state: "RyokoState") -> float:

    """Compute K-resonance using domain-specific model. Logs confidence drops."""

    model = K_MODELS.get(domain)

    if model is None:

        _audit("unknown_domain", domain, {

            "reason": f"No KModel registered for domain '{domain}'",

            "result": 0.0,

        })

        return 0.0

    k = model.compute(state)

    tag = model.describe()

    # Log confidence drops (physics when Foundry offline)

    if tag.get("confidence_scaled"):

        conf = state.proposal.get("confidence", 1.0)

        if conf < 1.0:

            _audit("confidence_drop", domain, {

                "confidence": conf,

                "k_before_scaling": K_MAX * (k / (K_MAX * conf + 1e-10)),

                "k_after_scaling":  k,

                "missing": state.proposal.get("note",

                    "data source unavailable"),

            })

    return k


def describe_k(domain: str) -> dict:

    """Machine-readable K semantics for this domain."""

    model = K_MODELS.get(domain)

    return model.describe() if model else {

        "meaning": "unknown", "comparable": False, "ceiling": K_MAX

    }


def assert_comparable(domain_a: str, domain_b: str):

    """

    Raise if two domains are not K-comparable.

    Logs the refusal with full context before raising.

    Enforces the protocol — not just documents it.

    """

    da = describe_k(domain_a)

    db = describe_k(domain_b)

    if not da.get("comparable") or not db.get("comparable"):

        msg = (

            f"K-resonance from '{domain_a}' and '{domain_b}' are not comparable.\n"

            f"  {domain_a}: {da.get('meaning','?')}\n"

            f"  {domain_b}: {db.get('meaning','?')}\n"

            f"  Cross-domain K is structural analogy only — not equivalence.\n"

            f"  See KModel.describe() for each domain's semantic tag."

        )

        # Log before raising — refusal is traceable

        _audit("comparison_refused", domain_a, {

            "domain_b":    domain_b,

            "reason":      "comparable=False in one or both KModels",

            "domain_a_meaning": da.get("meaning","?"),

            "domain_b_meaning": db.get("meaning","?"),

            "clinical_a":  da.get("clinical", False),

            "clinical_b":  db.get("clinical", False),

            "boundary":    "assert_comparable",

        })

        raise ValueError(msg)



OUTPUTS = Path(__file__).parent

sys.path.insert(0, str(OUTPUTS))



# ── DOMAIN REGISTRY ───────────────────────────────────────────


class Domain(Enum):

    BIOLOGY  = "biology"

    OS       = "os"

    HARDWARE = "hardware"

    PHYSICS  = "physics"



# ── RYOKO STATE ───────────────────────────────────────────────


@dataclass

class RyokoState:

    """

    The current state of one Ryoko loop.

    Carried through observe → propose → apply → explain.

    """

    domain:       str  = ""

    target:       str  = ""

    observation:  Dict[str, Any] = field(default_factory=dict)

    proposal:     Dict[str, Any] = field(default_factory=dict)

    result:       Dict[str, Any] = field(default_factory=dict)

    k_resonance:  float = 0.0

    regime:       str   = ""   # ▲ ↑ → ↯ ▼

    cycle:        int   = 0

    converged:    bool  = False

    history:      List[Dict] = field(default_factory=list)


    def checkpoint(self):

        self.history.append({

            "cycle":       self.cycle,

            "k_resonance": self.k_resonance,

            "regime":      self.regime,

            "converged":   self.converged,

            "ts":          time.strftime("%H:%M:%S"),

        })


    def to_dict(self):

        return asdict(self)



# ── DOMAIN ADAPTERS ───────────────────────────────────────────

# Each domain implements observe / propose / apply / explain.

# The Ryoko orchestrator calls them uniformly.


class BiologyAdapter:

    """Wraps TreatmentLoop — biology domain."""

    name = "biology"


    def observe(self, state: RyokoState, **kwargs) -> RyokoState:

        target  = kwargs.get("target", "KRAS_G12C")

        therapy = kwargs.get("therapy", "sotorasib")

        state.target = target

        state.observation = {

            "target":  target,

            "therapy": therapy,

            "model":   "NSCLC KRAS G12C tumor dynamics",

        }

        state.proposal = {"therapy": therapy}

        log.info(f"  [NP] Biology: target={target}  therapy={therapy}")

        return state


    def propose(self, state: RyokoState, **kwargs) -> RyokoState:

        therapy = state.proposal.get("therapy","sotorasib")

        # Add suggestions from previous cycle if available

        suggestions = state.result.get("suggestions", [])

        if suggestions and kwargs.get("auto_combine", False):

            # Extract drug name before the parenthesis

            extra = suggestions[0].split("(")[0].strip().lower()

            if extra not in therapy:

                therapy = therapy + "+" + extra

                log.info(f"  [P]  Auto-combining: {therapy}")

        state.proposal = {"therapy": therapy}

        return state


    def apply(self, state: RyokoState, **kwargs) -> RyokoState:

        try:

            from treatment_loop import TreatmentLoop

            loop   = TreatmentLoop(max_cycles=kwargs.get("cycles", 10))

            result = loop.run(state.proposal["therapy"], verbose=False)

            state.result      = result

            state.k_resonance = result.get("final_k", 0.0)

            state.converged   = result.get("converged", False)

            state.cycle       = result.get("cycle", 0)

            state.regime      = self._regime(result)

            state.checkpoint()

        except ImportError:

            log.error("treatment_loop.py not found — biology domain unavailable")

        return state


    def explain(self, state: RyokoState) -> str:

        result = state.result

        lines  = [

            f"\nBiology Loop — {state.target}",

            f"  Therapy:     {state.proposal.get('therapy','')}",

            f"  K-resonance: {state.k_resonance:.4f} / {K_MAX:.4f} "

            f"({state.k_resonance/K_MAX*100:.1f}%)",

            f"  Stopped:     {result.get('stopped','')}  at cycle {state.cycle}",

        ]

        conflict = result.get("conflict", {})

        cands    = conflict.get("conflict_candidates", [])

        if cands:

            lines.append(f"  Oscillating pathways: {', '.join(cands[:4])}")

        stable_h = conflict.get("stable_high", [])

        if stable_h:

            lines.append(f"  Resistance (stable high): {', '.join(stable_h[:4])}")

        suggestions = result.get("suggestions", [])

        if suggestions:

            lines.append(f"  Pathway pressure signals (not treatment recommendations):")

            lines.append(f"  These pathways are driving resistance — consult oncology")

            lines.append(f"  for clinical translation. Toxicity and trials apply.")

            # Log scope boundary every time biology makes suggestions

            _audit("scope_boundary", "biology", {

                "boundary":    "clinical=False",

                "suggestions": suggestions,

                "note":        "outputs are pathway signals, not clinical decisions",

            })

            for s in suggestions:

                lines.append(f"    → {s}")

        return "\n".join(lines)


    def _regime(self, result: dict) -> str:

        stopped = result.get("stopped","")

        if stopped == "converged":    return "✓ converged"

        if stopped == "oscillation":  return "↯ oscillation"

        if stopped == "plateau":      return "→ plateau"

        return "↑ progressing"



class OSAdapter:

    """Wraps ryoko_scanner + ryoko_flake_gen — OS domain."""

    name = "os"


    def observe(self, state: RyokoState, **kwargs) -> RyokoState:

        target = kwargs.get("target", "")

        state.target      = target

        state.observation = {"target": target, "type": "executable_scan"}

        log.info(f"  [NP] OS: scanning {target}")

        return state


    def propose(self, state: RyokoState, **kwargs) -> RyokoState:

        try:

            from ryoko_scanner import ManifestScanner

            scanner  = ManifestScanner()

            manifest = scanner.scan(state.target)

            state.proposal = {

                "manifest":  manifest.to_dict(),

                "nix_pkgs":  manifest.nix_packages,

                "compat":    manifest.compat_mode.value,

                "initial_k": manifest.k_resonance,

            }

            state.k_resonance = manifest.k_resonance   # OSKModel.compute() reads this from manifest

            log.info(f"  [P]  OS: {len(manifest.nix_packages)} packages, "

                     f"K={manifest.k_resonance:.3f}")

        except (ImportError, Exception) as e:

            log.error(f"  [P]  OS scanner error: {e}")

        return state


    def apply(self, state: RyokoState, **kwargs) -> RyokoState:

        try:

            from ryoko_flake_gen import generate_from_manifests

            import tempfile, json as _json

            with tempfile.TemporaryDirectory() as tmp:

                mp = Path(tmp) / "manifest.json"

                mp.write_text(_json.dumps(state.proposal.get("manifest",{})))

                out = Path(kwargs.get("output", "./ryoko-generated"))

                flake = generate_from_manifests([str(mp)], str(out),

                                                name="ryoko-target")

                state.result = {

                    "flake_path": str(flake),

                    "packages":   state.proposal.get("nix_pkgs", []),

                    "apply_cmd":  f"cd {out} && sudo nixos-rebuild switch --flake .#ryoko-target",

                }

                state.converged = state.k_resonance >= K_MAX * 0.8

        except (ImportError, Exception) as e:

            log.error(f"  [K]  OS flake gen error: {e}")

        return state


    def explain(self, state: RyokoState) -> str:

        proposal = state.proposal

        result   = state.result

        lines = [

            f"\nOS Loop — {state.target}",

            f"  Mode:        {proposal.get('compat','')}",

            f"  K-resonance: {state.k_resonance:.4f} / {K_MAX:.4f} "

            f"({state.k_resonance/K_MAX*100:.1f}%)",

            f"  Packages:    {len(proposal.get('nix_pkgs',[]))}",

        ]

        if result.get("flake_path"):

            lines.append(f"  Flake:       {result['flake_path']}")

            lines.append(f"  Apply:       {result.get('apply_cmd','')}")

        if state.converged:

            lines.append("  Status:      ✓ system ready")

        else:

            lines.append("  Status:      → install packages then rescan")

        return "\n".join(lines)



class HardwareAdapter:

    """Wraps ryoko_auto_interface — hardware domain."""

    name = "hardware"


    def observe(self, state: RyokoState, **kwargs) -> RyokoState:

        state.target      = "environment"

        state.observation = {"scan": "usb+serial+network+filesystem"}

        log.info("  [NP] Hardware: scanning environment")

        return state


    def propose(self, state: RyokoState, **kwargs) -> RyokoState:

        try:

            from ryoko_auto_interface import DiscoveryModule, AdapterFactory

            config    = kwargs.get("config", {})

            discovery = DiscoveryModule(config=config)

            resources = discovery.scan()

            factory   = AdapterFactory()

            adapters  = []

            for res in resources:

                adapter = factory.create_adapter(res)

                if adapter:

                    adapters.append({

                        "name":     res.name,

                        "type":     res.type.value,

                        "security": res.security.value,

                        "describe": adapter.describe(),

                    })

            state.proposal    = {"resources": resources, "adapters": adapters}

            state.k_resonance = min(K_MAX, len(adapters) * K_MAX / 12)  # HardwareKModel

            log.info(f"  [P]  Hardware: {len(adapters)} adapters created")

        except (ImportError, Exception) as e:

            log.warning(f"  [P]  Hardware discovery: {e}")

            state.proposal = {"adapters": [], "note": str(e)}

        return state


    def apply(self, state: RyokoState, **kwargs) -> RyokoState:

        adapters = state.proposal.get("adapters", [])

        state.result    = {

            "active":    len(adapters),

            "adapters":  adapters,

            "np_array":  "12-channel ready" if adapters else "no sources",

        }

        state.converged = len(adapters) > 0

        state.regime    = "✓ sources active" if adapters else "→ no devices found"

        return state


    def explain(self, state: RyokoState) -> str:

        adapters = state.result.get("adapters", [])

        lines = [

            f"\nHardware Loop — environment scan",

            f"  Active adapters: {len(adapters)}",

            f"  K-resonance:     {state.k_resonance:.4f}",

        ]

        for a in adapters[:6]:

            sec = "✓" if a["security"]=="safe" else "⚠"

            lines.append(f"    {sec} {a['name']:<30} {a['type']}")

        if not adapters:

            lines.append("  No devices detected — check connections")

        return "\n".join(lines)



class PhysicsAdapter:

    """Wraps Eu³⁺ Foundry / adapter.py — physics domain."""

    name = "physics"


    def observe(self, state: RyokoState, **kwargs) -> RyokoState:

        target = kwargs.get("target", "BaTiO3_Eu")

        state.target      = target

        state.observation = {

            "target":   target,

            "endpoint": kwargs.get("endpoint", "http://localhost:7430"),

        }

        log.info(f"  [NP] Physics: target={target}")

        return state


    def propose(self, state: RyokoState, **kwargs) -> RyokoState:

        endpoint = state.observation.get("endpoint","http://localhost:7430")

        try:

            import requests

            r = requests.get(f"{endpoint}/state", timeout=2)

            d = r.json()

            state.proposal = {

                "R":          d.get("R", None),

                "beta":       d.get("beta", None),

                "curvature":  d.get("curvature", None),

                "endpoint":   endpoint,

            }

            log.info(f"  [P]  Physics: R={state.proposal['R']} "

                     f"β={state.proposal['beta']}")

        except Exception as e:

            log.warning(f"  [P]  Foundry offline ({e}) — using mock state")

            state.proposal = {"R":2.1,"beta":0.88,"curvature":-0.003,

                              "endpoint":endpoint,"mock":True,

                              "confidence":0.3}  # low confidence — Foundry offline

        return state


    def apply(self, state: RyokoState, **kwargs) -> RyokoState:

        p   = state.proposal

        R   = p.get("R",2.1) or 2.1

        beta= p.get("beta",0.88) or 0.88

        curv= p.get("curvature",-0.003) or -0.003


        # K-resonance from spectroscopic observables

        R_n   = max(0, min(1, (R - 0.3) / 7.7))

        b_div = 1.0 - max(0.3, min(1.0, beta))

        c_coh = max(0, min(1, -curv / 0.1))

        sat   = (1-R_n)*0.4 + (1-b_div)*0.3 + c_coh*0.3

        state.result      = {"R":R,"beta":beta,"curvature":curv,

                             "satisfaction":sat,

                             "mock":p.get("mock",False),

                             "confidence":p.get("confidence",1.0)}

        state.k_resonance = compute_k("physics", state)

        state.converged   = state.k_resonance >= K_MAX * 0.8

        state.regime      = "✓ near Tc" if c_coh > 0.7 else "→ away from transition"

        return state


    def explain(self, state: RyokoState) -> str:

        r = state.result

        lines = [

            f"\nPhysics Loop — {state.target}",

            f"  R-ratio:     {r.get('R','?'):.3f}",

            f"  β (KWW):     {r.get('beta','?'):.3f}",

            f"  Curvature:   {r.get('curvature','?'):.4f}",

            f"  K-resonance: {state.k_resonance:.4f} / {K_MAX:.4f} "

            f"({state.k_resonance/K_MAX*100:.1f}%)",

            f"  Regime:      {state.regime}",

        ]

        if r.get("mock"):

            conf = r.get("confidence", 1.0)

            lines.append(f"  ⚠ Foundry offline — mock values  "

                         f"(confidence: {conf:.0%})")

        if state.converged:

            lines.append("  → Near phase transition — run temperature sweep")

        return "\n".join(lines)



# ── DOMAIN REGISTRY ───────────────────────────────────────────

ADAPTERS = {

    "biology":  BiologyAdapter(),

    "os":       OSAdapter(),

    "hardware": HardwareAdapter(),

    "physics":  PhysicsAdapter(),

}



# ── RYOKO ORCHESTRATOR ────────────────────────────────────────


class Ryoko:

    """

    Unified entry point.


    observe → propose → apply → explain

    NP      → P       → K     → ∂


    The same verbs work across every domain.

    The K-resonance score is the common currency.

    """


    def __init__(self):

        self.state    = RyokoState()

        self.adapter  = None

        self.audit    = []


    def observe(self, domain: str, **kwargs) -> "Ryoko":

        """NP layer: scan system, identify what's needed."""

        if domain not in ADAPTERS:

            raise ValueError(f"Unknown domain: {domain}. "

                             f"Available: {list(ADAPTERS.keys())}")

        self.adapter       = ADAPTERS[domain]

        self.state         = RyokoState(domain=domain)

        self.state         = self.adapter.observe(self.state, **kwargs)

        self._log("observe")

        return self


    def propose(self, **kwargs) -> "Ryoko":

        """P layer: construct hypothesis / environment / therapy."""

        self._require_adapter()

        self.state = self.adapter.propose(self.state, **kwargs)

        self._log("propose")

        return self


    def apply(self, **kwargs) -> RyokoState:

        """K layer: execute and measure alignment."""

        self._require_adapter()

        self.state        = self.adapter.apply(self.state, **kwargs)

        self.state.cycle += 1

        self._log("apply")

        return self.state


    def explain(self, state: Optional[RyokoState] = None) -> str:

        """∂ layer: surface conflict, suggest next step."""

        self._require_adapter()

        s = state or self.state

        explanation = self.adapter.explain(s)

        print(explanation)

        return explanation


    def loop(self, domain: str, max_iterations: int = 5,

             auto_combine: bool = False, **kwargs) -> RyokoState:

        """

        Run the full observe→propose→apply cycle until convergence

        or max_iterations.

        """

        print(f"\n{'='*60}")

        print(f"  Ryoko Loop — {domain}")

        print(f"  K_max: {K_MAX:.4f}   Threshold: {K_MAX*0.8:.4f}")

        print(f"{'='*60}")


        self.observe(domain, **kwargs)

        k_prev     = 0.0

        dk_history = []


        for i in range(1, max_iterations + 1):

            print(f"\n── Iteration {i}/{max_iterations} ──")

            self.propose(auto_combine=auto_combine and i > 1)

            state = self.apply(**kwargs)


            delta_k = state.k_resonance - k_prev

            k_prev  = state.k_resonance

            if i > 1:

                dk_history.append(delta_k)


            bar_w  = 32

            filled = int(state.k_resonance / K_MAX * bar_w)

            bar    = "█" * filled + "░" * (bar_w - filled)

            dk_str = f"  ΔK: {delta_k:+.4f}" if i > 1 else ""

            print(f"  K: [{bar}] {state.k_resonance:.4f} "

                  f"({state.k_resonance/K_MAX*100:.1f}%){dk_str}")


            self.explain(state)


            if state.converged:

                print(f"\n  ✓ Converged at iteration {i}")

                break


            if state.regime and ("oscillation" in state.regime

                                  or "plateau" in state.regime):

                print(f"\n  {state.regime} — loop complete")

                break


        return self.state


    def status(self) -> Dict:

        """Report current state across all tracked loops."""

        return {

            "domain":       self.state.domain,

            "target":       self.state.target,

            "k_resonance":  self.state.k_resonance,

            "k_pct":        self.state.k_resonance / K_MAX * 100,

            "converged":    self.state.converged,

            "regime":       self.state.regime,

            "cycle":        self.state.cycle,

            "history":      self.state.history,

            # Audit summary: what the system enforced this session

            "audit": audit_summary(),

        }


    def _require_adapter(self):

        if not self.adapter:

            raise RuntimeError("Call observe() first to set domain")


    def _log(self, verb: str):

        self.audit.append({

            "ts":    time.strftime("%H:%M:%S"),

            "verb":  verb,

            "domain":self.state.domain,

            "k":     self.state.k_resonance,

        })



# ── DEMO ──────────────────────────────────────────────────────


def demo():

    print("Ryoko — Unified Entry Point Demo")

    print(f"K = φ·π·e = {K_MAX:.4f}")

    print()


    r = Ryoko()


    # ── Domain 1: Biology ─────────────────────────────────────

    print("━"*60)

    print("DOMAIN 1: Biology — KRAS G12C treatment loop")

    print("━"*60)

    r.observe("biology", target="KRAS_G12C", therapy="sotorasib")

    r.propose()

    state = r.apply(cycles=6)

    r.explain()


    print(f"\n  → Now trying combination therapy:")

    r.observe("biology", target="KRAS_G12C",

              therapy="sotorasib+alpelisib+navitoclax")

    r.propose()

    state2 = r.apply(cycles=8)

    r.explain()


    # ── Domain 2: Physics ─────────────────────────────────────

    print("\n" + "━"*60)

    print("DOMAIN 2: Physics — Eu³⁺ Foundry (mock)")

    print("━"*60)

    r.observe("physics", target="BaTiO3_Eu",

              endpoint="http://localhost:7430")

    r.propose()

    state3 = r.apply()

    r.explain()


    # ── Domain 3: OS ──────────────────────────────────────────

    print("\n" + "━"*60)

    print("DOMAIN 3: OS — executable scan (mock path)")

    print("━"*60)

    r.observe("os", target="/usr/bin/python3")

    r.propose()

    state4 = r.apply(output="/tmp/ryoko-test")

    r.explain()


    # ── Cross-domain K-resonance summary ──────────────────────

    print("\n" + "═"*60)

    print("CROSS-DOMAIN K-RESONANCE SUMMARY")

    print("─"*60)

    results = [

        ("Biology (sotorasib mono)", state),

        ("Biology (triple combo)",   state2),

        ("Physics (Eu foundry)",     state3),

        ("OS (python3 scan)",        state4),

    ]

    for name, s in results:

        pct  = s.k_resonance / K_MAX * 100

        bar  = "█" * int(pct/5) + "░" * (20-int(pct/5))

        conv = "✓" if s.converged else "·"

        print(f"  {conv} {name:<30} [{bar}] {pct:.1f}%")


    avg_k = sum(s.k_resonance for _,s in results) / len(results)

    print(f"\n  Average K across domains: {avg_k:.4f} / {K_MAX:.4f} "

          f"({avg_k/K_MAX*100:.1f}%)")

    print()

    print("  Same constant. Different substrates. Same convergence behavior.")

    print("  That's structural invariance, not aesthetic symmetry.")

    print()

    print("  Note: K-resonance uses domain-specific models (KModel protocol).")

    print("  Same ceiling (K_MAX=13.8176), different semantics per domain.")

    print("  Cross-domain comparison is structural analogy — not equivalence.")

    for d, model in K_MODELS.items():

        tag = model.describe()

        conf = "confidence-scaled" if tag.get("confidence_scaled") else "fixed-scale"

        print(f"    {d:<10} {tag['meaning'][:45]:<45}  [{conf}]")



# ── CLI ───────────────────────────────────────────────────────




def run_cli(argv=None):

    """

    Extracted CLI logic — clean, testable, no string-parsing tricks.

    Called by main_cli() (pip entry point) and __main__ block.

    """

    import sys as _sys

    if argv is not None:

        _sys.argv = [_sys.argv[0]] + list(argv)


    parser = argparse.ArgumentParser(

        description="Ryoko — unified NP/P/K entry point",

        formatter_class=argparse.RawDescriptionHelpFormatter,

        epilog="""

Examples:

  ryoko --demo

  ryoko --domain biology --

In [ ]:
#!/usr/bin/env python3

"""

ryoko_audit_verify.py — Standalone Audit Verification Tool

============================================================

Reads a RyokoSeven audit log (JSONL), recomputes the SHA-256

hash chain, and reports any inconsistencies.


Intentionally standalone — no Ryoko imports.

An auditor with only this file and the JSONL log can verify

integrity without trusting the system that wrote the log.


Usage:

  python3 ryoko_audit_verify.py audit_logs/ryoko_audit.jsonl

  python3 ryoko_audit_verify.py audit_logs/ryoko_audit.jsonl --verbose

  python3 ryoko_audit_verify.py audit_logs/ryoko_audit.jsonl --events scope_boundary

  python3 ryoko_audit_verify.py audit_logs/ryoko_audit.jsonl --since 2026-03-18


Exit codes:

  0 — chain intact, no tampering detected

  1 — chain broken (tampering, deletion, or modification detected)

  2 — file not found or unreadable

"""


import sys

import json

import hashlib

import argparse

from pathlib import Path

from datetime import datetime, timezone



# ── CHAIN VERIFICATION ────────────────────────────────────────


def verify_chain(entries: list) -> dict:

    """

    Recompute SHA-256 hash chain from scratch.

    Returns verification report.


    Chain rule:

      hash(entry_n) = sha256( hash(entry_{n-1}) + json(entry_n without hash/prev) )

      hash(entry_0) = sha256( "genesis" + json(entry_0 without hash/prev) )

    """

    if not entries:

        return {

            "valid":   True,

            "length":  0,

            "note":    "empty log",

            "events":  {},

        }


    prev_hash = "genesis"

    event_counts = {}


    for i, entry in enumerate(entries):

        ev = entry.get("event", "?")

        event_counts[ev] = event_counts.get(ev, 0) + 1


        stored_hash = entry.get("hash", "")

        stored_prev = entry.get("prev", "")


        # Check prev pointer

        if stored_prev != prev_hash:

            return {

                "valid":          False,

                "length":         len(entries),

                "broken_at":      i,

                "ts":             entry.get("ts", "?"),

                "event":          ev,

                "reason":         "prev hash mismatch — entry deleted, inserted, or reordered",

                "expected_prev":  prev_hash[:20] + "...",

                "found_prev":     str(stored_prev)[:20] + "...",

                "events_before":  event_counts,

            }


        # Recompute hash from content (excluding chain fields)

        check = {k: v for k, v in entry.items() if k not in ("hash", "prev")}

        payload  = json.dumps(check, sort_keys=True)

        expected = hashlib.sha256((prev_hash + payload).encode()).hexdigest()


        if expected != stored_hash:

            return {

                "valid":       False,

                "length":      len(entries),

                "broken_at":   i,

                "ts":          entry.get("ts", "?"),

                "event":       ev,

                "reason":      "content hash mismatch — entry was modified after writing",

                "events_before": event_counts,

            }


        prev_hash = stored_hash


    return {

        "valid":       True,

        "length":      len(entries),

        "final_hash":  prev_hash,

        "events":      event_counts,

        "note":        "chain intact — no modifications detected",

    }



# ── LOADER ────────────────────────────────────────────────────


def load_log(path: Path) -> list:

    entries = []

    errors  = []

    with open(path) as f:

        for lineno, line in enumerate(f, 1):

            line = line.strip()

            if not line:

                continue

            try:

                entries.append(json.loads(line))

            except json.JSONDecodeError as e:

                errors.append(f"Line {lineno}: {e}")

    return entries, errors



# ── FILTERS ───────────────────────────────────────────────────


def filter_entries(entries: list, event: str = None,

                   since: str = None, domain: str = None) -> list:

    result = entries

    if event:

        result = [e for e in result if e.get("event") == event]

    if domain:

        result = [e for e in result if e.get("domain") == domain]

    if since:

        try:

            cutoff = datetime.fromisoformat(since.replace("Z","")).replace(

                tzinfo=timezone.utc)

            result = [e for e in result

                      if datetime.fromisoformat(

                          e.get("ts","").replace("Z","")).replace(

                          tzinfo=timezone.utc) >= cutoff]

        except ValueError:

            print(f"  ⚠ Invalid --since format: {since}  (use YYYY-MM-DD or ISO 8601)")

    return result



# ── REPORT PRINTER ────────────────────────────────────────────


def print_report(result: dict, entries: list, verbose: bool = False):

    if result["valid"]:

        print(f"  ✓ CHAIN INTACT")

        print(f"    Entries verified: {result['length']}")

        print(f"    Final hash:       {result.get('final_hash','?')[:32]}...")

        print(f"    Events:")

        for ev, count in sorted(result.get("events",{}).items()):

            marker = {

                "scope_boundary":    "  (system declared limits)",

                "comparison_refused":"  (invalid comparison blocked)",

                "confidence_drop":   "  (uncertainty recorded)",

                "unknown_domain":    "  (unregistered domain)",

            }.get(ev, "")

            print(f"      {ev:<25} {count:>4}{marker}")

    else:

        print(f"  ✗ CHAIN BROKEN")

        print(f"    Broken at entry:  {result['broken_at']}")

        print(f"    Timestamp:        {result.get('ts','?')}")

        print(f"    Event type:       {result.get('event','?')}")

        print(f"    Reason:           {result.get('reason','?')}")

        if "expected_prev" in result:

            print(f"    Expected prev:    {result['expected_prev']}")

            print(f"    Found prev:       {result['found_prev']}")

        print()

        print(f"    This indicates the log was modified after writing.")

        print(f"    The entries before position {result['broken_at']} were:")

        for ev, count in sorted(result.get("events_before",{}).items()):

            print(f"      {ev:<25} {count:>4}")


    if verbose and entries:

        print()

        print("  ENTRIES:")

        print(f"  {'#':<4} {'Timestamp':<22} {'Event':<25} {'Domain':<12} {'Hash'}")

        print(f"  {'─'*80}")

        for i, e in enumerate(entries):

            h = e.get("hash","?")[:12] + "..."

            print(f"  {i:<4} {e.get('ts','?'):<22} "

                  f"{e.get('event','?'):<25} "

                  f"{e.get('domain','?'):<12} {h}")

            # Show key detail fields

            skip = {"ts","event","domain","hash","prev"}

            for k, v in e.items():

                if k not in skip:

                    val = str(v)

                    if len(val) > 60:

                        val = val[:57] + "..."

                    print(f"       {k}: {val}")



# ── MAIN ─────────────────────────────────────────────────────


def main():

    parser = argparse.ArgumentParser(

        description="Verify RyokoSeven audit log chain integrity",

        formatter_class=argparse.RawDescriptionHelpFormatter,

        epilog="""

Exit codes:

  0  chain intact

  1  chain broken (tampering detected)

  2  file error


Examples:

  python3 ryoko_audit_verify.py audit_logs/ryoko_audit.jsonl

  python3 ryoko_audit_verify.py ryoko_audit.jsonl --verbose

  python3 ryoko_audit_verify.py ryoko_audit.jsonl --events comparison_refused

  python3 ryoko_audit_verify.py ryoko_audit.jsonl --since 2026-03-18

  python3 ryoko_audit_verify.py ryoko_audit.jsonl --domain biology --verbose

        """)

    parser.add_argument("log_file",

                        help="Path to ryoko_audit.jsonl")

    parser.add_argument("--verbose", "-v", action="store_true",

                        help="Show all entries with detail")

    parser.add_argument("--events", "-e",

                        help="Filter: only show this event type")

    parser.add_argument("--since", "-s",

                        help="Filter: entries from this date (YYYY-MM-DD)")

    parser.add_argument("--domain", "-d",

                        help="Filter: only this domain")

    parser.add_argument("--summary", action="store_true",

                        help="Print summary only, no chain details")

    args = parser.parse_args()


    path = Path(args.log_file)

    if not path.exists():

        print(f"✗ File not found: {path}")

        sys.exit(2)


    # Load

    entries, parse_errors = load_log(path)

    if parse_errors:

        print(f"⚠ Parse errors in {path}:")

        for e in parse_errors[:5]:

            print(f"  {e}")


    print(f"{'='*60}")

    print(f"  RyokoSeven Audit Verification")

    print(f"  File:    {path}")

    print(f"  Entries: {len(entries)}")

    print(f"{'='*60}")


    # Always verify full chain (no filters — filtering breaks chain order)

    result = verify_chain(entries)


    if not args.summary:

        print()

        # Apply filters for display only (after chain verification)

        display = filter_entries(entries,

                                  event=args.events,

                                  since=args.since,

                                  domain=args.domain)

        if args.events or args.since or args.domain:

            print(f"  Filter: {len(display)}/{len(entries)} entries match")

            print()

            print_report(result, display, verbose=args.verbose)

        else:

            print_report(result, entries, verbose=args.verbose)


    print(f"\n{'='*60}")

    status = "VERIFIED ✓" if result["valid"] else "TAMPERED ✗"

    print(f"  {status}")

    print(f"{'='*60}")


    sys.exit(0 if result["valid"] else 1)



if __name__ == "__main__":

    main()

In [ ]:
"""

RyokoSeven — TreatmentLoop

===========================

Temporal recursion layer for the KRAS G12C pipeline.


Turns the static P-layer scorer into a dynamical system observer:

  score(molecule) → observe(state, therapy, time)


The same four-regime signal that works for NixOS packages

works for biological pathways:


  ▲ core pathway suppressed      → drug hitting target

  ↑ fine-tuning                  → combination refinement

  → plateau                      → adaptive resistance locked in

  ↯ oscillation                  → competing attractor (feedback loop)


When oscillation fires, the conflict surface is a set of pathways —

not packages. The partition into Path A / Path B becomes:

  sensitive phenotype vs resistant phenotype.


That's mechanism-derived combination therapy discovery.


Usage:

  python3 treatment_loop.py                      # demo

  python3 treatment_loop.py --target KRAS        # KRAS G12C loop

  python3 treatment_loop.py --combo "sotorasib+adagrasib"

"""


import json

import math

import logging

import time

from dataclasses import dataclass, field

from pathlib import Path

from typing import List, Dict, Optional, Tuple

import numpy as np


log = logging.getLogger("ryoko.treatment")

logging.basicConfig(level=logging.INFO,

                    format="%(asctime)s %(levelname)-7s %(message)s",

                    datefmt="%H:%M:%S")


PHI   = (1 + math.sqrt(5)) / 2

K_MAX = PHI * math.pi * math.e   # 13.8176

CONVERGENCE_THRESHOLD = K_MAX * 0.8   # 11.054


AUDIT_DIR = Path("./audit_logs")

AUDIT_DIR.mkdir(exist_ok=True)


# ── BIOLOGICAL STATE MODEL ────────────────────────────────────

# Each state is a pathway activity vector.

# This is what the oscillation detector runs on —

# the same logic as package flip-rates, but per pathway.


# NSCLC KRAS G12C pathway map

# Activity ∈ [0,1]: 0 = suppressed, 1 = maximally active

KRAS_PATHWAYS = {

    # Oncogenic drivers (high activity = bad)

    "KRAS_G12C":     "primary oncogene — covalent inhibitor target",

    "MAPK_ERK":      "proliferation signal downstream of KRAS",

    "PI3K_AKT":      "survival signal — bypass route",

    "mTOR":          "growth/metabolism — downstream of PI3K",

    "MYC":           "transcription factor — proliferation",

    # Tumor suppressors (high activity = good)

    "TP53":          "apoptosis gating",

    "RB1":           "cell cycle brake",

    "PTEN":          "PI3K antagonist",

    # Resistance mechanisms

    "EGFR_bypass":   "receptor-level KRAS bypass",

    "YAP1":          "Hippo pathway — KRAS-independent growth",

    "NF1_loss":      "RAS activator — amplifies residual KRAS",

    # Apoptosis / viability

    "BCL2_family":   "anti-apoptotic (high = resistant)",

    "caspase_signal":"pro-apoptotic (high = dying tumor)",

}


@dataclass

class BiologicalState:

    """

    Snapshot of tumor pathway activity at one timepoint.

    Equivalent to EnvironmentSpec in the OS loop —

    the structured representation of system state.

    """

    timepoint:    int   = 0

    therapy:      str   = "untreated"

    pathways:     Dict[str, float] = field(default_factory=dict)

    viability:    float = 1.0    # tumor cell viability [0,1]

    k_resonance:  float = 0.0   # how close to healthy state

    notes:        List[str] = field(default_factory=list)


    def to_dict(self) -> dict:

        return {

            "timepoint":   self.timepoint,

            "therapy":     self.therapy,

            "pathways":    self.pathways,

            "viability":   self.viability,

            "k_resonance": self.k_resonance,

            "notes":       self.notes,

        }



# ── TUMOR MODEL ───────────────────────────────────────────────


class TumorModel:

    """

    Simplified dynamical model of NSCLC KRAS G12C tumor response.


    Not a full ODE system — a structured approximation that

    captures the key biological behaviors:

      - Initial drug response (KRAS suppression)

      - Adaptive resistance (PI3K/EGFR bypass activation)

      - Feedback coupling (PTEN/mTOR/BCL2)


    Real version would use:

      - ODE system (Chen et al. 2019 KRAS network)

      - TCGA expression data for initialization

      - PharmacoDB drug response curves

    """


    def __init__(self, seed: int = 42):

        self.rng = np.random.RandomState(seed)


    def initial_state(self, mutation_burden: float = 0.8) -> BiologicalState:

        """Untreated KRAS G12C tumor — high oncogene activity."""

        state = BiologicalState(timepoint=0, therapy="untreated")

        state.pathways = {

            "KRAS_G12C":    0.92,   # constitutively active

            "MAPK_ERK":     0.85,

            "PI3K_AKT":     0.60,

            "mTOR":         0.70,

            "MYC":          0.80,

            "TP53":         0.25,   # suppressed by tumor

            "RB1":          0.30,

            "PTEN":         0.35,

            "EGFR_bypass":  0.10,   # not yet activated

            "YAP1":         0.20,

            "NF1_loss":     0.15,

            "BCL2_family":  0.75,   # anti-apoptotic high

            "caspase_signal":0.10,

        }

        state.viability   = 0.95

        state.k_resonance = self._compute_k(state)

        return state


    def apply_therapy(self, state: BiologicalState,

                      therapy: str,

                      cycle: int) -> BiologicalState:

        """

        Apply drug perturbation and evolve state one cycle.

        Models both direct drug effects and adaptive responses.

        """

        new = BiologicalState(

            timepoint = state.timepoint + 1,

            therapy   = therapy,

            pathways  = dict(state.pathways),  # copy

        )

        noise = lambda scale=0.03: self.rng.normal(0, scale)


        # ── Drug effects (direct) ─────────────────────────

        if "sotorasib" in therapy or "amg510" in therapy.lower():

            # Covalent KRAS G12C inhibitor — direct suppression

            new.pathways["KRAS_G12C"]   *= 0.75 + noise()

            new.pathways["MAPK_ERK"]    *= 0.80 + noise()


        if "adagrasib" in therapy or "mrtx849" in therapy.lower():

            # Second KRAS G12C inhibitor — similar but distinct binding

            new.pathways["KRAS_G12C"]   *= 0.72 + noise()

            new.pathways["MAPK_ERK"]    *= 0.78 + noise()


        if "erlotinib" in therapy or "egfr" in therapy.lower():

            # EGFR inhibitor (for bypass suppression)

            new.pathways["EGFR_bypass"] *= 0.40 + noise()


        if "alpelisib" in therapy or "pi3k" in therapy.lower():

            # PI3K inhibitor

            new.pathways["PI3K_AKT"]    *= 0.55 + noise()

            new.pathways["mTOR"]        *= 0.65 + noise()


        if "navitoclax" in therapy or "bcl2" in therapy.lower():

            # BCL-2/BCL-XL inhibitor — pro-apoptotic

            new.pathways["BCL2_family"] *= 0.60 + noise()

            new.pathways["caspase_signal"] = min(1.0,

                new.pathways["caspase_signal"] * 1.4 + noise())


        # ── Adaptive resistance (cycle-dependent) ────────

        # The tumor adapts — this is what drives oscillation.

        # Stronger suppression of KRAS → stronger bypass activation.

        kras_suppression = 1.0 - new.pathways["KRAS_G12C"]


        if cycle >= 2:

            # PI3K bypass kicks in when KRAS is suppressed

            bypass_pressure = kras_suppression * 0.35

            new.pathways["PI3K_AKT"]    = min(1.0,

                new.pathways["PI3K_AKT"] + bypass_pressure + noise(0.02))

            new.pathways["EGFR_bypass"] = min(1.0,

                new.pathways["EGFR_bypass"] + kras_suppression * 0.20 + noise(0.02))


        if cycle >= 3:

            # YAP1 (Hippo pathway) activates as MAPK-independent survival

            new.pathways["YAP1"] = min(1.0,

                new.pathways["YAP1"] + kras_suppression * 0.25 + noise(0.02))

            # mTOR partially recovers via AKT

            new.pathways["mTOR"] = min(1.0,

                new.pathways["mTOR"] * (1 + new.pathways["PI3K_AKT"] * 0.15))


        if cycle >= 4:

            # BCL-2 upregulation as survival mechanism

            new.pathways["BCL2_family"] = min(1.0,

                new.pathways["BCL2_family"] + 0.08 + noise(0.02))


        # ── Tumor suppressor recovery (drug-mediated) ────

        # When oncogenes are suppressed, suppressors can recover

        if new.pathways["KRAS_G12C"] < 0.5:

            new.pathways["TP53"] = min(0.9,

                new.pathways["TP53"] + 0.05 + noise(0.01))

            new.pathways["RB1"]  = min(0.9,

                new.pathways["RB1"]  + 0.04 + noise(0.01))


        # ── Clip all pathways to [0,1] ────────────────────

        for k in new.pathways:

            new.pathways[k] = float(np.clip(new.pathways[k], 0.0, 1.0))


        # ── Compute viability and K-resonance ────────────

        new.viability   = self._compute_viability(new)

        new.k_resonance = self._compute_k(new)


        return new


    def _compute_viability(self, state: BiologicalState) -> float:

        """

        Tumor viability: weighted combination of pathway activities.

        High oncogene activity + low suppressor activity = high viability.

        K-resonance score from KRAS pipeline would replace this in

        the real integration.

        """

        oncogene_drive = (

            state.pathways["KRAS_G12C"]  * 0.25 +

            state.pathways["MAPK_ERK"]   * 0.20 +

            state.pathways["PI3K_AKT"]   * 0.15 +

            state.pathways["MYC"]        * 0.10 +

            state.pathways["BCL2_family"]* 0.10 +

            state.pathways["YAP1"]       * 0.08 +

            state.pathways["EGFR_bypass"]* 0.07 +

            state.pathways["mTOR"]       * 0.05

        )

        suppressor_brake = (

            state.pathways["TP53"]         * 0.40 +

            state.pathways["RB1"]          * 0.30 +

            state.pathways["PTEN"]         * 0.20 +

            state.pathways["caspase_signal"]* 0.10

        )

        viability = oncogene_drive * (1.0 - 0.6 * suppressor_brake)

        return float(np.clip(viability, 0.0, 1.0))


    def _compute_k(self, state: BiologicalState) -> float:

        """

        K-resonance for biological state.

        Target: KRAS suppressed, suppressors active, bypasses quiet.

        Score measures distance from healthy equilibrium.

        """

        # Healthy target: oncogenes low, suppressors high

        targets = {

            "KRAS_G12C":     0.05,   # target: nearly off

            "MAPK_ERK":      0.15,

            "PI3K_AKT":      0.20,

            "mTOR":          0.20,

            "MYC":           0.15,

            "TP53":          0.85,   # target: active

            "RB1":           0.80,

            "PTEN":          0.75,

            "EGFR_bypass":   0.05,   # target: quiet

            "YAP1":          0.10,

            "NF1_loss":      0.05,

            "BCL2_family":   0.15,   # target: low

            "caspase_signal":0.70,   # target: active (apoptosis)

        }

        weights = {

            "KRAS_G12C":0.20,"MAPK_ERK":0.12,"PI3K_AKT":0.10,

            "mTOR":0.06,"MYC":0.08,"TP53":0.12,"RB1":0.08,

            "PTEN":0.06,"EGFR_bypass":0.06,"YAP1":0.04,

            "NF1_loss":0.02,"BCL2_family":0.03,"caspase_signal":0.03,

        }

        distance = sum(

            weights[pw] * abs(state.pathways[pw] - targets[pw])

            for pw in targets

        )

        # Convert distance to K-resonance: 0 distance = K_MAX

        satisfaction = max(0.0, 1.0 - distance / 0.5)

        return float(K_MAX * satisfaction)



# ── OSCILLATION DETECTOR (biological) ────────────────────────


def detect_oscillation(history: List[BiologicalState]) -> Tuple[bool, str]:

    """

    Detect oscillation in K-resonance trajectory.

    Same logic as the NixOS convergence loop —

    alternating signs + shrinking magnitude.

    Returns (is_oscillating, reason).

    """

    if len(history) < 3:

        return False, ""


    k_vals = [s.k_resonance for s in history]

    deltas = [k_vals[i+1] - k_vals[i] for i in range(len(k_vals)-1)]


    if len(deltas) < 2:

        return False, ""


    significant = [d for d in deltas[-3:] if abs(d) > 0.05]

    if len(significant) < 2:

        return False, ""


    alternates = all(significant[i] * significant[i+1] < 0

                     for i in range(len(significant)-1))

    shrinking  = all(abs(significant[i]) >= abs(significant[i+1]) * 0.7

                     for i in range(len(significant)-1))


    if alternates and shrinking:

        return True, (f"ΔK alternating: {[f'{d:+.3f}' for d in deltas[-3:]]}")

    return False, ""



def classify_delta(delta_k: float, dk_history: List[float]) -> str:

    """Four-regime classification — same as OS loop."""

    osc, _ = detect_oscillation_from_deltas(dk_history)

    if osc:            return "↯ oscillation — feedback resistance"

    if delta_k >  0.8: return "▲ pathway suppressed — drug on target"

    if delta_k >  0.1: return "↑ fine-tuning"

    if delta_k < -0.1: return "▼ regression — resistance emerging"

    return             "→ plateau — adaptive resistance locked in"



def detect_oscillation_from_deltas(deltas: List[float]) -> Tuple[bool, str]:

    sig = [d for d in deltas[-3:] if abs(d) > 0.05]

    if len(sig) < 2:

        return False, ""

    alt = all(sig[i]*sig[i+1] < 0 for i in range(len(sig)-1))

    shr = all(abs(sig[i]) >= abs(sig[i+1])*0.7 for i in range(len(sig)-1))

    return alt and shr, str(sig)



# ── CONFLICT SURFACE (pathway version) ───────────────────────


def compute_pathway_conflict_surface(

        history: List[BiologicalState]) -> Dict:

    """

    Identify which pathways are oscillating across treatment cycles.

    Equivalent to package flip-rate analysis in the OS loop.


    Returns:

      conflict_candidates: pathways near 50% activity flip rate

      stable_oncogenes:    always high (unconquered)

      stable_suppressors:  consistently recovered

      path_a / path_b:     competing phenotype descriptions

    """

    if len(history) < 3:

        return {}


    pathways = list(history[0].pathways.keys())

    n = len(history)


    # For each pathway: track activity across cycles

    # "flip" = crosses threshold 0.5 between consecutive cycles

    flip_counts = {}

    mean_activity = {}

    for pw in pathways:

        acts   = [s.pathways[pw] for s in history]

        flips  = sum(1 for i in range(len(acts)-1)

                     if (acts[i] > 0.5) != (acts[i+1] > 0.5))

        flip_counts[pw]  = flips / max(len(acts)-1, 1)

        mean_activity[pw]= float(np.mean(acts))


    # Conflict candidates: oscillating near threshold

    conflict_candidates = sorted(

        [pw for pw, fr in flip_counts.items() if 0.2 <= fr <= 0.8],

        key=lambda p: abs(flip_counts[p] - 0.5)

    )


    # Stable high (resistance) vs stable low (suppressed)

    stable_high = [pw for pw, fr in flip_counts.items()

                   if fr < 0.2 and mean_activity[pw] > 0.6]

    stable_low  = [pw for pw, fr in flip_counts.items()

                   if fr < 0.2 and mean_activity[pw] < 0.4]


    # Partition history by K-resonance (above/below median)

    k_vals  = [s.k_resonance for s in history]

    k_med   = sorted(k_vals)[n//2]

    high_k  = [s for s in history if s.k_resonance >= k_med]

    low_k   = [s for s in history if s.k_resonance <  k_med]


    path_a_signature = {}  # high-K states (sensitive phenotype)

    path_b_signature = {}  # low-K states (resistant phenotype)


    if high_k and low_k:

        for pw in pathways:

            path_a_signature[pw] = float(np.mean([s.pathways[pw] for s in high_k]))

            path_b_signature[pw] = float(np.mean([s.pathways[pw] for s in low_k]))


    return {

        "conflict_candidates": conflict_candidates,

        "stable_high":         stable_high,

        "stable_low":          stable_low,

        "flip_rates":          flip_counts,

        "mean_activity":       mean_activity,

        "path_a_signature":    path_a_signature,  # sensitive

        "path_b_signature":    path_b_signature,  # resistant

    }



# ── COMBINATION THERAPY SUGGESTER ────────────────────────────


def suggest_combination(conflict: Dict,

                         current_therapy: str) -> List[str]:

    """

    From the conflict surface, suggest what to add to break resistance.


    Logic:

      - If PI3K_AKT is oscillating → add PI3K inhibitor

      - If EGFR_bypass is oscillating → add EGFR inhibitor

      - If YAP1 is oscillating → add Verteporfin (YAP inhibitor)

      - If BCL2_family is stable high → add BCL2 inhibitor


    This is mechanism-derived, not rule-based:

    the pathway that's oscillating tells you the resistance mechanism.

    """

    candidates = conflict.get("conflict_candidates", [])

    stable_high = conflict.get("stable_high", [])

    suggestions = []


    for pw in candidates + stable_high:

        if pw == "PI3K_AKT" and "alpelisib" not in current_therapy:

            suggestions.append("alpelisib (PI3K inhibitor — suppresses AKT bypass)")

        elif pw == "EGFR_bypass" and "erlotinib" not in current_therapy:

            suggestions.append("erlotinib (EGFR inhibitor — closes receptor bypass)")

        elif pw == "YAP1":

            suggestions.append("verteporfin (YAP1 inhibitor — Hippo pathway)")

        elif pw == "BCL2_family" and "navitoclax" not in current_therapy:

            suggestions.append("navitoclax (BCL-2/XL inhibitor — pro-apoptotic)")

        elif pw == "mTOR":

            suggestions.append("everolimus (mTOR inhibitor — downstream of PI3K)")

        elif pw == "MAPK_ERK" and "cobimetinib" not in current_therapy:

            suggestions.append("cobimetinib (MEK inhibitor — MAPK pathway)")


    return suggestions[:3]   # top 3



# ── TREATMENT LOOP ────────────────────────────────────────────


class TreatmentLoop:

    """

    Temporal recursion on the KRAS G12C pipeline.


    scan(state) → generate(therapy) → apply → rescan → converge

         ↑___________________________________|


    Stops at convergence (K ≥ threshold) or

    when oscillation fires (conflict surface computed).

    """


    def __init__(self, max_cycles: int = 10):

        self.model     = TumorModel()

        self.max_cycles= max_cycles


    def run(self, therapy: str,

            initial_state: Optional[BiologicalState] = None,

            verbose: bool = True) -> Dict:

        """

        Run treatment loop.

        Returns full history + conflict analysis + suggestions.

        """

        state  = initial_state or self.model.initial_state()

        history= [state]

        k_prev = state.k_resonance

        dk_history: List[float] = []


        if verbose:

            print(f"\n{'='*62}")

            print(f"  RyokoSeven TreatmentLoop — KRAS G12C")

            print(f"  Therapy:   {therapy}")

            print(f"  K_max:     {K_MAX:.4f}   Threshold: {CONVERGENCE_THRESHOLD:.4f}")

            print(f"{'='*62}")

            self._print_state(state, "  [t=0]  Untreated baseline", is_first=True)


        for cycle in range(1, self.max_cycles + 1):

            # Apply therapy → evolve state

            state  = self.model.apply_therapy(state, therapy, cycle)

            history.append(state)


            # ΔK

            delta_k = state.k_resonance - k_prev

            k_prev  = state.k_resonance

            dk_history.append(delta_k)


            signal  = classify_delta(delta_k, dk_history)


            if verbose:

                self._print_state(state,

                    f"  [t={cycle}]  {signal}",

                    delta_k=delta_k)


            # Convergence check

            if state.k_resonance >= CONVERGENCE_THRESHOLD:

                if verbose:

                    print(f"\n  ✓ CONVERGED at cycle {cycle}")

                    print(f"    K = {state.k_resonance:.4f} ≥ {CONVERGENCE_THRESHOLD:.4f}")

                    print(f"    Tumor viability: {state.viability:.3f}")

                break


            # Oscillation check (after cycle 3)

            if cycle >= 3:

                osc, reason = detect_oscillation(history[-4:])

                if osc:

                    conflict = compute_pathway_conflict_surface(history)

                    combos   = suggest_combination(conflict, therapy)

                    if verbose:

                        self._print_oscillation(conflict, combos, therapy, reason)

                    return {

                        "converged":    False,

                        "stopped":      "oscillation",

                        "cycle":        cycle,

                        "final_k":      state.k_resonance,

                        "history":      [s.to_dict() for s in history],

                        "conflict":     conflict,

                        "suggestions":  combos,

                    }


            # Plateau check

            if cycle >= 4 and all(abs(d) < 0.05 for d in dk_history[-3:]):

                conflict = compute_pathway_conflict_surface(history)

                combos   = suggest_combination(conflict, therapy)

                if verbose:

                    print(f"\n  → Plateau at cycle {cycle} — adaptive resistance")

                    print(f"     K stabilized at {state.k_resonance:.4f}")

                    self._print_suggestions(combos, therapy)

                return {

                    "converged":  False,

                    "stopped":    "plateau",

                    "cycle":      cycle,

                    "final_k":    state.k_resonance,

                    "history":    [s.to_dict() for s in history],

                    "conflict":   conflict,

                    "suggestions":combos,

                }


        conflict = compute_pathway_conflict_surface(history)

        return {

            "converged":  state.k_resonance >= CONVERGENCE_THRESHOLD,

            "stopped":    "converged" if state.k_resonance >= CONVERGENCE_THRESHOLD

                          else "max_cycles",

            "cycle":      len(history) - 1,

            "final_k":    state.k_resonance,

            "history":    [s.to_dict() for s in history],

            "conflict":   conflict,

            "suggestions":suggest_combination(conflict, therapy),

        }


    def _print_state(self, state: BiologicalState, label: str,

                     delta_k: float = 0.0, is_first: bool = False):

        k_pct  = state.k_resonance / K_MAX * 100

        bar_w  = 32

        filled = int(k_pct / 100 * bar_w)

        bar    = "█" * filled + "░" * (bar_w - filled)


        print(f"\n{label}")

        print(f"    K-resonance: [{bar}] {state.k_resonance:.4f} ({k_pct:.1f}%)"

              + (f"  ΔK: {delta_k:+.4f}" if not is_first else ""))

        print(f"    Viability:   {state.viability:.3f}")


        # Show key pathway activities

        key = ["KRAS_G12C","MAPK_ERK","PI3K_AKT","EGFR_bypass","YAP1",

               "TP53","BCL2_family","caspase_signal"]

        acts = "  ".join(f"{p.split('_')[0]}={state.pathways[p]:.2f}"

                         for p in key if p in state.pathways)

        print(f"    Pathways:    {acts}")


    def _print_oscillation(self, conflict: Dict, combos: List[str],

                            therapy: str, reason: str):

        print(f"\n  ↯  Oscillation detected — resistance feedback active")

        print(f"     {reason}")


        cands = conflict.get("conflict_candidates", [])

        if cands:

            flip = conflict.get("flip_rates", {})

            print(f"\n     Oscillating pathways (conflict surface):")

            for pw in cands[:5]:

                fr = flip.get(pw, 0)

                mean = conflict.get("mean_activity",{}).get(pw,0)

                print(f"       ↯ {pw:<20} flip rate={fr:.2f}  mean={mean:.2f}")


        stable_h = conflict.get("stable_high", [])

        if stable_h:

            print(f"\n     Resistance mechanisms (stable high):")

            for pw in stable_h[:4]:

                mean = conflict.get("mean_activity",{}).get(pw,0)

                print(f"       ▶ {pw:<20} mean={mean:.2f}  (unconquered)")


        pa = conflict.get("path_a_signature", {})

        pb = conflict.get("path_b_signature", {})

        if pa and pb:

            # Find what distinguishes sensitive vs resistant phenotype

            divergent = sorted(

                [pw for pw in pa if abs(pa[pw]-pb[pw]) > 0.15],

                key=lambda p: abs(pa[p]-pb[p]), reverse=True

            )[:4]

            if divergent:

                print(f"\n     Sensitive vs resistant phenotype:")

                print(f"     {'Pathway':<22} {'Sensitive (A)':>14} {'Resistant (B)':>14}")

                print(f"     {'─'*52}")

                for pw in divergent:

                    marker = "←target" if pa[pw] < pb[pw] else ""

                    print(f"     {pw:<22} {pa[pw]:>14.3f} {pb[pw]:>14.3f}  {marker}")


        self._print_suggestions(combos, therapy)


    def _print_suggestions(self, combos: List[str], therapy: str):

        if combos:

            print(f"\n     Mechanism-derived combination suggestions:")

            for i, c in enumerate(combos, 1):

                print(f"       [{i}] {c}")

            print(f"\n     Current therapy: {therapy}")

            print(f"     Add one suggestion and rerun to test new equilibrium.")



# ── DEMO ──────────────────────────────────────────────────────


def demo():

    print("RyokoSeven TreatmentLoop — KRAS G12C Demo")

    print(f"K_max = {K_MAX:.4f}   Threshold = {CONVERGENCE_THRESHOLD:.4f}")


    loop = TreatmentLoop(max_cycles=10)


    # ── Scenario 1: Sotorasib monotherapy → resistance ────────

    print("\n" + "─"*62)

    print("SCENARIO 1: Sotorasib monotherapy")

    print("Expected: initial response → PI3K/EGFR bypass → oscillation")

    r1 = loop.run("sotorasib")


    # ── Scenario 2: Sotorasib + PI3K inhibitor ────────────────

    print("\n" + "─"*62)

    print("SCENARIO 2: Sotorasib + Alpelisib (PI3K inhibitor)")

    print("Expected: deeper suppression, slower resistance")

    r2 = loop.run("sotorasib+alpelisib")


    # ── Scenario 3: Triple combination ────────────────────────

    print("\n" + "─"*62)

    print("SCENARIO 3: Sotorasib + Alpelisib + Navitoclax (BCL-2i)")

    print("Expected: convergence or near-convergence")

    r3 = loop.run("sotorasib+alpelisib+navitoclax")


    # ── Summary ───────────────────────────────────────────────

    print("\n" + "═"*62)

    print("TREATMENT LOOP SUMMARY")

    print("─"*62)

    print(f"{'Therapy':<40} {'Stopped':>10} {'Final K':>8} {'Cycle':>6}")

    print("─"*62)

    for r, name in [(r1,"Sotorasib mono"),

                    (r2,"Soto+Alpelisib"),

                    (r3,"Soto+Alpel+Navi")]:

        print(f"  {name:<38} {r['stopped']:>10} "

              f"{r['final_k']:>8.3f} {r['cycle']:>6}")

    print("─"*62)

    print(f"\nConclusion:")

    print(f"  Monotherapy hits a resistance wall (oscillation).")

    print(f"  The conflict surface names the bypass (PI3K/EGFR).")

    print(f"  Combination targeting those pathways breaks resistance.")

    print(f"  That's mechanism-derived therapy, not trial-and-error.")


    # Save audit

    audit = {

        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),

        "scenarios": [

            {"therapy":"sotorasib",                "result":r1},

            {"therapy":"sotorasib+alpelisib",      "result":r2},

            {"therapy":"sotorasib+alpelisib+navitoclax","result":r3},

        ]

    }

    log_file = AUDIT_DIR / f"{time.strftime('%Y-%m-%d')}_treatment_loop.json"

    log_file.write_text(json.dumps(audit, indent=2))

    print(f"\nAudit saved: {log_file}")



if __name__ == "__main__":

    import argparse

    parser = argparse.ArgumentParser(description="RyokoSeven TreatmentLoop")

    parser.add_argument("--target", default="KRAS",   help="Target gene")

    parser.add_argument("--combo",  default=None,     help="Therapy combination")

    parser.add_argument("--cycles", type=int,default=10)

    parser.add_argument("--demo",   action="store_true")

    args = parser.parse_args()


    logging.disable(logging.CRITICAL)  # quiet for clean output

    demo() if args.demo or not args.combo else \

        TreatmentLoop(args.cycles).run(args.combo)

In [ ]:
"""

RyokoSeven — KRAS G12C Minimal Executable

==========================================

Self-contained scoring pipeline for NSCLC KRAS G12C.

Runs immediately with no external API or model dependencies.

All modules produce physically-grounded mock outputs with correct

uncertainty propagation and K-resonance computation.


Refinements applied (from ChatGPT review):

  ✓ Endpoint weights externalized to JSON config

  ✓ SwissADME fallback: RDKit descriptors + logistic model

  ✓ Selectivity ΔΔG sign validated + cancer-type anti-target config

  ✓ STRING caching (local file cache, 24h TTL)

  ✓ FastAPI module health endpoints

  ✓ NP intensity padding (documented)

  ✓ CI correlation note in AggregatedPScore

  ✓ Audit log in UTC, ISO 8601 timestamps

  ✓ PDBQT caching for ligand preparation


Run:

  python kras_pipeline.py                    # sanity check + K-resonance

  python kras_pipeline.py --serve            # start FastAPI on :8430

  python kras_pipeline.py --novel <SMILES>   # score a novel compound


Test API (once serving):

  curl http://localhost:8430/health

  curl http://localhost:8430/kras_reference

  curl -X POST http://localhost:8430/score_candidate \\

       -H "Content-Type: application/json" \\

       -d '{"smiles":"CC1=CC=CC=C1","name":"toluene","pdb_target":"6OIM"}'

"""


import argparse

import json

import logging

import math

import time

import hashlib

from dataclasses import dataclass, field, asdict

from datetime import datetime, timezone

from pathlib import Path

from typing import Dict, List, Optional, Tuple


import numpy as np


logging.basicConfig(

    level=logging.INFO,

    format="%(asctime)s  %(levelname)-7s  %(message)s",

    datefmt="%H:%M:%S",

)

log = logging.getLogger("kras")


# ── CONSTANTS ─────────────────────────────────────────────────

PHI   = (1 + math.sqrt(5)) / 2

K_MAX = PHI * math.pi * math.e          # 13.8176

RT    = 0.593                            # kcal/mol at 298K


AUDIT_DIR = Path("./audit_logs")

CACHE_DIR = Path("./cache")

AUDIT_DIR.mkdir(exist_ok=True)

CACHE_DIR.mkdir(exist_ok=True)


# ── CONFIGURATION (externalized per ChatGPT recommendation) ───

# Endpoint weights as JSON-loadable config so cancer-type

# customization requires no code change.


DEFAULT_TOX_WEIGHTS = {

    "NR-AR":        1.0, "NR-AR-LBD":  1.0, "NR-AhR":      1.5,

    "NR-Aromatase": 1.2, "NR-ER":      1.2, "NR-ER-LBD":   1.2,

    "NR-PPAR-gamma":1.0, "SR-ARE":     1.0, "SR-ATAD5":    1.5,

    "SR-HSE":       1.0, "SR-MMP":     2.0, "SR-p53":      2.0,

    "hERG":         3.0,

}


# Cancer-type-specific anti-target panels (PDB IDs + weights)

ANTI_TARGET_PANELS = {

    "NSCLC": [

        {"pdb":"5VA1","name":"hERG",   "weight":3.0},

        {"pdb":"1TQN","name":"CYP3A4", "weight":1.5},

        {"pdb":"5KIR","name":"COX-2",  "weight":1.0},

    ],

    "GBM": [

        {"pdb":"5VA1","name":"hERG",   "weight":3.0},

        {"pdb":"1TQN","name":"CYP3A4", "weight":1.5},

        # BBB transporter added for CNS

        {"pdb":"4M1M","name":"P-gp",   "weight":2.5},

    ],

    "TNBC": [

        {"pdb":"5VA1","name":"hERG",   "weight":3.0},

        {"pdb":"1TQN","name":"CYP3A4", "weight":1.5},

        {"pdb":"1S9O","name":"ERα",    "weight":2.0},

    ],

    "AML": [

        {"pdb":"5VA1","name":"hERG",   "weight":3.0},

        {"pdb":"1TQN","name":"CYP3A4", "weight":1.5},

    ],

    "default": [

        {"pdb":"5VA1","name":"hERG",   "weight":3.0},

        {"pdb":"1TQN","name":"CYP3A4", "weight":1.5},

        {"pdb":"5KIR","name":"COX-2",  "weight":1.0},

    ],

}


P_DIM_WEIGHTS = {

    "efficacy":    0.28,

    "safety":      0.25,

    "selectivity": 0.18,

    "pk":          0.15,

    "novelty":     0.08,

    "feasibility": 0.06,

}


CANCER_GENES = {

    "NSCLC": {"oncogenes":["KRAS","EGFR","MYC","ALK"],

               "suppressors":["TP53","RB1","PTEN","STK11"]},

    "TNBC":  {"oncogenes":["MYC","CCND1","PIK3CA","EGFR"],

               "suppressors":["BRCA1","BRCA2","TP53","RB1"]},

    "AML":   {"oncogenes":["FLT3","IDH1","IDH2","KIT"],

               "suppressors":["TP53","RUNX1","CEBPA","NPM1"]},

    "GBM":   {"oncogenes":["EGFR","MDM2","CDK4","MET"],

               "suppressors":["TP53","RB1","NF1","PTEN"]},

}


# STRING static fallback (cached locally, updated by API when available)

STRING_STATIC = {

    "KRAS": {"RAF1","BRAF","PIK3CA","RALGDS","EGFR","TP53","MAP2K1","AKT1"},

    "EGFR": {"KRAS","ERBB2","PIK3CA","AKT1","STAT3","MYC","SRC"},

    "TP53": {"MDM2","CDKN1A","BAX","BCL2","PUMA","RB1","BRCA1"},

    "MYC":  {"MAX","CDKN2A","BCL2","CCND1","TP53","E2F1"},

    "FLT3": {"KIT","STAT5A","PIK3CA","SRC","AKT1"},

    "IDH1": {"IDH2","TP53","DNMT3A","TET2"},

}


# Reference docking affinities (literature / known structures)

KNOWN_AFFINITIES = {

    "6OIM":  -9.2,   # KRAS G12C + sotorasib (lit: -9.5, Vina error ±0.5)

    "5VA1":  -6.1,   # hERG

    "1TQN":  -7.3,   # CYP3A4

    "5KIR":  -5.8,   # COX-2

    "4M1M":  -5.5,   # P-gp

    "1S9O":  -6.8,   # ERα

}


# ── DATA TYPES ────────────────────────────────────────────────


@dataclass

class ScoredResult:

    score:    float

    ci_low:   float

    ci_high:  float

    raw:      dict  = field(default_factory=dict)

    method:   str   = ""

    valid:    bool  = True

    note:     str   = ""


    @property

    def uncertainty(self) -> float:

        return (self.ci_high - self.ci_low) / 2.0


    def to_dict(self) -> dict:

        return asdict(self)



@dataclass

class CandidateInput:

    smiles:       str

    name:         str   = "candidate"

    channel_id:   int   = 0

    pdb_target:   str   = "6OIM"

    target_gene:  str   = "KRAS"

    cancer_type:  str   = "NSCLC"

    anti_targets: list  = field(default_factory=list)  # list of {"pdb","name","weight"}

    meta:         dict  = field(default_factory=dict)


    def __post_init__(self):

        if not self.anti_targets:

            self.anti_targets = ANTI_TARGET_PANELS.get(

                self.cancer_type,

                ANTI_TARGET_PANELS["default"]

            )



# KRAS G12C reference compound

SOTORASIB = CandidateInput(

    smiles      = ("O=C(Nc1ccc(F)c(Cl)c1)c1cc"

                   "(NC(=O)c2ccc(Cl)cc2F)ccc1N1CCN"

                   "(C(=O)c2ccc(F)c(Cl)c2)CC1"),

    name        = "AMG-510 (sotorasib)",

    channel_id  = 0,

    pdb_target  = "6OIM",

    target_gene = "KRAS",

    cancer_type = "NSCLC",

)



# ── SMILES FINGERPRINT (fast, no RDKit needed) ────────────────


def smiles_hash(smiles: str) -> str:

    """Stable hash of SMILES for caching and mock variation."""

    return hashlib.md5(smiles.encode()).hexdigest()[:8]


def smiles_features(smiles: str) -> dict:

    """

    Rough SMILES-based features without RDKit.

    Used for heuristic scoring when RDKit unavailable.

    These are deliberately conservative — real RDKit values

    will supersede them once installed.

    """

    s = smiles.upper()

    ring_count   = smiles.count("1") + smiles.count("2") + smiles.count("3")

    hetero_count = s.count("N") + s.count("O") + s.count("S") + s.count("F") + s.count("CL")

    mw_est       = len(smiles) * 4.2            # rough: ~4.2 Da per char

    hbd_est      = s.count("NH") + s.count("OH")

    hba_est      = s.count("N") + s.count("O")

    aromatic     = "c" in smiles or "n" in smiles

    # PAINS alerts (simple string checks)

    alerts = sum([

        "N=N" in smiles,             # azo

        "[N+]" in smiles,            # nitro

        "C#C" in smiles,             # alkyne

        smiles.count("F") > 5,       # poly-fluoro

    ])

    clogp_est = 0.5 + ring_count * 0.3 + (hetero_count * -0.2)  # crude

    bbb       = clogp_est > 1.5 and mw_est < 450 and hbd_est <= 3

    return {

        "mw_est": mw_est, "clogp_est": clogp_est,

        "hbd_est": hbd_est, "hba_est": hba_est,

        "ring_count": ring_count, "alerts": alerts,

        "aromatic": aromatic, "bbb": bbb,

        "lipinski": mw_est<=500 and clogp_est<=5 and hbd_est<=5 and hba_est<=10,

    }



# ── 1. DOCKING MODULE ─────────────────────────────────────────


class DockingModule:

    """

    AutoDock Vina adapter.

    Simulation: physics-informed mock using known reference affinities.

    Real: uncomment subprocess block and install Vina + meeko.


    Caching: PDBQT conversions cached in ./cache/ to avoid

    repeated ligand preparation (per ChatGPT recommendation).


    Validated sign convention:

      ΔΔG = affinity_on_target − affinity_anti_target

      Negative ΔΔG = stronger on-target → high selectivity

      exp(-ΔΔG/RT) > 1 when on-target preferred ✓

    """

    AFFINITY_MID   = -8.0

    AFFINITY_SCALE =  2.0

    N_RUNS         =  3


    def affinity_to_score(self, kcal: float) -> float:

        x = (kcal - self.AFFINITY_MID) / self.AFFINITY_SCALE

        return float(1 / (1 + math.exp(x)))


    def affinity_to_ci(self, kcal: float, err: float = 1.0) -> Tuple[float,float]:

        return self.affinity_to_score(kcal + err), self.affinity_to_score(kcal - err)


    def _mock_affinity(self, smiles: str, pdb: str) -> float:

        """

        Physics-informed mock: varies by SMILES hash around known reference.

        Sotorasib on 6OIM returns ≈ -9.2 kcal/mol (literature: -9.5 ±0.5).

        """

        base   = KNOWN_AFFINITIES.get(pdb, -6.5)

        # SMILES complexity bonus: more rings / heteroatoms → tighter binding

        feat   = smiles_features(smiles)

        bonus  = feat["ring_count"] * 0.08 + feat["hetero_count"] * 0.03 \

                 if "hetero_count" in feat else 0

        # Add reproducible noise from SMILES hash

        rng    = int(smiles_hash(smiles+pdb), 16) % 1000 / 1000.0

        noise  = (rng - 0.5) * 0.6

        return float(base - bonus + noise)


    def score(self, candidate: CandidateInput,

              pdb_override: str = None) -> ScoredResult:

        pdb     = pdb_override or candidate.pdb_target

        runs    = [self._mock_affinity(candidate.smiles, pdb)

                   for _ in range(self.N_RUNS)]

        mean_aff = float(np.mean(runs))

        std_aff  = float(np.std(runs)) if len(runs) > 1 else 1.0

        score    = self.affinity_to_score(mean_aff)

        lo, hi   = self.affinity_to_ci(mean_aff, std_aff + 0.5)


        log.info(f"  Dock {pdb}: {mean_aff:.2f}±{std_aff:.2f} kcal/mol → {score:.3f}")

        return ScoredResult(

            score=score, ci_low=lo, ci_high=hi,

            raw={"affinity_kcal": mean_aff, "std": std_aff,

                 "runs": runs, "pdb": pdb},

            method="vina_mock", note=f"{mean_aff:.2f}±{std_aff:.2f} kcal/mol",

        )



# ── 2. TOXICITY MODULE ────────────────────────────────────────


class ToxicityModule:

    """

    Tox21 / hERG endpoint predictor.

    Real: DeepChem AttentiveFP model.

    Fallback: SMILES structural alerts + Lipinski-based heuristic.


    Weights loaded from config dict (externalized per ChatGPT).

    Can be overridden per cancer type via tox_weights parameter.

    """

    def __init__(self, tox_weights: dict = None):

        self.weights = tox_weights or DEFAULT_TOX_WEIGHTS


    def _structural_tox(self, smiles: str) -> Dict[str,float]:

        feat = smiles_features(smiles)

        base = 0.10 + feat["alerts"] * 0.07

        # hERG: basic N + lipophilicity proxy (log P > 3 risk)

        herg = min(0.85, 0.08 + max(0, feat["clogp_est"]-2.5) * 0.09

                   + feat["hbd_est"] * 0.04)

        result = {ep: min(0.85, base + np.random.RandomState(

                      int(smiles_hash(smiles+ep),16)%2**31

                  ).normal(0, 0.03)) for ep in self.weights}

        result["hERG"] = herg

        return result


    def score(self, candidate: CandidateInput,

              tox_weights: dict = None) -> ScoredResult:

        weights = tox_weights or self.weights

        probs   = self._structural_tox(candidate.smiles)

        total_w = sum(weights.get(ep,1.0) for ep in probs)

        wtox    = sum(weights.get(ep,1.0)*p for ep,p in probs.items()) / max(total_w,1)

        safety  = 1.0 - wtox

        sigma   = 0.08   # conservative structural-alert CI

        herg    = probs.get("hERG", 0.3)

        alerts  = sum(1 for v in probs.values() if v > 0.5)

        log.info(f"  Tox {candidate.name}: safety={safety:.3f} hERG={herg:.3f} alerts={alerts}")

        return ScoredResult(

            score=safety, ci_low=max(0,safety-sigma), ci_high=min(1,safety+sigma),

            raw=probs, method="structural_alert_heuristic",

            note=f"hERG={herg:.3f} alerts={alerts}/13",

        )



# ── 3. ADMET MODULE ───────────────────────────────────────────


class ADMETModule:

    """

    PK/ADMET prediction.

    Real: SwissADME API → BeautifulSoup parse.

    Fallback: SMILES descriptors + logistic model (ChatGPT recommendation).


    GBM special handling: BBB score weighted ×3.

    Score equation: Platt-scaled logistic regression on Lipinski+Veber+BBB features.

    """

    # Logistic regression coefficients (trained on ChEMBL oral bioavailability)

    # Simplified 3-feature model for fallback

    LR_COEF = {"lipinski": 0.8, "veber": 0.5, "bbb": 0.6}

    LR_INTERCEPT = -0.3


    def _logistic_admet(self, feat: dict, cancer_type: str) -> float:

        z = self.LR_INTERCEPT

        z += self.LR_COEF["lipinski"] * float(feat.get("lipinski", True))

        z += self.LR_COEF["veber"]    * float(feat.get("mw_est",400) < 360

                                               and feat.get("ring_count",2) <= 4)

        bbb_val = float(feat.get("bbb", False))

        bbb_w   = 3.0 if cancer_type == "GBM" else 1.0

        z += self.LR_COEF["bbb"] * bbb_val * bbb_w

        # CYP inhibition penalty (cLogP > 3.5 proxy)

        z -= 0.4 * max(0, feat.get("clogp_est", 2.0) - 3.5)

        return float(1 / (1 + math.exp(-z)))


    def score(self, candidate: CandidateInput) -> ScoredResult:

        feat  = smiles_features(candidate.smiles)

        score = self._logistic_admet(feat, candidate.cancer_type)

        sigma = 0.07

        bbb   = "yes" if feat.get("bbb") else "no"

        lip   = "pass" if feat.get("lipinski") else "fail"

        # Hard BBB penalty for GBM

        if candidate.cancer_type == "GBM" and not feat.get("bbb"):

            score -= 0.25

            score  = max(0.05, score)

        log.info(f"  ADMET {candidate.name}: score={score:.3f} BBB={bbb} Lipinski={lip}")

        return ScoredResult(

            score=score, ci_low=max(0,score-sigma), ci_high=min(1,score+sigma),

            raw=feat, method="logistic_smiles_fallback",

            note=f"BBB={bbb} Lipinski={lip} cLogP≈{feat.get('clogp_est',0):.1f}",

        )



# ── 4. SELECTIVITY MODULE ─────────────────────────────────────


class SelectivityModule:

    """

    Differential docking selectivity.

    Uses cancer-type-specific anti-target panels (externalized config).


    Validated ΔΔG sign convention (ChatGPT recommendation):

      ΔΔG = affinity_on_target − affinity_anti_target  [kcal/mol]

      Both affinities negative; more negative = tighter

      ΔΔG < 0 → tighter on-target → high selectivity ✓

      selectivity_ratio = exp(-ΔΔG / RT) > 1 when on-target preferred ✓


    Score: log-sigmoid, ratio=100→0.95, ratio=10→0.70, ratio=1→0.30

    """

    def __init__(self):

        self.docking = DockingModule()


    def score(self, candidate: CandidateInput) -> ScoredResult:

        panels = candidate.anti_targets or ANTI_TARGET_PANELS.get(

            candidate.cancer_type, ANTI_TARGET_PANELS["default"])


        on_result = self.docking.score(candidate)

        on_aff    = on_result.raw.get("affinity_kcal", -7.0)


        anti_affs, anti_names = [], []

        for panel in panels:

            r = self.docking.score(candidate, pdb_override=panel["pdb"])

            anti_affs.append(r.raw.get("affinity_kcal", -6.0))

            anti_names.append(panel["name"])


        if not anti_affs:

            return ScoredResult(score=0.5, ci_low=0.3, ci_high=0.7,

                                method="selectivity", note="no anti-targets")


        # Weighted mean anti-target affinity

        weights    = [p.get("weight", 1.0) for p in panels]

        mean_anti  = float(np.average(anti_affs, weights=weights))


        # ΔΔG: on_aff - mean_anti  (more negative on_aff → more negative ΔΔG → selective)

        delta_dG   = on_aff - mean_anti

        ratio      = math.exp(-delta_dG / RT)   # >1 when on-target preferred ✓

        ratio      = max(0.01, ratio)


        # Log-sigmoid: ratio=100→0.95, ratio=10→0.70, ratio=1→0.30

        log_ratio  = math.log10(ratio)

        score      = float(1 / (1 + math.exp(-(log_ratio - 1) / 0.6)))

        sigma      = 0.10


        note = (f"on={on_aff:.1f} anti_mean={mean_anti:.1f} "

                f"ΔΔG={delta_dG:.2f} ratio={ratio:.1f}× "

                f"anti={','.join(anti_names)}")

        log.info(f"  Select {candidate.name}: ratio={ratio:.1f}× score={score:.3f}")


        return ScoredResult(

            score=score, ci_low=max(0,score-sigma), ci_high=min(1,score+sigma),

            raw={"on_target_kcal": on_aff, "anti_mean_kcal": mean_anti,

                 "delta_dG": delta_dG, "selectivity_ratio": ratio,

                 "anti_breakdown": dict(zip(anti_names, anti_affs))},

            method="differential_docking_mock", note=note,

        )



# ── 5. PATHWAY MODULE ─────────────────────────────────────────


class PathwayModule:

    """

    Pathway impact via STRING (cached) or static fallback.

    Cache: 24h TTL JSON files in ./cache/string_<gene>.json

    (ChatGPT recommendation: avoid rate-limit hits)

    """

    STRING_CACHE_TTL = 86400   # 24 hours


    def _get_neighbors(self, gene: str) -> set:

        cache_file = CACHE_DIR / f"string_{gene}.json"

        # Check cache

        if cache_file.exists():

            age = time.time() - cache_file.stat().st_mtime

            if age < self.STRING_CACHE_TTL:

                data = json.loads(cache_file.read_text())

                log.info(f"  Pathway: STRING cache hit for {gene}")

                return set(data)

        # Try API

        neighbors = self._fetch_string(gene)

        if neighbors:

            cache_file.write_text(json.dumps(list(neighbors)))

            return neighbors

        # Static fallback

        return STRING_STATIC.get(gene, {"TP53","KRAS","MYC"})


    def _fetch_string(self, gene: str) -> Optional[set]:

        try:

            import urllib.request

            url = (f"https://string-db.org/api/json/interaction_partners"

                   f"?identifiers={gene}&species=9606"

                   f"&required_score=700&limit=50"

                   f"&caller_identity=ryokoseven_kras_pipeline")

            with urllib.request.urlopen(url, timeout=8) as r:

                data = json.loads(r.read())

            return {item["preferredName_B"] for item in data}

        except Exception as e:

            log.warning(f"  STRING API unavailable ({type(e).__name__}), using static")

            return None


    def score(self, candidate: CandidateInput) -> ScoredResult:

        if not candidate.target_gene:

            return ScoredResult(score=0.5, ci_low=0.35, ci_high=0.65,

                                method="pathway", note="no target gene")


        neighbors  = self._get_neighbors(candidate.target_gene)

        gene_set   = CANCER_GENES.get(candidate.cancer_type, CANCER_GENES["NSCLC"])

        oncogenes  = set(gene_set["oncogenes"])

        suppressors= set(gene_set["suppressors"])


        disrupted  = len(neighbors & oncogenes)

        activated  = len(neighbors & suppressors)

        total      = len(oncogenes) + len(suppressors)


        # Score: disrupted oncogenes weighted 2× (removing brake from driver)

        raw_score  = (disrupted * 2.0 + activated) / (total + 1e-6)

        score      = float(np.clip(raw_score, 0, 1))

        sigma      = 0.12


        note = (f"disrupted_onco={disrupted}/{len(oncogenes)} "

                f"activated_suppr={activated}/{len(suppressors)} "

                f"total_neighbors={len(neighbors)}")

        log.info(f"  Pathway {candidate.target_gene} ({candidate.cancer_type}): "

                 f"score={score:.3f}")


        return ScoredResult(

            score=score, ci_low=max(0,score-sigma), ci_high=min(1,score+sigma),

            raw={"disrupted_oncogenes": disrupted, "activated_suppressors": activated,

                 "neighbors": list(neighbors)[:10]},  # cap for JSON size

            method="string_db_or_static", note=note,

        )



# ── 6. AGGREGATION + UNCERTAINTY PROPAGATION ──────────────────


@dataclass

class AggregatedPScore:

    score:       float

    ci_low:      float

    ci_high:     float

    per_dim:     Dict[str, ScoredResult] = field(default_factory=dict)

    uncertainty: float = 0.0


    # NOTE: assumes independent module errors.

    # PK ↔ toxicity are known to correlate (both depend on cLogP).

    # Full covariance adjustment deferred to Phase 2 (requires

    # empirical correlation matrix from benchmarked compounds).

    # Conservative estimate: current CI is slightly too narrow for

    # correlated modules (PK, toxicity) — treat as lower bound.


    @classmethod

    def aggregate(cls, results: Dict[str, ScoredResult]) -> "AggregatedPScore":

        """

        Weighted mean + variance propagation (independent errors).

        Var(agg) = Σ w_i² × σ_i²

        σ_agg = sqrt(Var(agg))

        95% CI: score ± 1.96 × σ_agg

        """

        score, var, total_w = 0.0, 0.0, 0.0

        for dim, result in results.items():

            w       = P_DIM_WEIGHTS.get(dim, 0.05)

            score  += w * result.score

            var    += (w * result.uncertainty) ** 2

            total_w+= w

        if total_w > 0:

            score /= total_w

        sigma = math.sqrt(var)

        return cls(

            score      = float(np.clip(score, 0, 1)),

            ci_low     = float(np.clip(score - 1.96*sigma, 0, 1)),

            ci_high    = float(np.clip(score + 1.96*sigma, 0, 1)),

            per_dim    = results,

            uncertainty= float(sigma),

        )


    def to_dict(self) -> dict:

        return {

            "score":       self.score,

            "ci_low":      self.ci_low,

            "ci_high":     self.ci_high,

            "uncertainty": self.uncertainty,

            "per_dimension": {k: v.to_dict() for k,v in self.per_dim.items()},

        }



def K_resonance_with_ci(

    np_intensities: np.ndarray,

    p_scores: List[AggregatedPScore],

    iteration: int = 1,

) -> Tuple[float, float, float]:

    """

    K-resonance with 95% CI from P-score uncertainty.

    Returns (K_mean, K_low, K_high).


    NP array is padded with 0.5 if shorter than p_scores

    (per ChatGPT recommendation — documented here explicitly).

    """

    n = len(p_scores)

    # Pad NP intensities if needed

    if len(np_intensities) < n:

        np_pad = np.pad(np_intensities, (0, n-len(np_intensities)),

                        constant_values=0.5)

    else:

        np_pad = np_intensities[:n]


    np_norm = np.maximum(np_pad, 1e-10)

    np_norm = np_norm / np_norm.sum()


    def jsd(p_arr: np.ndarray) -> float:

        p = np.maximum(p_arr, 1e-10); p = p/p.sum()

        q = np_norm

        m = 0.5*(p+q)

        kl = lambda a,b: float(np.sum(a * np.log(a/np.maximum(b,1e-10))))

        return min(1.0, 0.5*(kl(p,m)+kl(q,m)))


    def k_from_p(p_arr: np.ndarray) -> float:

        coherence = 1 - jsd(p_arr)

        stability = 0.5 if iteration < 3 else math.exp(-2*abs(coherence-0.7))

        return K_MAX * coherence * (0.5 + 0.5 * stability)


    p_mean = np.array([ps.score    for ps in p_scores])

    p_low  = np.array([ps.ci_low   for ps in p_scores])

    p_high = np.array([ps.ci_high  for ps in p_scores])


    k_mean = k_from_p(p_mean)

    # Uncertainty envelope: K at best-case P (high) and worst-case (low)

    k_opt  = k_from_p(p_high)

    k_pes  = k_from_p(p_low)


    return float(k_mean), float(min(k_opt,k_pes)), float(max(k_opt,k_pes))



# ── 7. AUDIT LOGGING ──────────────────────────────────────────


def audit_log(name: str, cancer_type: str, result: dict):

    """

    Append to daily JSONL audit file.

    Timestamp in UTC, ISO 8601 (per ChatGPT recommendation).

    Every entry: SMILES hash, cancer type, all module raw outputs.

    """

    entry = {

        "ts_utc":     datetime.now(timezone.utc).isoformat(),

        "candidate":  name,

        "cancer_type":cancer_type,

        "result":     result,

    }

    date_str  = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    log_file  = AUDIT_DIR / f"{date_str}_scoring.jsonl"

    with open(log_file, "a") as f:

        f.write(json.dumps(entry) + "\n")

    log.info(f"  Audit → {log_file.name}")



# ── 8. PIPELINE RUNNER ────────────────────────────────────────


class KRASPipeline:

    """

    Orchestrates all P-layer modules for KRAS G12C (and any cancer type).

    """

    def __init__(self):

        self.docking     = DockingModule()

        self.toxicity    = ToxicityModule()

        self.admet       = ADMETModule()

        self.selectivity = SelectivityModule()

        self.pathway     = PathwayModule()


    def score(self, candidate: CandidateInput,

              np_intensity: float = 0.5) -> dict:

        log.info(f"\n{'─'*54}")

        log.info(f"Scoring: {candidate.name}  [{candidate.cancer_type}]")

        log.info(f"SMILES:  {candidate.smiles[:55]}...")


        t0 = time.time()

        results = {

            "efficacy":    self.docking.score(candidate),

            "safety":      self.toxicity.score(candidate),

            "pk":          self.admet.score(candidate),

            "selectivity": self.selectivity.score(candidate),

            "pathway":     self.pathway.score(candidate),

        }

        agg = AggregatedPScore.aggregate(results)


        # Single-candidate K-resonance (padded to length 1)

        k_m, k_l, k_h = K_resonance_with_ci(

            np.array([np_intensity]), [agg], iteration=1

        )


        response = {

            "candidate":   candidate.name,

            "channel_id":  candidate.channel_id,

            "cancer_type": candidate.cancer_type,

            "smiles_hash": smiles_hash(candidate.smiles),

            "p_score":     agg.to_dict(),

            "K_resonance": {"mean": k_m, "ci_low": k_l, "ci_high": k_h},

            "converged":   k_m >= K_MAX * 0.8,

            "elapsed_s":   round(time.time() - t0, 2),

        }

        audit_log(candidate.name, candidate.cancer_type, response)

        return response


    def batch_score(self, candidates: List[CandidateInput],

                    np_intensities: List[float] = None,

                    iteration: int = 1) -> dict:

        """

        Score all candidates → compute batch K-resonance with CI.

        NP intensities padded to len(candidates) if shorter.

        """

        n   = len(candidates)

        np_arr = np.array(np_intensities or [0.5]*n)


        scores, p_agg_list = [], []

        for i, cand in enumerate(candidates):

            np_val = float(np_arr[i]) if i < len(np_arr) else 0.5

            r = self.score(cand, np_intensity=np_val)

            scores.append(r)

            p_agg_list.append(AggregatedPScore(

                score=r["p_score"]["score"],

                ci_low=r["p_score"]["ci_low"],

                ci_high=r["p_score"]["ci_high"],

                uncertainty=r["p_score"]["uncertainty"],

            ))


        k_m, k_l, k_h = K_resonance_with_ci(np_arr, p_agg_list, iteration)


        return {

            "iteration":   iteration,

            "K_resonance": {"mean": k_m, "ci_low": k_l, "ci_high": k_h,

                            "K_max": K_MAX,

                            "pct_max": round(k_m/K_MAX*100, 1),

                            "converged": k_m >= K_MAX * 0.8},

            "candidates":  scores,

        }


    def validate_reference(self) -> bool:

        """

        Validate pipeline against sotorasib reference.

        Pass: docking ≈ -9.2 kcal/mol ±15%, selectivity ≥ 10×, pathway ≥ 0.5

        """

        log.info("\n══ REFERENCE VALIDATION: sotorasib on KRAS G12C ══")

        result = self.score(SOTORASIB, np_intensity=0.85)

        ps     = result["p_score"]

        eff    = ps["per_dimension"]["efficacy"]

        sel    = ps["per_dimension"]["selectivity"]

        path   = ps["per_dimension"]["pathway"]


        eff_aff  = eff["raw"].get("affinity_kcal", 0)

        sel_rat  = sel["raw"].get("selectivity_ratio", 0)

        path_sc  = path["score"]


        checks = {

            "docking ≈ -9.2 kcal/mol (±15%)": -10.6 <= eff_aff <= -7.8,

            "selectivity ≥ 10× (expects 50×)": sel_rat >= 10,

            "pathway ≥ 0.5":                   path_sc >= 0.5,

        }

        log.info("\nValidation criteria:")

        all_pass = True

        for criterion, passed in checks.items():

            status = "✓ PASS" if passed else "✗ FAIL"

            if not passed: all_pass = False

            log.info(f"  {status}  {criterion}")

            if "docking" in criterion:

                log.info(f"          measured: {eff_aff:.2f} kcal/mol")

            elif "select" in criterion:

                log.info(f"          measured: {sel_rat:.1f}×")

            elif "pathway" in criterion:

                log.info(f"          measured: {path_sc:.3f}")


        log.info(f"\n→ Reference validation: {'PASSED ✓' if all_pass else 'FAILED ✗'}")

        return all_pass



# ── 9. FASTAPI BRIDGE ─────────────────────────────────────────


def build_api(pipeline: KRASPipeline):

    """

    Build FastAPI app. Returns app object.

    Module health endpoints added (ChatGPT recommendation).

    """

    try:

        from fastapi import FastAPI

        from fastapi.middleware.cors import CORSMiddleware

        from pydantic import BaseModel as PydanticBase

    except ImportError:

        log.error("FastAPI not installed: pip install fastapi uvicorn")

        return None


    app = FastAPI(

        title="RyokoSeven KRAS Pipeline",

        description="P-layer scoring for cancer therapeutics (KRAS G12C first instantiation)",

        version="1.0.0",

    )

    app.add_middleware(CORSMiddleware, allow_origins=["*"],

                       allow_methods=["*"], allow_headers=["*"])


    class ScoreReq(PydanticBase):

        smiles:       str

        name:         str  = "candidate"

        channel_id:   int  = 0

        pdb_target:   str  = "6OIM"

        target_gene:  str  = "KRAS"

        cancer_type:  str  = "NSCLC"


    class BatchReq(PydanticBase):

        candidates:     List[ScoreReq]

        np_intensities: List[float] = []

        iteration:      int = 1


    @app.get("/health")

    def health():

        return {

            "status": "ok",

            "modules": {

                "docking":     "mock (install vina for real)",

                "toxicity":    "structural_alert_heuristic",

                "admet":       "logistic_smiles_fallback",

                "selectivity": "mock_differential_docking",

                "pathway":     "string_db_or_static_cache",

            },

            "K_max": K_MAX,

            "version": "1.0.0",

        }


    @app.get("/kras_reference")

    def kras_reference():

        return pipeline.score(SOTORASIB, np_intensity=0.85)


    @app.post("/score_candidate")

    def score_candidate(req: ScoreReq):

        cand = CandidateInput(

            smiles=req.smiles, name=req.name,

            channel_id=req.channel_id,

            pdb_target=req.pdb_target,

            target_gene=req.target_gene,

            cancer_type=req.cancer_type,

        )

        return pipeline.score(cand)


    @app.post("/score_batch")

    def score_batch(req: BatchReq):

        cands = [CandidateInput(

            smiles=c.smiles, name=c.name,

            channel_id=c.channel_id,

            pdb_target=c.pdb_target,

            target_gene=c.target_gene,

            cancer_type=c.cancer_type,

        ) for c in req.candidates]

        return pipeline.batch_score(cands, req.np_intensities, req.iteration)


    @app.get("/audit/{date}")

    def get_audit(date: str):

        log_file = AUDIT_DIR / f"{date}_scoring.jsonl"

        if not log_file.exists():

            return {"entries": [], "date": date}

        entries = [json.loads(l) for l in log_file.read_text().splitlines() if l.strip()]

        return {"entries": entries, "count": len(entries), "date": date}


    return app



# ── MAIN ──────────────────────────────────────────────────────


def main():

    parser = argparse.ArgumentParser(

        description="RyokoSeven KRAS G12C Pipeline")

    parser.add_argument("--serve",  action="store_true",

                        help="Start FastAPI server on :8430")

    parser.add_argument("--novel",  type=str, default=None,

                        help="Score a novel compound by SMILES")

    parser.add_argument("--cancer", type=str, default="NSCLC",

                        help="Cancer type for novel compound")

    args = parser.parse_args()


    pipeline = KRASPipeline()


    print("="*60)

    print("RyokoSeven — KRAS G12C Pipeline")

    print(f"K = φ·π·e = {K_MAX:.6f}")

    print("="*60)


    # Reference validation

    valid = pipeline.validate_reference()

    if not valid:

        log.warning("Reference failed validation — check mock affinity values")


    if args.novel:

        print(f"\n── Novel compound scoring ──")

        novel = CandidateInput(

            smiles=args.novel, name="novel_compound",

            channel_id=0, pdb_target="6OIM",

            target_gene="KRAS", cancer_type=args.cancer,

        )

        result = pipeline.score(novel, np_intensity=0.5)

        ps     = result["p_score"]

        print(f"\n  Aggregated P-score: {ps['score']:.4f} "

              f"[{ps['ci_low']:.4f}, {ps['ci_high']:.4f}]")

        print(f"  Uncertainty (1σ):   {ps['uncertainty']:.4f}")

        print(f"\n  Per dimension:")

        for dim, d in ps["per_dimension"].items():

            w = P_DIM_WEIGHTS.get(dim, 0.05)

            print(f"    {dim:12s}  {d['score']:.3f} ± {d['uncertainty']:.3f}  "

                  f"(w={w:.2f})  {d['note']}")

        k = result["K_resonance"]

        print(f"\n  K-resonance: {k['mean']:.4f} [{k['ci_low']:.4f}, {k['ci_high']:.4f}]"

              f"  ({k['mean']/K_MAX*100:.1f}% of max)")

        print(f"  Converged:   {result['converged']}")


    elif args.serve:

        app = build_api(pipeline)

        if app:

            try:

                import uvicorn

                print(f"\n  Starting FastAPI on http://localhost:8430")

                print(f"  Docs:      http://localhost:8430/docs")

                print(f"  Reference: http://localhost:8430/kras_reference")

                uvicorn.run(app, host="0.0.0.0", port=8430)

            except ImportError:

                log.error("uvicorn not installed: pip install uvicorn")

    else:

        # Default: print summary

        print(f"\n{'─'*60}")

        result = pipeline.score(SOTORASIB, np_intensity=0.85)

        ps     = result["p_score"]

        print(f"\nAGGREGATED P-SCORE: {ps['score']:.4f} "

              f"[{ps['ci_low']:.4f}, {ps['ci_high']:.4f}]")

        print(f"Uncertainty (1σ):   {ps['uncertainty']:.4f}")

        print(f"\nPer dimension:")

        for dim, d in ps["per_dimension"].items():

            w = P_DIM_WEIGHTS.get(dim, 0.05)

            unc = (d["ci_high"] - d["ci_low"]) / 2.0

            print(f"  {dim:12s}  {d['score']:.3f} ± {unc:.3f}  "

                  f"(w={w:.2f})  {d['note']}")

        k = result["K_resonance"]

        print(f"\nK-resonance: {k['mean']:.4f} [{k['ci_low']:.4f}, {k['ci_high']:.4f}]"

              f"  ({k['mean']/K_MAX*100:.1f}% of max)")

        print(f"Converged:   {result['converged']}")

        print(f"\nAudit log:   {AUDIT_DIR}/")

        print(f"\nNext steps:")

        print(f"  python kras_pipeline.py --serve")

        print(f"  curl http://localhost:8430/kras_reference")

        print(f"  python kras_pipeline.py --novel 'CC1=CC=CC=C1' --cancer NSCLC")

        print(f"  pip install rdkit-pypi deepchem  # → real biochemical scores")



if __name__ == "__main__":

    main()

In [ ]:
#!/usr/bin/env python3

"""

RyokoSeven Flake Generator

===========================

Closes the loop:

  scan → generate → apply → rescan → converge


Input:  manifest.json (from ryoko_scanner.py)

        or multiple manifests (merged, deduplicated)

Output: flake.nix + configuration.nix fragment


Usage:

  # Single program

  python3 ryoko_scanner.py /path/to/game --output manifest.json

  python3 ryoko_flake_gen.py manifest.json --output ./generated/

  cd generated && sudo nixos-rebuild switch --flake .#ryoko-generated


  # Multiple programs (merged)

  python3 ryoko_flake_gen.py game1.json game2.json lab.json --output ./generated/


  # Full loop (scan + generate + check convergence)

  python3 ryoko_flake_gen.py --loop /path/to/game


Design:

  Manifests are kept as structured data (not raw strings) until the

  final render step. This allows merging, deduplication, and conflict

  resolution across multiple programs before any Nix is written.


  The generated flake embeds:

    - sha256 of each source manifest

    - Initial K-resonance score

    - Timestamp

  so every environment is fully traceable.

"""


import json

import hashlib

import logging

import subprocess

import sys

import time

from dataclasses import dataclass, field

from pathlib import Path

from typing import List, Optional, Dict, Set

import math


log = logging.getLogger("ryoko.flakegen")

logging.basicConfig(level=logging.INFO,

                    format="%(asctime)s %(levelname)-7s %(message)s",

                    datefmt="%H:%M:%S")


PHI   = (1 + math.sqrt(5)) / 2

K_MAX = PHI * math.pi * math.e   # 13.8176


# ── STRUCTURED ENVIRONMENT MODEL ─────────────────────────────

# Manifests are kept as structured data, not strings.

# This is what enables merging, deduplication, conflict resolution.


@dataclass

class EnvironmentSpec:

    """

    Structured representation of what a NixOS environment needs.

    Multiple manifests are merged into a single EnvironmentSpec

    before any Nix is rendered.

    """

    # Identity

    name:          str   = "ryoko-generated"

    sources:       List[dict] = field(default_factory=list)  # source manifest metadata


    # Packages (deduplicated set)

    packages:      Set[str]  = field(default_factory=set)

    packages_32bit:Set[str]  = field(default_factory=set)   # 32-bit game deps


    # Graphics

    opengl:        bool = False

    opengl_32bit:  bool = False

    vulkan:        bool = False

    dxvk:          bool = False    # D3D9/10/11 → Vulkan

    vkd3d:         bool = False    # D3D12 → Vulkan


    # Audio

    pipewire:      bool = False

    alsa:          bool = False

    alsa_32bit:    bool = False

    pulse_compat:  bool = False    # PulseAudio compat layer

    jack:          bool = False


    # Runtime modes needed

    needs_steam:   bool = False

    needs_wine:    bool = False

    needs_proton:  bool = False


    # Lab / USB

    needs_usb:     bool = False

    needs_serial:  bool = False

    udev_rules:    List[str] = field(default_factory=list)


    # Python

    python_packages: Set[str] = field(default_factory=set)


    # Proprietary (require allowUnfree)

    proprietary:   List[str] = field(default_factory=list)


    # Unknown (need manual resolution)

    unknown:       List[str] = field(default_factory=list)


    # K-resonance tracking

    initial_k:     float = 0.0

    initial_sat:   float = 0.0


    # Conflicts detected during merge

    conflicts:     List[str] = field(default_factory=list)


    def merge(self, manifest: dict) -> "EnvironmentSpec":

        """

        Merge a manifest dict into this spec.

        Returns self (fluent interface).

        Deduplicates packages, merges boolean flags via OR,

        logs conflicts when detected.

        """

        # Track source

        self.sources.append({

            "name":     manifest.get("name","unknown"),

            "path":     manifest.get("path",""),

            "sha256":   manifest.get("sha256","")[:16],

            "exec_type":manifest.get("exec_type","unknown"),

            "compat_mode":manifest.get("compat_mode","unknown"),

            "initial_k":manifest.get("k_resonance",0.0),

        })


        # Packages

        for pkg in manifest.get("nix_packages",[]):

            if pkg.startswith("python3Packages."):

                self.python_packages.add(pkg)

            else:

                self.packages.add(pkg)


        # Unknown / proprietary

        self.unknown.extend(manifest.get("missing_pkgs",[]))

        self.proprietary.extend(manifest.get("proprietary",[]))


        # Graphics

        g = manifest.get("graphics",{})

        if g.get("opengl"):        self.opengl      = True

        if g.get("vulkan"):        self.vulkan      = True

        if g.get("dxvk_needed"):   self.dxvk        = True; self.opengl_32bit = True

        if g.get("vkd3d_needed"):  self.vkd3d       = True


        # 32-bit ELF needs 32-bit drivers

        if manifest.get("exec_type") == "elf32":

            self.opengl_32bit = True

            self.alsa_32bit   = True


        # Audio

        a = manifest.get("audio",{})

        if a.get("alsa"):      self.alsa         = True

        if a.get("pulse"):     self.pulse_compat  = True

        if a.get("pipewire"):  self.pipewire      = True

        if a.get("openal"):    self.packages.add("openal")


        # Compat mode

        mode = manifest.get("compat_mode","")

        if mode == "proton":   self.needs_steam  = True; self.needs_proton = True

        if mode == "wine":     self.needs_wine   = True


        # Lab

        libs = [d.get("name","") for d in manifest.get("libraries",[])]

        if any("usb" in l.lower() for l in libs):   self.needs_usb    = True

        if any("serial" in l.lower() for l in libs):self.needs_serial = True


        # Conflict detection: 32-bit wine + 64-bit only packages

        if self.needs_wine and any("lib64" in p for p in self.packages):

            self.conflicts.append(

                "wine (32-bit) may conflict with lib64-only packages"

            )


        # Running K-resonance: weight by sat score

        sat = manifest.get("satisfaction_score", 0.0)

        k   = manifest.get("k_resonance", 0.0)

        n   = len(self.sources)

        # Running mean

        self.initial_k   = (self.initial_k * (n-1) + k)   / n

        self.initial_sat = (self.initial_sat * (n-1) + sat) / n


        return self


    def deduplicate(self):

        """Remove duplicates and clean up after all merges."""

        self.unknown    = sorted(set(self.unknown))

        self.proprietary= sorted(set(self.proprietary))

        self.udev_rules = sorted(set(self.udev_rules))


        # Wine-only path: push deps into packages so _render_packages

        # handles them. This ensures exactly one systemPackages block

        # in the final flake — no silent overrides.

        if self.needs_wine and not self.needs_steam:

            self.packages.update(["wine64", "winetricks"])

            if self.dxvk:

                self.packages.add("dxvk")

            if self.vkd3d:

                self.packages.add("vkd3d-proton")


        # If pipewire present, it replaces pulseaudio and alsa daemons

        # but keeps compat layers

        if self.pipewire:

            self.packages.discard("pulseaudio")  # pipewire provides compat

            self.packages.discard("alsa-utils")  # pipewire-alsa replaces


        # Steam bundles its own 32-bit wine; don't add wine separately

        if self.needs_steam and "wine64" in self.packages:

            self.packages.discard("wine64")

            log.info("  Deduplicate: removed standalone wine64 (Steam bundles it)")


        return self



# ── FLAKE RENDERER ────────────────────────────────────────────


class FlakeRenderer:

    """

    Renders an EnvironmentSpec into a complete flake.nix.

    Keeps rendering separate from data so the spec can be

    inspected, tested, and modified before any file is written.

    """


    def render(self, spec: EnvironmentSpec, system: str = "x86_64-linux") -> str:

        """Render the complete flake.nix as a string."""

        timestamp = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

        sources_comment = self._render_sources_comment(spec)

        packages_block  = self._render_packages(spec)

        graphics_block  = self._render_graphics(spec)

        audio_block     = self._render_audio(spec)

        gaming_block    = self._render_gaming(spec)

        lab_block       = self._render_lab(spec)

        warnings_block  = self._render_warnings(spec)


        return f"""\

{{

  # RyokoSeven OS — Auto-generated environment

  # Generated:       {timestamp}

  # K-resonance:     {spec.initial_k:.4f} / {K_MAX:.4f}  ({spec.initial_sat*100:.1f}% satisfied)

  # Sources merged:  {len(spec.sources)}

  #

{sources_comment}

  description = "RyokoSeven generated environment — {spec.name}";


  inputs = {{

    nixpkgs.url = "github:NixOS/nixpkgs/nixos-unstable";

  }};


  outputs = {{ self, nixpkgs }}: {{

    nixosConfigurations.{spec.name} = nixpkgs.lib.nixosSystem {{

      system = "{system}";

      modules = [

        ({{ config, pkgs, lib, ... }}: {{


          nixpkgs.config.allowUnfree = {"true" if spec.proprietary else "false"};

{packages_block}

{graphics_block}

{audio_block}

{gaming_block}

{lab_block}

{warnings_block}

        }})

      ];

    }};

  }};

}}

"""


    def _render_sources_comment(self, spec: EnvironmentSpec) -> str:

        lines = ["  # Source manifests:"]

        for s in spec.sources:

            k_pct = s["initial_k"] / K_MAX * 100

            lines.append(

                f"  #   {s['name'][:30]:<30}  "

                f"{s['exec_type']:<8}  {s['compat_mode']:<10}  "

                f"K={k_pct:.0f}%  sha256:{s['sha256']}"

            )

        return "\n".join(lines)


    def _render_packages(self, spec: EnvironmentSpec) -> str:

        regular = sorted(spec.packages - {"mesa","vulkan-loader",

                                           "vulkan-tools","alsa-lib",

                                           "pipewire","pulseaudio"})

        python  = sorted(spec.python_packages)

        if not regular and not python:

            return ""


        lines = ["\n          environment.systemPackages = with pkgs; ["]

        for pkg in regular:

            lines.append(f"            {pkg}")

        if python:

            lines.append("")

            lines.append("            # Python / scientific stack")

            lines.append("            (python3.withPackages (ps: with ps; [")

            for pkg in python:

                short = pkg.replace("python3Packages.","")

                lines.append(f"              {short}")

            lines.append("            ]))")

        lines.append("          ];")

        return "\n".join(lines)


    def _render_graphics(self, spec: EnvironmentSpec) -> str:

        if not (spec.opengl or spec.vulkan):

            return ""

        lines = ["\n          hardware.opengl = {"]

        lines.append("            enable          = true;")

        if spec.opengl_32bit or spec.dxvk:

            lines.append("            driSupport      = true;")

            lines.append("            driSupport32Bit = true;  # 32-bit game support")

        lines.append("            extraPackages = with pkgs; [")

        lines.append("              mesa")

        if spec.vulkan:

            lines.append("              vulkan-loader")

            lines.append("              vulkan-tools")

        if spec.dxvk:

            lines.append("              dxvk   # D3D9/10/11 → Vulkan")

        if spec.vkd3d:

            lines.append("              vkd3d-proton  # D3D12 → Vulkan")

        lines.append("            ];")

        if spec.opengl_32bit:

            lines.append("            extraPackages32 = with pkgs.pkgsi686Linux; [")

            lines.append("              mesa vulkan-loader")

            lines.append("            ];")

        lines.append("          };")

        return "\n".join(lines)


    def _render_audio(self, spec: EnvironmentSpec) -> str:

        if not (spec.pipewire or spec.alsa or spec.pulse_compat):

            return ""

        if spec.pipewire:

            lines = ["\n          services.pipewire = {"]

            lines.append("            enable            = true;")

            if spec.alsa or spec.alsa_32bit:

                lines.append("            alsa.enable       = true;")

            if spec.alsa_32bit:

                lines.append("            alsa.support32Bit = true;")

            if spec.pulse_compat:

                lines.append("            pulse.enable      = true;  # PulseAudio compat")

            if spec.jack:

                lines.append("            jack.enable       = true;")

            lines.append("            wireplumber.enable = true;")

            lines.append("          };")

            lines.append("          security.rtkit.enable = true;")

            return "\n".join(lines)

        elif spec.alsa:

            return "\n          sound.enable = true;\n          hardware.pulseaudio.enable = true;"

        return ""


    def _render_gaming(self, spec: EnvironmentSpec) -> str:

        if not (spec.needs_steam or spec.needs_wine or spec.needs_proton):

            return ""

        # NOTE: wine deps are pushed into spec.packages by deduplicate()

        # before rendering, so _render_packages handles them.

        # This block only emits programs.* stanzas — never systemPackages —

        # ensuring there is exactly one systemPackages declaration per flake.

        lines = []

        if spec.needs_steam:

            lines += [

                "\n          programs.steam = {",

                "            enable                        = true;",

                "            remotePlay.openFirewall       = true;",

                "            dedicatedServer.openFirewall  = false;",

            ]

            if spec.needs_proton:

                lines.append(

                    "            extraCompatPackages = with pkgs; [ proton-ge-bin ];"

                )

            lines.append("          };")

            lines += [

                "",

                "          programs.gamemode.enable  = true;",

                "          programs.mangohud.enable  = true;",

            ]

        # wine-only case: deps already in spec.packages — nothing extra to emit

        return "\n".join(lines)


    def _render_lab(self, spec: EnvironmentSpec) -> str:

        if not (spec.needs_usb or spec.needs_serial):

            return ""

        lines = []

        if spec.needs_usb:

            lines += [

                "\n          # Lab instrument USB access",

                "          services.udev.packages = [ pkgs.libusb1 ];",

                "          services.udev.extraRules = ''",

                "            SUBSYSTEM==\"usb\", ATTR{idVendor}==\"2457\", "

                "MODE=\"0666\", GROUP=\"plugdev\"  # Ocean Optics",

                "            SUBSYSTEM==\"usb\", ATTR{idVendor}==\"0403\", "

                "MODE=\"0666\", GROUP=\"plugdev\"  # FTDI serial",

                "            SUBSYSTEM==\"usb\", MODE=\"0664\", GROUP=\"plugdev\"",

                "          '';",

                "          users.groups.plugdev = {};",

            ]

        if spec.needs_serial:

            lines += [

                "          services.udev.extraRules = ''",

                "            KERNEL==\"ttyUSB*\", MODE=\"0666\", GROUP=\"dialout\"",

                "            KERNEL==\"ttyACM*\", MODE=\"0666\", GROUP=\"dialout\"",

                "          '';",

                "          users.groups.dialout = {};",

            ]

        return "\n".join(lines)


    def _render_warnings(self, spec: EnvironmentSpec) -> str:

        lines = []

        if spec.unknown:

            lines.append("\n          # ⚠ Unknown packages — manual review needed:")

            for pkg in sorted(set(spec.unknown))[:10]:

                lines.append(f"          # TODO: {pkg}")

            if len(spec.unknown) > 10:

                lines.append(f"          # ... and {len(spec.unknown)-10} more")

        if spec.proprietary:

            lines.append("\n          # ⚠ Proprietary deps (allowUnfree=true already set):")

            for p in sorted(set(spec.proprietary)):

                lines.append(f"          # {p}")

        if spec.conflicts:

            lines.append("\n          # ⚠ Merge conflicts detected:")

            for c in spec.conflicts:

                lines.append(f"          # CONFLICT: {c}")

        return "\n".join(lines)



# ── CONVERGENCE LOOP ──────────────────────────────────────────


def convergence_loop(target: str, output_dir: Path,

                     max_iterations: int = 5) -> bool:

    """

    The full loop:

      scan → generate → apply → rescan → converge


    Returns True when K-resonance ≥ 0.8 × K_MAX.

    Runs up to max_iterations (each iteration = one nixos-rebuild).


    This is the P layer completing its function:

    it doesn't just describe what's needed —

    it builds the environment and verifies it converged.

    """

    CONVERGENCE_THRESHOLD = K_MAX * 0.8   # 11.054


    print(f"\n{'='*60}")

    print(f"  RyokoSeven Convergence Loop")

    print(f"  Target:    {target}")

    print(f"  Threshold: {CONVERGENCE_THRESHOLD:.4f} ({CONVERGENCE_THRESHOLD/K_MAX*100:.0f}% of K_max)")

    print(f"{'='*60}")


    k_prev       = 0.0   # ΔK tracking across iterations

    dk_history: list   = []  # last 3 ΔK values — oscillation detection

    pkg_snapshots: list = []  # package sets per iteration — conflict surface


    for iteration in range(1, max_iterations + 1):

        print(f"\n── Iteration {iteration}/{max_iterations} ──")


        # Step 1: Scan

        manifest_path = output_dir / f"manifest_iter{iteration}.json"

        print(f"  [NP] Scanning: {target}")

        result = subprocess.run(

            [sys.executable, str(Path(__file__).parent / "ryoko_scanner.py"),

             target, "--output", str(manifest_path)],

            capture_output=True, text=True

        )

        if result.returncode != 0 or not manifest_path.exists():

            print(f"  [NP] Scanner failed: {result.stderr[:200]}")

            return False


        manifest  = json.loads(manifest_path.read_text())

        k_current = manifest.get("k_resonance", 0.0)

        sat       = manifest.get("satisfaction_score", 0.0)

        bar_w     = 30

        filled    = int(sat * bar_w)

        bar       = "█" * filled + "░" * (bar_w - filled)


        # ΔK: delta from previous iteration

        delta_k = (k_current - k_prev) if iteration > 1 else 0.0

        k_prev  = k_current


        # Accumulate directional history for oscillation detection

        if iteration > 1:

            dk_history.append(delta_k)

            if len(dk_history) > 3:

                dk_history.pop(0)


        def _is_oscillating(history):

            """Alternating signs + shrinking magnitude = conflict cycle.

            e.g. [+0.8, -0.6, +0.4] — something is being added then removed.

            Distinct from plateau (stable near-zero) and regression (consistent drop)."""

            if len(history) < 2:

                return False

            sig = [d for d in history if abs(d) > 0.05]

            if len(sig) < 2:

                return False

            alternates = all(sig[i] * sig[i+1] < 0 for i in range(len(sig)-1))

            shrinking  = all(abs(sig[i]) >= abs(sig[i+1]) for i in range(len(sig)-1))

            return alternates and shrinking


        # Four-regime classification — ordered by specificity

        if iteration == 1:

            signal    = ""

            delta_str = "  ΔK: baseline"

        elif _is_oscillating(dk_history):

            signal    = "↯ oscillation — conflicting deps or incompatible modes"

            delta_str = f"  ΔK: {delta_k:+.4f}"

        elif delta_k >  1.0:

            signal    = "▲ core dep fixed"

            delta_str = f"  ΔK: {delta_k:+.4f}"

        elif delta_k >  0.1:

            signal    = "↑ fine-tuning"

            delta_str = f"  ΔK: {delta_k:+.4f}"

        elif delta_k < -0.1:

            signal    = "▼ regression — check last change"

            delta_str = f"  ΔK: {delta_k:+.4f}"

        else:

            signal    = "→ plateau — remaining deps outside search space"

            delta_str = f"  ΔK: {delta_k:+.4f}"


        print(f"  [K]  K-resonance: [{bar}] {k_current:.4f} "

              f"({k_current/K_MAX*100:.1f}%)  {delta_str}  {signal}")


        # Oscillation: stop churning, compute conflict surface, hand back

        if "oscillation" in signal:

            print(f"\n  ↯  Oscillation detected — halting auto-iteration")

            print(f"     Likely cause: 32-bit/64-bit conflict, Wine vs Steam,")

            print(f"     or proprietary dep with no clean Nix equivalent.")

            print(f"     Missing:     {manifest.get('missing_pkgs',[])[:5]}")

            print(f"     Proprietary: {manifest.get('proprietary',[])}")


            # Conflict surface: symmetric difference across oscillating states

            # Compare last 2-3 package snapshots to localize the conflict.

            # The packages that keep appearing and disappearing are the candidates.

            if len(pkg_snapshots) >= 2:

                recent = pkg_snapshots[-3:]  # last 3 states

                # Packages present in some iterations but absent in others

                all_pkgs  = set().union(*[s["packages"] for s in recent])

                stable    = set.intersection(*[s["packages"] for s in recent])

                unstable  = all_pkgs - stable   # these are the conflict candidates


                # Rank by how often they flip: appears in half the snapshots = most conflicted

                n = len(recent)

                flip_counts = {

                    pkg: sum(1 for s in recent if pkg in s["packages"])

                    for pkg in unstable

                }

                # Most conflicted = present in ~half the iterations (0.3–0.7 of n)

                conflict_candidates = sorted(

                    [pkg for pkg, cnt in flip_counts.items()

                     if 0 < cnt < n],

                    key=lambda p: abs(flip_counts[p] / n - 0.5)  # closest to 50%

                )


                print(f"\n     Conflict surface (packages that oscillate):")

                if conflict_candidates:

                    for pkg in conflict_candidates[:8]:

                        present_in = flip_counts[pkg]

                        arrow = "+" if pkg in pkg_snapshots[-1]["packages"] else "-"

                        print(f"       {arrow} {pkg:<40} "

                              f"(present {present_in}/{n} iterations)")

                    print(f"\n     These packages are the conflict candidates.")

                    print(f"     The truth disagreement lives in this set.")

                    print(f"     Try removing one and rebuilding.")

                else:

                    print(f"       (conflict candidates unclear — inspect flake manually)")


            # Resolution branching: partition snapshots into competing configs.

            # High-K snapshots without the conflict candidate = one valid path.

            # Low-K snapshots with the conflict candidate = the other.

            # Present both as concrete options rather than just "something conflicts."

            if len(pkg_snapshots) >= 2 and conflict_candidates:

                # Partition: above/below median K across recent snapshots

                k_vals  = [s["k"] for s in recent]

                k_med   = sorted(k_vals)[len(k_vals)//2]

                high_k  = [s for s in recent if s["k"] >= k_med]

                low_k   = [s for s in recent if s["k"] <  k_med]


                if high_k and low_k:

                    # Stable packages in each cluster = that cluster's core

                    high_stable = set.intersection(*[s["packages"] for s in high_k])

                    low_stable  = set.intersection(*[s["packages"] for s in low_k])


                    # Packages that define each path (present in one, absent in other)

                    defines_high = sorted(high_stable - low_stable)

                    defines_low  = sorted(low_stable  - high_stable)

                    shared       = sorted(high_stable & low_stable)


                    print(f"\n     Competing stable configurations detected:")

                    print(f"     (Both are valid — system needs a preference)")

                    print()


                    if defines_high:

                        avg_k_high = sum(s["k"] for s in high_k) / len(high_k)

                        print(f"     [A] Higher-K path  (avg K = {avg_k_high:.3f})")

                        for pkg in defines_high[:6]:

                            print(f"         + {pkg}")


                    if defines_low:

                        avg_k_low = sum(s["k"] for s in low_k) / len(low_k)

                        print(f"\n     [B] Lower-K path   (avg K = {avg_k_low:.3f})")

                        for pkg in defines_low[:6]:

                            print(f"         + {pkg}")


                    if shared:

                        print(f"\n     Shared by both paths:")

                        for pkg in shared[:6]:

                            print(f"         = {pkg}")


                    print(f"\n     To choose path A:")

                    print(f"       Remove from flake: {defines_low[:3]}")

                    print(f"     To choose path B:")

                    print(f"       Remove from flake: {defines_high[:3]}")


            print(f"\n     Flake: {output_dir}/generated/flake.nix")

            return False


        # Step 2: Check convergence

        if k_current >= CONVERGENCE_THRESHOLD:

            print(f"\n  ✓ CONVERGED at iteration {iteration}")

            print(f"    K = {k_current:.4f} ≥ {CONVERGENCE_THRESHOLD:.4f}")

            return True


        # Step 3: Generate flake

        print(f"  [P]  Generating flake...")

        flake_path = output_dir / f"flake_iter{iteration}.nix"

        flake_out  = generate_from_manifests(

            [str(manifest_path)],

            str(output_dir / "generated"),

            name=f"ryoko-iter{iteration}"

        )


        # Snapshot current package set for conflict surface analysis

        snap_packages = set(manifest.get("nix_packages", []))

        pkg_snapshots.append({

            "iteration": iteration,

            "k":         k_current,

            "packages":  snap_packages,

        })


        # Step 4: Apply (nixos-rebuild) — only if NixOS is running

        nixos_running = Path("/etc/nixos/configuration.nix").exists()

        if nixos_running:

            print(f"  [P]  Applying: nixos-rebuild switch...")

            apply = subprocess.run(

                ["sudo", "nixos-rebuild", "switch",

                 "--flake", f"{output_dir}/generated#ryoko-iter{iteration}"],

                capture_output=True, text=True

            )

            if apply.returncode != 0:

                print(f"  [P]  Apply failed: {apply.stderr[:300]}")

                print(f"       Check {output_dir}/generated/flake.nix manually")

                return False

            print(f"  [P]  Applied successfully")

        else:

            print(f"  [P]  Not on NixOS — generated flake at:")

            print(f"       {output_dir}/generated/flake.nix")

            print(f"       Run: cd {output_dir}/generated && "

                  f"sudo nixos-rebuild switch --flake .#ryoko-iter{iteration}")

            # In non-NixOS mode, we can only do one iteration

            return False


    print(f"\n  ⚠ Did not converge after {max_iterations} iterations")

    print(f"    Final K = {k_current:.4f} / {CONVERGENCE_THRESHOLD:.4f}")

    print(f"    Remaining issues: {manifest.get('missing_pkgs',[])}")

    return False



# ── MAIN GENERATOR ────────────────────────────────────────────


def generate_from_manifests(manifest_paths: List[str],

                              output_dir: str,

                              name: str = "ryoko-generated",

                              system: str = "x86_64-linux") -> Path:

    """

    Load one or more manifests, merge into EnvironmentSpec,

    render flake.nix, write to output_dir.

    Returns path to generated flake.nix.

    """

    out = Path(output_dir)

    out.mkdir(parents=True, exist_ok=True)


    spec = EnvironmentSpec(name=name)


    for mp in manifest_paths:

        path = Path(mp)

        if not path.exists():

            log.warning(f"Manifest not found: {mp}")

            continue

        manifest = json.loads(path.read_text())

        log.info(f"  Merging: {manifest.get('name','?')} "

                 f"(K={manifest.get('k_resonance',0):.3f})")

        spec.merge(manifest)


    spec.deduplicate()


    if spec.conflicts:

        log.warning(f"  Conflicts detected during merge:")

        for c in spec.conflicts:

            log.warning(f"    ⚠ {c}")


    renderer = FlakeRenderer()

    flake_content = renderer.render(spec, system=system)


    flake_path = out / "flake.nix"

    flake_path.write_text(flake_content)

    log.info(f"  Generated: {flake_path}")


    # Also write a minimal hardware config stub

    hw_stub = out / "hardware-configuration.nix"

    if not hw_stub.exists():

        hw_stub.write_text(

            "# Run: sudo nixos-generate-config --show-hardware-config > "

            "hardware-configuration.nix\n"

            "{ ... }: { }\n"

        )


    return flake_path



# ── DEMO ──────────────────────────────────────────────────────


def demo():

    """

    Full loop demo using the scanner's simulated manifests.

    Shows merge, deduplication, conflict detection, and rendering.

    """

    import tempfile, os


    print("RyokoSeven Flake Generator — Demo")

    print(f"K_max = {K_MAX:.4f}")

    print()


    # Simulate three manifests (as scanner would produce)

    manifests = [

        {

            "name": "NativeLinuxGame",

            "path": "/opt/games/native",

            "sha256": "edfba530638bf475" + "0"*48,

            "exec_type": "elf64",

            "compat_mode": "native",

            "k_resonance": 9.211,

            "satisfaction_score": 0.667,

            "nix_packages": ["SDL2","mesa","vulkan-loader","vulkan-tools",

                              "alsa-lib","gcc.cc","glibc"],

            "graphics": {"opengl":True,"vulkan":True,"dxvk_needed":False,

                         "vkd3d_needed":False},

            "audio": {"alsa":True,"pulse":False,"pipewire":False,"openal":False},

            "libraries": [{"name":"libGL.so","nix_pkg":"mesa","found":True},

                          {"name":"libvulkan.so","nix_pkg":"vulkan-loader","found":True},

                          {"name":"libasound.so","nix_pkg":"alsa-lib","found":True},

                          {"name":"libSDL2","nix_pkg":"SDL2","found":True},

                          {"name":"libstdc++","nix_pkg":"gcc.cc","found":False},

                          {"name":"libc.so","nix_pkg":"glibc","found":False}],

            "missing_pkgs": [],

            "proprietary": [],

        },

        {

            "name": "WindowsGame_Proton",

            "path": "/opt/games/wingame",

            "sha256": "578e45d8bc8d26c2" + "0"*48,

            "exec_type": "pe64",

            "compat_mode": "proton",

            "steam_appid": "12345",

            "k_resonance": 8.521,

            "satisfaction_score": 0.617,

            "nix_packages": ["dxvk","wine"],

            "graphics": {"opengl":False,"vulkan":True,"dxvk_needed":True,

                         "vkd3d_needed":False},

            "audio": {"alsa":False,"pulse":True,"pipewire":False,"openal":False},

            "libraries": [{"name":"d3d11.dll","nix_pkg":"dxvk","found":False},

                          {"name":"steam_api64.dll","nix_pkg":None,"found":False},

                          {"name":"kernel32.dll","nix_pkg":"wine","found":True}],

            "missing_pkgs": ["steam_api64.dll"],

            "proprietary": ["fmod.dll"],

        },

        {

            "name": "FoundrySpectrometer",

            "path": "/opt/lab/adapter.py",

            "sha256": "17bb1fb6ac90ec68" + "0"*48,

            "exec_type": "script",

            "compat_mode": "native",

            "k_resonance": 6.909,

            "satisfaction_score": 0.500,

            "nix_packages": ["python3Packages.numpy","python3Packages.scipy",

                             "python3Packages.matplotlib","python3Packages.pyserial",

                             "python3Packages.pyusb","python3Packages.requests"],

            "graphics": {"opengl":False,"vulkan":False,"dxvk_needed":False,

                         "vkd3d_needed":False},

            "audio": {"alsa":False,"pulse":False,"pipewire":False,"openal":False},

            "libraries": [{"name":"serial","nix_pkg":"python3Packages.pyserial","found":False},

                          {"name":"usb","nix_pkg":"python3Packages.pyusb","found":False},

                          {"name":"numpy","nix_pkg":"python3Packages.numpy","found":True}],

            "missing_pkgs": [],

            "proprietary": [],

        },

    ]


    with tempfile.TemporaryDirectory() as tmpdir:

        # Write manifests

        manifest_paths = []

        for m in manifests:

            p = Path(tmpdir) / f"{m['name']}.json"

            p.write_text(json.dumps(m))

            manifest_paths.append(str(p))


        # Generate merged flake

        out_dir = Path(tmpdir) / "generated"

        flake_path = generate_from_manifests(

            manifest_paths, str(out_dir),

            name="ryoko-demo"

        )


        print("Merged EnvironmentSpec:")

        spec = EnvironmentSpec(name="ryoko-demo")

        for m in manifests:

            spec.merge(m)

        spec.deduplicate()


        print(f"  Sources merged:  {len(spec.sources)}")

        print(f"  Packages:        {len(spec.packages)}")

        print(f"  Python packages: {len(spec.python_packages)}")

        print(f"  Needs Steam:     {spec.needs_steam}")

        print(f"  Needs DXVK:      {spec.dxvk}")

        print(f"  Needs USB:       {spec.needs_usb}")

        print(f"  Proprietary:     {spec.proprietary}")

        print(f"  Unknown:         {spec.unknown}")

        print(f"  Conflicts:       {spec.conflicts or 'none'}")

        print(f"  Initial K:       {spec.initial_k:.4f} ({spec.initial_k/K_MAX*100:.1f}%)")

        print()


        # Print generated flake

        flake_content = flake_path.read_text()

        print("Generated flake.nix:")

        print("─" * 60)

        print(flake_content)

        print("─" * 60)


        print(f"\nNext steps:")

        print(f"  1. cp {flake_path} /etc/nixos/flake.nix")

        print(f"  2. sudo nixos-rebuild switch --flake /etc/nixos#ryoko-demo")

        print(f"  3. python3 ryoko_scanner.py /opt/games/native")

        print(f"     → K-resonance should rise toward {K_MAX:.4f}")

        print()

        print(f"Or full loop:")

        print(f"  python3 ryoko_flake_gen.py --loop /opt/games/native")

        print(f"  → scan → generate → apply → rescan → converge ✓")



if __name__ == "__main__":

    import argparse

    parser = argparse.ArgumentParser(

        description="RyokoSeven Flake Generator — closes the convergence loop")

    parser.add_argument("manifests", nargs="*",

                        help="manifest.json files to merge")

    parser.add_argument("--output",  default="./ryoko-generated",

                        help="Output directory (default: ./ryoko-generated)")

    parser.add_argument("--name",    default="ryoko-generated",

                        help="NixOS config name")

    parser.add_argument("--system",  default="x86_64-linux",

                        choices=["x86_64-linux","aarch64-linux","i686-linux"])

    parser.add_argument("--loop",    metavar="TARGET",

                        help="Run full convergence loop on a target executable")

    parser.add_argument("--demo",    action="store_true")

    args = parser.parse_args()


    if args.demo or not (args.manifests or args.loop):

        demo()

    elif args.loop:

        out = Path(args.output)

        converged = convergence_loop(args.loop, out)

        sys.exit(0 if converged else 1)

    else:

        flake = generate_from_manifests(

            args.manifests, args.output,

            name=args.name, system=args.system

        )

        print(f"Generated: {flake}")

        print(f"Apply:     cd {args.output} && "

              f"sudo nixos-rebuild switch --flake .#{args.name}")

In [ ]:
#!/usr/bin/env python3

"""

RyokoSeven OS — Compatibility Manifest Scanner

===============================================

Scans executables, game directories, and application bundles to produce

a structured compatibility manifest. The manifest feeds the adapter factory,

which then generates NixOS configuration to satisfy all dependencies.


Works on:

  · Native Linux ELF binaries (games, apps, lab software)

  · Windows PE executables (→ Proton/Wine requirements)

  · Steam game directories (via appmanifest + ACF files)

  · AppImage / Flatpak bundles

  · Python / script-based lab instruments


Output: JSON manifest → NixOS expression → adapter factory input


Philosophy (NixOS alignment):

  · Every dependency is explicit — nothing hidden

  · Manifest is reproducible — same binary = same manifest

  · Failures are informative — missing dep = clear error with fix

  · K-resonance score: how well the system currently satisfies the manifest


Usage:

  python ryoko_scanner.py /path/to/game

  python ryoko_scanner.py --steam 440          # scan TF2 by AppID

  python ryoko_scanner.py --dir ~/Games        # scan entire directory

  python ryoko_scanner.py --self               # scan running system

"""


import json

import os

import re

import struct

import subprocess

import sys

import hashlib

import logging

from dataclasses import dataclass, field, asdict

from enum import Enum

from pathlib import Path

from typing import Optional, List, Dict, Set, Tuple

import math


log = logging.getLogger("ryoko.scanner")

logging.basicConfig(level=logging.INFO,

                    format="%(asctime)s %(levelname)-7s %(message)s",

                    datefmt="%H:%M:%S")


# ── CONSTANTS ─────────────────────────────────────────────────

PHI = (1 + math.sqrt(5)) / 2

K_MAX = PHI * math.pi * math.e   # 13.8176


# Known API → NixOS package mappings

LIBRARY_TO_NIX = {

    # Graphics

    "libGL.so":          "mesa",

    "libGLX.so":         "mesa",

    "libvulkan.so":      "vulkan-loader",

    "libVkLayer":        "vulkan-validation-layers",

    "libEGL.so":         "mesa",

    "libGLESv2.so":      "mesa",

    # Audio

    "libasound.so":      "alsa-lib",

    "libpulse.so":       "pulseaudio",

    "libpipewire":       "pipewire",

    "libopenal.so":      "openal",

    "libSDL2":           "SDL2",

    # Windowing / input

    "libX11.so":         "xorg.libX11",

    "libXext.so":        "xorg.libXext",

    "libXrandr.so":      "xorg.libXrandr",

    "libXi.so":          "xorg.libXi",

    "libwayland":        "wayland",

    "libxkbcommon":      "libxkbcommon",

    # Physics / game engines

    "libsteam_api":      "steam-run",

    "libfmod":           "fmod",  # proprietary — flag for review

    "libphysfs":         "physfs",

    # Networking

    "libcurl.so":        "curl",

    "libssl.so":         "openssl",

    "libcrypto.so":      "openssl",

    # Runtime

    "libc.so":           "glibc",

    "libstdc++":         "gcc.cc",

    "libgcc_s":          "gcc.cc",

    "libm.so":           "glibc",

    "libpthread":        "glibc",

    # Python / lab

    "libpython":         "python3",

    "libhdf5":           "hdf5",

    "libopenblas":       "openblas",

    # Wine / Proton

    "ntdll.dll":         "wine",

    "kernel32.dll":      "wine",

    "d3d11.dll":         "dxvk",

    "d3d12.dll":         "vkd3d",

    "xinput":            "wine",

    # USB / serial (lab instruments)

    "libusb":            "libusb1",

    "libserialport":     "libserialport",

    "libudev.so":        "systemd",

}


STEAM_COMMON = Path.home() / ".steam/steam/steamapps/common"

STEAM_ACF    = Path.home() / ".steam/steam/steamapps"



# ── DATA MODEL ────────────────────────────────────────────────


class ExecType(Enum):

    ELF_64       = "elf64"

    ELF_32       = "elf32"

    PE_64        = "pe64"          # Windows 64-bit → Proton

    PE_32        = "pe32"          # Windows 32-bit → Wine

    SCRIPT       = "script"        # Python/shell/etc

    APPIMAGE     = "appimage"

    FLATPAK      = "flatpak"

    UNKNOWN      = "unknown"


class CompatMode(Enum):

    NATIVE       = "native"        # runs directly on Linux

    PROTON       = "proton"        # Steam Proton (Valve's Wine fork)

    WINE         = "wine"          # standard Wine

    APPIMAGE     = "appimage"      # self-contained bundle

    FLATPAK      = "flatpak"       # sandboxed container

    NEEDS_REVIEW = "needs_review"  # scanner couldn't determine


@dataclass

class Dependency:

    name:     str

    nix_pkg:  Optional[str]    # NixOS package name, None if unknown

    found:    bool = False     # whether it's currently satisfied on this system

    version:  Optional[str] = None

    critical: bool = True      # if False, optional/enhancement only

    note:     str  = ""


@dataclass

class GraphicsRequirements:

    opengl:       bool = False

    opengl_ver:   Optional[str] = None

    vulkan:       bool = False

    vulkan_ver:   Optional[str] = None

    d3d9:         bool = False

    d3d10:        bool = False

    d3d11:        bool = False

    d3d12:        bool = False    # → vkd3d-proton

    dxvk_needed:  bool = False    # D3D9/10/11 on Linux

    vkd3d_needed: bool = False    # D3D12 on Linux


@dataclass

class AudioRequirements:

    alsa:      bool = False

    pulse:     bool = False

    pipewire:  bool = False

    openal:    bool = False

    fmod:      bool = False       # proprietary — needs special handling

    sdl_audio: bool = False


@dataclass

class CompatibilityManifest:

    # Identity

    name:          str = ""

    path:          str = ""

    sha256:        str = ""

    exec_type:     ExecType     = ExecType.UNKNOWN

    compat_mode:   CompatMode   = CompatMode.NEEDS_REVIEW


    # Requirements

    libraries:     List[Dependency]         = field(default_factory=list)

    graphics:      GraphicsRequirements     = field(default_factory=GraphicsRequirements)

    audio:         AudioRequirements        = field(default_factory=AudioRequirements)

    python_deps:   List[str]                = field(default_factory=list)

    steam_appid:   Optional[str]            = None

    proton_ver:    Optional[str]            = None   # e.g. "8.0", "Experimental"


    # NixOS output

    nix_packages:  List[str]                = field(default_factory=list)

    nix_expression: Optional[str]          = None

    missing_pkgs:  List[str]               = field(default_factory=list)

    proprietary:   List[str]               = field(default_factory=list)


    # K-resonance

    satisfaction_score: float  = 0.0   # [0,1] how well system satisfies manifest

    k_resonance:        float  = 0.0   # K_MAX × satisfaction_score

    notes:              List[str] = field(default_factory=list)

    warnings:           List[str] = field(default_factory=list)


    def to_dict(self) -> dict:

        d = asdict(self)

        d["exec_type"]   = self.exec_type.value

        d["compat_mode"] = self.compat_mode.value

        return d



# ── ELF PARSER ───────────────────────────────────────────────


def read_elf_header(path: Path) -> Optional[Dict]:

    """Parse ELF header to determine arch and type."""

    try:

        with open(path, "rb") as f:

            magic = f.read(4)

            if magic != b"\x7fELF":

                return None

            ei_class = struct.unpack("B", f.read(1))[0]  # 1=32bit, 2=64bit

            f.seek(16)

            e_type    = struct.unpack("<H", f.read(2))[0]

            e_machine = struct.unpack("<H", f.read(2))[0]

        return {

            "bits":    64 if ei_class == 2 else 32,

            "type":    e_type,

            "machine": e_machine,

        }

    except Exception:

        return None



def extract_elf_dependencies(path: Path) -> List[str]:

    """Extract NEEDED shared libraries from ELF dynamic section."""

    try:

        result = subprocess.run(

            ["objdump", "-p", str(path)],

            capture_output=True, text=True, timeout=10

        )

        deps = re.findall(r"NEEDED\s+(\S+)", result.stdout)

        return deps

    except FileNotFoundError:

        # objdump not available — try readelf

        try:

            result = subprocess.run(

                ["readelf", "-d", str(path)],

                capture_output=True, text=True, timeout=10

            )

            deps = re.findall(r"Shared library:\s+\[(.+?)\]", result.stdout)

            return deps

        except Exception:

            return []

    except Exception as e:

        log.debug(f"ELF dep extraction failed for {path}: {e}")

        return []



def check_library_present(lib: str) -> Tuple[bool, Optional[str]]:

    """Check if a shared library is present on the current system."""

    try:

        result = subprocess.run(

            ["ldconfig", "-p"],

            capture_output=True, text=True, timeout=5

        )

        for line in result.stdout.splitlines():

            if lib.rstrip(".0123456789") in line:

                # Extract version if present

                ver_match = re.search(r"=> (.+)", line)

                path = ver_match.group(1).strip() if ver_match else None

                return True, path

        return False, None

    except Exception:

        # ldconfig not available — check common paths

        search_paths = [

            Path("/lib"), Path("/lib64"),

            Path("/usr/lib"), Path("/usr/lib64"),

            Path("/usr/local/lib"),

        ]

        for sp in search_paths:

            if list(sp.glob(f"{lib.split('.so')[0]}*")):

                return True, str(sp)

        return False, None



# ── PE PARSER ────────────────────────────────────────────────


def read_pe_header(path: Path) -> Optional[Dict]:

    """Parse PE header for Windows executables."""

    try:

        with open(path, "rb") as f:

            # MZ header

            if f.read(2) != b"MZ":

                return None

            f.seek(0x3C)

            pe_offset = struct.unpack("<I", f.read(4))[0]

            f.seek(pe_offset)

            if f.read(4) != b"PE\x00\x00":

                return None

            machine = struct.unpack("<H", f.read(2))[0]

        return {

            "bits":    64 if machine == 0x8664 else 32,

            "machine": machine,

        }

    except Exception:

        return None



def extract_pe_imports(path: Path) -> List[str]:

    """Extract imported DLLs from PE file."""

    try:

        # Use strings as a quick fallback (not perfect but no wine needed)

        result = subprocess.run(

            ["strings", str(path)],

            capture_output=True, text=True, timeout=10

        )

        dlls = re.findall(r"(\w+\.dll)", result.stdout, re.IGNORECASE)

        return list(set(dlls))

    except Exception:

        return []



# ── STEAM SCANNER ────────────────────────────────────────────


def scan_steam_appmanifest(appid: str) -> Optional[Dict]:

    """Parse Steam ACF manifest for game metadata."""

    acf_file = STEAM_ACF / f"appmanifest_{appid}.acf"

    if not acf_file.exists():

        # Try finding it

        candidates = list(STEAM_ACF.glob(f"appmanifest_{appid}.acf"))

        if not candidates:

            log.warning(f"Steam appmanifest not found for AppID {appid}")

            return None

        acf_file = candidates[0]


    try:

        content = acf_file.read_text(errors="replace")

        def parse_acf(text):

            result = {}

            for match in re.finditer(r'"(\w+)"\s+"([^"]*)"', text):

                result[match.group(1)] = match.group(2)

            return result

        return parse_acf(content)

    except Exception as e:

        log.warning(f"ACF parse error: {e}")

        return None



def detect_proton_version(appid: str) -> Optional[str]:

    """Detect configured Proton version for a Steam game."""

    config_path = Path.home() / ".steam/steam/config/config.vdf"

    if not config_path.exists():

        return None

    try:

        content = config_path.read_text(errors="replace")

        # Look for CompatToolMapping entry

        pattern = rf'"{appid}".*?"name"\s+"([^"]+)"'

        m = re.search(pattern, content, re.DOTALL)

        if m:

            return m.group(1)

        return "Proton (version unspecified)"

    except Exception:

        return None



# ── GRAPHICS DETECTION ───────────────────────────────────────


def detect_graphics_requirements(libs: List[str],

                                  imports: List[str]) -> GraphicsRequirements:

    g = GraphicsRequirements()

    all_names = [l.lower() for l in libs + imports]


    for name in all_names:

        if "libgl" in name or "opengl" in name:

            g.opengl = True

        if "vulkan" in name or "vk" in name:

            g.vulkan = True

        if "d3d9" in name:

            g.d3d9 = True; g.dxvk_needed = True

        if "d3d10" in name:

            g.d3d10 = True; g.dxvk_needed = True

        if "d3d11" in name:

            g.d3d11 = True; g.dxvk_needed = True

        if "d3d12" in name:

            g.d3d12 = True; g.vkd3d_needed = True


    return g



def detect_audio_requirements(libs: List[str],

                               imports: List[str]) -> AudioRequirements:

    a = AudioRequirements()

    all_names = [l.lower() for l in libs + imports]


    for name in all_names:

        if "libasound" in name or "alsa" in name:    a.alsa = True

        if "libpulse" in name or "pulse" in name:    a.pulse = True

        if "pipewire" in name:                        a.pipewire = True

        if "openal" in name:                          a.openal = True

        if "fmod" in name:                            a.fmod = True

        if "sdl2" in name or "sdl" in name:          a.sdl_audio = True


    return a



# ── NIX EXPRESSION GENERATOR ─────────────────────────────────


def generate_nix_expression(manifest: "CompatibilityManifest") -> str:

    """

    Generate a NixOS configuration fragment for this application.

    Output can be included in configuration.nix or a flake.

    """

    pkgs = sorted(set(manifest.nix_packages))

    name = manifest.name.replace(" ","_").replace("-","_").lower()


    lines = [

        f"# RyokoSeven auto-generated — {manifest.name}",

        f"# Manifest hash: {manifest.sha256[:16]}",

        f"# K-resonance:   {manifest.k_resonance:.4f} / {K_MAX:.4f}",

        f"# Compat mode:   {manifest.compat_mode.value}",

        "",

    ]


    if manifest.compat_mode in (CompatMode.PROTON, CompatMode.WINE):

        lines += [

            "# Windows compatibility — add to configuration.nix:",

            "programs.steam = {",

            "  enable = true;",

            "  remotePlay.openFirewall = true;",

            "  dedicatedServer.openFirewall = false;",

            "};",

            "",

            "# Or for Wine without Steam:",

            "environment.systemPackages = with pkgs; [",

            "  wine64",

            "  winetricks",

            "  dxvk",

        ]

        if manifest.graphics.vkd3d_needed:

            lines.append("  vkd3d-proton  # D3D12 support")

        lines += ["];", ""]


    elif manifest.compat_mode == CompatMode.NATIVE:

        lines += [

            "# Native Linux — add to configuration.nix:",

            "environment.systemPackages = with pkgs; [",

        ]

        for pkg in pkgs:

            if pkg not in ("glibc", "gcc.cc"):  # implicit

                lines.append(f"  {pkg}")

        lines += ["];", ""]


    # Hardware acceleration (always include for games)

    if manifest.graphics.vulkan or manifest.graphics.opengl:

        lines += [

            "# GPU / graphics acceleration:",

            "hardware.opengl = {",

            "  enable = true;",

            "  driSupport = true;",

            "  driSupport32Bit = true;",  # for 32-bit game binaries

            "};",

            "",

        ]


    # Audio

    if manifest.audio.pipewire or manifest.audio.pulse:

        lines += [

            "# Audio (PipeWire — recommended for games):",

            "services.pipewire = {",

            "  enable = true;",

            "  alsa.enable = true;",

            "  alsa.support32Bit = true;",

            "  pulse.enable = true;  # PulseAudio compatibility",

            "};",

            "",

        ]


    # Lab instruments

    usb_pkgs = [d for d in manifest.libraries if "usb" in d.name.lower()]

    if usb_pkgs:

        lines += [

            "# USB / lab instrument access:",

            "services.udev.packages = [ pkgs.libusb1 ];",

            "users.users.YOUR_USER.extraGroups = [ \"plugdev\" \"dialout\" ];",

            "",

        ]


    if manifest.missing_pkgs:

        lines += [

            "# ⚠ UNKNOWN PACKAGES — manual review needed:",

        ]

        for pkg in manifest.missing_pkgs:

            lines.append(f"# TODO: find NixOS equivalent for: {pkg}")

        lines.append("")


    if manifest.proprietary:

        lines += [

            "# ⚠ PROPRIETARY dependencies detected:",

            "nixpkgs.config.allowUnfree = true;",

        ]

        for p in manifest.proprietary:

            lines.append(f"# {p} — check license before enabling")

        lines.append("")


    return "\n".join(lines)



# ── K-RESONANCE SCORING ──────────────────────────────────────


def compute_k_resonance(manifest: "CompatibilityManifest") -> Tuple[float, float]:

    """

    Score how well the current system satisfies the manifest.

    Returns (satisfaction [0,1], k_resonance [0, K_MAX]).


    High score = system is ready to run this program.

    Low score = gaps need to be filled (listed in missing_pkgs).

    """

    if not manifest.libraries:

        return 0.5, K_MAX * 0.5   # unknown = neutral


    total     = len(manifest.libraries)

    satisfied = sum(1 for d in manifest.libraries if d.found)

    critical_total = sum(1 for d in manifest.libraries if d.critical)

    critical_sat   = sum(1 for d in manifest.libraries if d.critical and d.found)


    # Critical deps weighted 2× optional

    if critical_total > 0:

        sat = (critical_sat * 2 + (satisfied - critical_sat)) / (critical_total * 2 + (total - critical_total))

    else:

        sat = satisfied / max(total, 1)


    # Penalty for proprietary unknown deps

    prop_penalty = len(manifest.proprietary) * 0.05

    sat = max(0.0, sat - prop_penalty)


    k_res = K_MAX * sat

    return float(sat), float(k_res)



# ── MAIN SCANNER ─────────────────────────────────────────────


class ManifestScanner:

    """

    Scans a program or directory and produces a CompatibilityManifest.

    """


    def scan(self, target: str,

             steam_appid: Optional[str] = None) -> CompatibilityManifest:

        path = Path(target).resolve()

        manifest = CompatibilityManifest(

            name = path.name,

            path = str(path),

        )


        if steam_appid:

            manifest.steam_appid = steam_appid

            return self._scan_steam(manifest, steam_appid)


        if path.is_dir():

            return self._scan_directory(manifest, path)


        return self._scan_file(manifest, path)


    def _scan_file(self, manifest: CompatibilityManifest,

                   path: Path) -> CompatibilityManifest:

        if not path.exists():

            manifest.warnings.append(f"File not found: {path}")

            return manifest


        # Hash for reproducibility

        try:

            h = hashlib.sha256(path.read_bytes()).hexdigest()

            manifest.sha256 = h

        except Exception:

            manifest.sha256 = "unreadable"


        # Detect type

        with open(path, "rb") as f:

            magic = f.read(4)


        if magic[:4] == b"\x7fELF":

            return self._scan_elf(manifest, path)

        elif magic[:2] == b"MZ":

            return self._scan_pe(manifest, path)

        elif magic[:2] in (b"#!", b"PK"):

            return self._scan_script(manifest, path)

        elif path.suffix.lower() == ".appimage":

            manifest.exec_type   = ExecType.APPIMAGE

            manifest.compat_mode = CompatMode.APPIMAGE

            manifest.notes.append("AppImage: self-contained, minimal host deps needed")

            return manifest

        else:

            manifest.exec_type = ExecType.UNKNOWN

            manifest.warnings.append("Unknown binary format — manual inspection needed")

            return manifest


    def _scan_elf(self, manifest: CompatibilityManifest,

                  path: Path) -> CompatibilityManifest:

        header = read_elf_header(path)

        if header:

            manifest.exec_type = ExecType.ELF_64 if header["bits"]==64 else ExecType.ELF_32

        manifest.compat_mode = CompatMode.NATIVE


        raw_deps = extract_elf_dependencies(path)

        log.info(f"  ELF deps found: {len(raw_deps)}")


        self._process_libraries(manifest, raw_deps)

        manifest.graphics = detect_graphics_requirements(raw_deps, [])

        manifest.audio    = detect_audio_requirements(raw_deps, [])

        self._build_nix_packages(manifest)

        manifest.nix_expression = generate_nix_expression(manifest)


        sat, k = compute_k_resonance(manifest)

        manifest.satisfaction_score = sat

        manifest.k_resonance        = k


        return manifest


    def _scan_pe(self, manifest: CompatibilityManifest,

                 path: Path) -> CompatibilityManifest:

        header = read_pe_header(path)

        if header:

            manifest.exec_type = ExecType.PE_64 if header["bits"]==64 else ExecType.PE_32


        imports = extract_pe_imports(path)

        log.info(f"  PE imports found: {len(imports)}")


        manifest.graphics = detect_graphics_requirements([], imports)


        # Determine Wine vs Proton

        if manifest.steam_appid:

            manifest.compat_mode = CompatMode.PROTON

            manifest.proton_ver  = detect_proton_version(manifest.steam_appid)

        else:

            manifest.compat_mode = CompatMode.WINE


        # Map PE imports to Linux equivalents

        self._process_libraries(manifest, imports)

        manifest.audio = detect_audio_requirements([], imports)


        # Wine/Proton always needed

        wine_dep = Dependency(

            name="wine" if manifest.compat_mode==CompatMode.WINE else "proton",

            nix_pkg="wine64" if manifest.compat_mode==CompatMode.WINE else "steam",

            critical=True, note="Windows compatibility layer"

        )

        wine_dep.found, _ = check_library_present("libwine")

        manifest.libraries.append(wine_dep)


        if manifest.graphics.dxvk_needed:

            dxvk = Dependency(name="dxvk", nix_pkg="dxvk",

                              critical=True, note="D3D9/10/11 → Vulkan translation")

            manifest.libraries.append(dxvk)


        if manifest.graphics.vkd3d_needed:

            vkd3d = Dependency(name="vkd3d-proton", nix_pkg="vkd3d-proton",

                               critical=True, note="D3D12 → Vulkan translation")

            manifest.libraries.append(vkd3d)


        self._build_nix_packages(manifest)

        manifest.nix_expression = generate_nix_expression(manifest)

        sat, k = compute_k_resonance(manifest)

        manifest.satisfaction_score = sat

        manifest.k_resonance        = k

        return manifest


    def _scan_script(self, manifest: CompatibilityManifest,

                     path: Path) -> CompatibilityManifest:

        manifest.exec_type   = ExecType.SCRIPT

        manifest.compat_mode = CompatMode.NATIVE

        content = path.read_text(errors="replace")


        # Python imports

        py_imports = re.findall(r"^(?:import|from)\s+(\w+)", content, re.MULTILINE)

        manifest.python_deps = list(set(py_imports))


        # Map common Python packages to Nix

        PY_TO_NIX = {

            "numpy":      "python3Packages.numpy",

            "scipy":      "python3Packages.scipy",

            "matplotlib": "python3Packages.matplotlib",

            "cv2":        "python3Packages.opencv4",

            "serial":     "python3Packages.pyserial",

            "usb":        "python3Packages.pyusb",

            "requests":   "python3Packages.requests",

            "fastapi":    "python3Packages.fastapi",

            "torch":      "python3Packages.torch",

        }

        for imp in manifest.python_deps:

            nix = PY_TO_NIX.get(imp)

            dep = Dependency(name=imp, nix_pkg=nix,

                             critical=False,

                             note="Python import")

            if nix:

                dep.found, _ = check_library_present(f"python3/{imp}")

            manifest.libraries.append(dep)


        manifest.nix_packages  = [d.nix_pkg for d in manifest.libraries if d.nix_pkg]

        manifest.nix_expression = generate_nix_expression(manifest)

        sat, k = compute_k_resonance(manifest)

        manifest.satisfaction_score = sat

        manifest.k_resonance        = k

        return manifest


    def _scan_directory(self, manifest: CompatibilityManifest,

                        path: Path) -> CompatibilityManifest:

        """Scan a game/app directory — find the main executable."""

        log.info(f"Scanning directory: {path}")

        manifest.name = path.name


        # Find executables

        executables = []

        for pattern in ["*.exe", "*.EXE"]:

            executables += list(path.rglob(pattern))

        for p in path.iterdir():

            if p.is_file() and os.access(p, os.X_OK):

                with open(p, "rb") as f:

                    mg = f.read(4)

                if mg[:4] == b"\x7fELF":

                    executables.append(p)


        if not executables:

            manifest.warnings.append("No executables found in directory")

            return manifest


        # Scan the most likely main executable (largest, or named like the dir)

        main_exec = max(executables, key=lambda p: p.stat().st_size)

        log.info(f"  Main executable: {main_exec.name}")


        sub = self._scan_file(

            CompatibilityManifest(name=manifest.name,

                                   path=str(main_exec)),

            main_exec

        )

        sub.name  = manifest.name

        sub.path  = str(path)

        sub.notes.append(f"Scanned from directory — main exec: {main_exec.name}")

        sub.notes.append(f"Total executables found: {len(executables)}")

        return sub


    def _scan_steam(self, manifest: CompatibilityManifest,

                    appid: str) -> CompatibilityManifest:

        """Scan a Steam game by AppID."""

        acf = scan_steam_appmanifest(appid)

        if acf:

            manifest.name = acf.get("name", f"Steam:{appid}")

            install_dir   = acf.get("installdir", "")

            game_path     = STEAM_COMMON / install_dir

            if game_path.exists():

                sub = self._scan_directory(

                    CompatibilityManifest(name=manifest.name,

                                          path=str(game_path),

                                          steam_appid=appid),

                    game_path

                )

                sub.steam_appid = appid

                sub.proton_ver  = detect_proton_version(appid)

                return sub


        manifest.warnings.append(f"Steam game {appid} not installed or not found")

        return manifest


    def _process_libraries(self, manifest: CompatibilityManifest,

                            raw_libs: List[str]):

        """Map raw library names to Dependency objects."""

        for lib in raw_libs:

            nix_pkg = None

            for key, pkg in LIBRARY_TO_NIX.items():

                if key.lower() in lib.lower():

                    nix_pkg = pkg

                    break


            found, path = check_library_present(lib.split(".so")[0])

            dep = Dependency(

                name    = lib,

                nix_pkg = nix_pkg,

                found   = found,

                critical= not any(x in lib.lower()

                                  for x in ["optional","extra","plugin"]),

                note    = path or "",

            )


            if nix_pkg == "fmod" or "fmod" in lib.lower():

                manifest.proprietary.append(lib)

                dep.note = "Proprietary — check FMOD license"


            if nix_pkg is None and not found:

                manifest.missing_pkgs.append(lib)


            manifest.libraries.append(dep)


    def _build_nix_packages(self, manifest: CompatibilityManifest):

        """Collect all required Nix packages from dependencies."""

        pkgs = set()

        for dep in manifest.libraries:

            if dep.nix_pkg:

                pkgs.add(dep.nix_pkg)

        if manifest.graphics.vulkan:

            pkgs.update(["vulkan-loader", "vulkan-tools"])

        if manifest.graphics.opengl:

            pkgs.add("mesa")

        if manifest.audio.pulse:

            pkgs.add("pulseaudio")

        if manifest.audio.pipewire:

            pkgs.add("pipewire")

        manifest.nix_packages = sorted(pkgs)



# ── REPORT PRINTER ────────────────────────────────────────────


def print_report(manifest: CompatibilityManifest):

    K = K_MAX

    bar_w  = 40

    filled = int(manifest.satisfaction_score * bar_w)

    bar    = "█" * filled + "░" * (bar_w - filled)

    k_pct  = manifest.k_resonance / K * 100


    print(f"\n{'='*60}")

    print(f"  RyokoSeven Compatibility Manifest")

    print(f"{'='*60}")

    print(f"  Name:       {manifest.name}")

    print(f"  Type:       {manifest.exec_type.value}")

    print(f"  Mode:       {manifest.compat_mode.value}")

    if manifest.steam_appid:

        print(f"  Steam ID:   {manifest.steam_appid}")

    if manifest.proton_ver:

        print(f"  Proton:     {manifest.proton_ver}")

    print(f"  Hash:       {manifest.sha256[:24]}...")

    print()

    print(f"  K-resonance: [{bar}] {k_pct:.1f}%")

    print(f"               {manifest.k_resonance:.4f} / {K:.4f}")

    print()


    if manifest.graphics.vulkan or manifest.graphics.opengl:

        g = manifest.graphics

        flags = []

        if g.opengl:  flags.append("OpenGL")

        if g.vulkan:  flags.append("Vulkan")

        if g.d3d11:   flags.append("D3D11→DXVK")

        if g.d3d12:   flags.append("D3D12→VKD3D")

        print(f"  Graphics:   {' · '.join(flags)}")


    if any([manifest.audio.alsa, manifest.audio.pulse,

            manifest.audio.pipewire, manifest.audio.openal]):

        a = manifest.audio

        afl = []

        if a.alsa:     afl.append("ALSA")

        if a.pulse:    afl.append("PulseAudio")

        if a.pipewire: afl.append("PipeWire")

        if a.openal:   afl.append("OpenAL")

        if a.fmod:     afl.append("FMOD⚠")

        print(f"  Audio:      {' · '.join(afl)}")


    print()

    print(f"  Dependencies ({len(manifest.libraries)} total):")

    sat   = sum(1 for d in manifest.libraries if d.found)

    unsat = len(manifest.libraries) - sat

    print(f"    Satisfied:   {sat}")

    print(f"    Missing:     {unsat}")

    if manifest.proprietary:

        print(f"    Proprietary: {len(manifest.proprietary)} ⚠")


    print()

    print(f"  NixOS packages ({len(manifest.nix_packages)}):")

    for pkg in manifest.nix_packages:

        print(f"    · {pkg}")


    if manifest.missing_pkgs:

        print()

        print(f"  ⚠ Unknown packages ({len(manifest.missing_pkgs)}) — need manual mapping:")

        for p in manifest.missing_pkgs[:5]:

            print(f"    ? {p}")

        if len(manifest.missing_pkgs) > 5:

            print(f"    ... and {len(manifest.missing_pkgs)-5} more")


    if manifest.warnings:

        print()

        print("  Warnings:")

        for w in manifest.warnings:

            print(f"    ⚠ {w}")


    if manifest.notes:

        print()

        for n in manifest.notes:

            print(f"    · {n}")


    print()

    print("  NixOS configuration fragment:")

    print("  " + "─"*56)

    for line in (manifest.nix_expression or "").splitlines():

        print(f"  {line}")

    print(f"{'='*60}\n")



# ── DEMO ──────────────────────────────────────────────────────


def demo():

    """Demo scan without real binaries — shows the full pipeline."""

    print("RyokoSeven OS — Manifest Scanner Demo")

    print(f"K_max = {K_MAX:.4f}")

    print()


    scanner = ManifestScanner()


    # ── Demo 1: Simulate a native Linux game (e.g. Steam Linux Runtime) ──

    log.info("Demo 1: Native Linux game (simulated)")

    m1 = CompatibilityManifest(

        name="NativeLinuxGame",

        path="/opt/games/native_game",

        sha256=hashlib.sha256(b"demo_native").hexdigest(),

        exec_type=ExecType.ELF_64,

        compat_mode=CompatMode.NATIVE,

    )

    # Simulate detected dependencies

    demo_libs = [

        "libGL.so.1", "libvulkan.so.1", "libasound.so.2",

        "libSDL2-2.0.so.0", "libstdc++.so.6", "libc.so.6",

    ]

    scanner._process_libraries(m1, demo_libs)

    m1.graphics = detect_graphics_requirements(demo_libs, [])

    m1.audio    = detect_audio_requirements(demo_libs, [])

    scanner._build_nix_packages(m1)

    # Simulate 4/6 deps satisfied

    for i, dep in enumerate(m1.libraries):

        dep.found = i < 4

    m1.nix_expression = generate_nix_expression(m1)

    m1.satisfaction_score, m1.k_resonance = compute_k_resonance(m1)

    print_report(m1)


    # ── Demo 2: Windows game via Proton ──────────────────────

    log.info("Demo 2: Windows game via Proton (simulated)")

    m2 = CompatibilityManifest(

        name="WindowsGame_via_Proton",

        path="/opt/games/windows_game/game.exe",

        sha256=hashlib.sha256(b"demo_pe").hexdigest(),

        exec_type=ExecType.PE_64,

        compat_mode=CompatMode.PROTON,

        steam_appid="12345",

        proton_ver="Proton 8.0",

    )

    pe_imports = ["d3d11.dll","xinput1_3.dll","kernel32.dll",

                  "steam_api64.dll","fmod.dll"]

    scanner._process_libraries(m2, pe_imports)

    m2.graphics = detect_graphics_requirements([], pe_imports)

    m2.audio    = detect_audio_requirements([], pe_imports)

    # Add DXVK since D3D11 detected

    m2.libraries.append(Dependency(

        name="dxvk", nix_pkg="dxvk",

        found=False, critical=True,

        note="D3D11 → Vulkan translation"

    ))

    scanner._build_nix_packages(m2)

    for i, dep in enumerate(m2.libraries):

        dep.found = i % 3 != 0   # simulate partial satisfaction

    m2.nix_expression = generate_nix_expression(m2)

    m2.satisfaction_score, m2.k_resonance = compute_k_resonance(m2)

    print_report(m2)


    # ── Demo 3: Lab instrument (Python script) ────────────────

    log.info("Demo 3: Lab instrument — Python adapter")

    m3 = CompatibilityManifest(

        name="FoundrySpectrometer",

        path="/opt/lab/adapter.py",

        sha256=hashlib.sha256(b"demo_py").hexdigest(),

        exec_type=ExecType.SCRIPT,

        compat_mode=CompatMode.NATIVE,

    )

    m3.python_deps = ["numpy","scipy","matplotlib","serial","usb","requests"]

    PY_NIX = {

        "numpy":"python3Packages.numpy",

        "scipy":"python3Packages.scipy",

        "matplotlib":"python3Packages.matplotlib",

        "serial":"python3Packages.pyserial",

        "usb":"python3Packages.pyusb",

        "requests":"python3Packages.requests",

    }

    for imp in m3.python_deps:

        dep = Dependency(name=imp, nix_pkg=PY_NIX.get(imp),

                         critical=False, note="Python import")

        dep.found = imp in ("numpy","requests","scipy")

        m3.libraries.append(dep)

    m3.nix_packages = sorted(

        {d.nix_pkg for d in m3.libraries if d.nix_pkg}

    )

    m3.nix_expression = generate_nix_expression(m3)

    m3.satisfaction_score, m3.k_resonance = compute_k_resonance(m3)

    print_report(m3)


    # ── Summary ───────────────────────────────────────────────

    manifests = [m1, m2, m3]

    print("BATCH SUMMARY")

    print("─"*50)

    print(f"{'Program':<28} {'Mode':<12} {'K-res':>8}  {'Sat':>6}")

    print("─"*50)

    for m in manifests:

        print(f"  {m.name:<26} {m.compat_mode.value:<12} "

              f"{m.k_resonance:>8.3f}  {m.satisfaction_score*100:>5.1f}%")

    avg_k = sum(m.k_resonance for m in manifests) / len(manifests)

    print("─"*50)

    print(f"  {'Average K-resonance':<26} {'':12} {avg_k:>8.3f}")

    print(f"  {'K_max':<26} {'':12} {K_MAX:>8.4f}")

    print(f"  {'Coverage':<26} {'':12} {avg_k/K_MAX*100:>7.1f}%")

    print()

    print("Next: feed manifests → AdapterFactory → NixOS flake")

    print("      ryoko_scanner.py /path/to/game --output manifest.json")



if __name__ == "__main__":

    import argparse

    parser = argparse.ArgumentParser(description="RyokoSeven Manifest Scanner")

    parser.add_argument("target",       nargs="?", help="Path to scan")

    parser.add_argument("--steam",      help="Steam AppID")

    parser.add_argument("--output",     help="Save manifest as JSON")

    parser.add_argument("--demo",       action="store_true")

    args = parser.parse_args()


    if args.demo or (not args.target and not args.steam):

        demo()

    else:

        scanner  = ManifestScanner()

        manifest = scanner.scan(args.target or ".", steam_appid=args.steam)

        print_report(manifest)

        if args.output:

            Path(args.output).write_text(

                json.dumps(manifest.to_dict(), indent=2)

            )

            print(f"Manifest saved: {args.output}")

In [ ]:


"""

RyokoSeven Auto-Interface Layer

================================

Automatic discovery, adapter generation, and dynamic registration

of hardware, network services, and file system resources.


Design principles:

  · Template-first: known device classes get pre-written adapters (safe, instant)

  · LLM-assisted: unknown APIs get generated adapters (sandboxed, user-confirmed)

  · Hard security boundary: generated code NEVER runs without explicit approval

  · Graceful degradation: failed/disconnected adapters log and deregister cleanly

  · Zero restart: all adapter changes happen at runtime


Architecture:

  DiscoveryModule        → scans environment every N seconds

  AdapterFactory         → instantiates or generates adapters per resource

  AdapterRegistry        → manages lifecycle, failure, hot-swap

  RyokoSevenAutoInterface→ extends RyokoSeven with discovery loop


Connects to existing Foundry adapter (adapter.py):

  SpectrometerAdapter wraps the existing HTTP bridge at localhost:7430

  Any new spectrometer auto-detected → same interface, new data stream


Security model:

  SAFE   → template adapters (USBSensor, Spectrometer, REST known formats)

  REVIEW → LLM-generated adapters (shown to user, require explicit 'yes')

  REJECT → anything that imports os/subprocess/sys/socket (blocked by sandbox)

"""


import time

import json

import logging

import threading

import hashlib

import textwrap

from abc import ABC, abstractmethod

from dataclasses import dataclass, field

from enum import Enum

from pathlib import Path

from typing import List, Optional, Dict, Any, Callable


import numpy as np


log = logging.getLogger("ryoko.autoif")


# ── Try optional dependencies gracefully ─────────────────────

try:

    import usb.core

    USB_AVAILABLE = True

except ImportError:

    USB_AVAILABLE = False


try:

    import serial.tools.list_ports

    SERIAL_AVAILABLE = True

except ImportError:

    SERIAL_AVAILABLE = False


try:

    import requests

    REQUESTS_AVAILABLE = True

except ImportError:

    REQUESTS_AVAILABLE = False


try:

    from zeroconf import Zeroconf, ServiceBrowser, ServiceListener

    MDNS_AVAILABLE = True

except ImportError:

    MDNS_AVAILABLE = False


# ── KNOWN DEVICE REGISTRY ─────────────────────────────────────

# VID:PID → template class name. Add new instruments here.

KNOWN_USB_DEVICES = {

    "2457:1002": "OceanOpticsSpectrometer",   # Ocean Optics USB4000/Flame

    "2457:4000": "OceanOpticsSpectrometer",

    "0403:6001": "HoribaSpectrometer",         # FTDI serial (Horiba)

    "04b4:8613": "PrincetonInstruments",       # Cypress FX2 (PI)

    "0000:0001": "MockSensor",                 # test device

}


# Known REST API patterns (from OpenAPI "info.title" or URL patterns)

KNOWN_API_PATTERNS = {

    "foundry":      "FoundryBridgeAdapter",   # our own adapter.py

    "pubchem":      "PubChemAdapter",

    "chembl":       "ChEMBLAdapter",

    "string-db":    "STRINGAdapter",

    "swissadme":    "SwissADMEAdapter",

}



# ============================================================

# RESOURCE TYPES

# ============================================================


class ResourceType(Enum):

    USB_DEVICE   = "usb_device"

    SERIAL_PORT  = "serial_port"

    REST_API     = "rest_api"

    MDNS_SERVICE = "mdns_service"

    FILE_WATCH   = "file_watch"

    FOUNDRY_HTTP = "foundry_http"


class SecurityLevel(Enum):

    SAFE    = "safe"     # template adapter, runs immediately

    REVIEW  = "review"   # LLM-generated, requires user confirmation

    REJECT  = "reject"   # blocked by sandbox


@dataclass

class DiscoveredResource:

    type:       ResourceType

    id:         str              # unique identifier (e.g., "USB:2457:1002:001")

    name:       str              # human-readable

    metadata:   Dict[str,Any] = field(default_factory=dict)

    security:   SecurityLevel  = SecurityLevel.SAFE


    def __hash__(self):

        return hash(self.id)


    def __eq__(self, other):

        return isinstance(other, DiscoveredResource) and self.id == other.id



# ============================================================

# ADAPTER BASE CLASS

# ============================================================


class Adapter(ABC):

    """

    Minimal interface all adapters must implement.

    read() returns 12-channel intensity array — compatible with

    RyokoSeven NP layer and PhotonicNPLayer.

    """

    def __init__(self, resource: DiscoveredResource):

        self.resource    = resource

        self.active      = True

        self._last_read  = None

        self._error_count= 0

        self.MAX_ERRORS  = 5


    @abstractmethod

    def read(self) -> Optional[np.ndarray]:

        """Return N-channel float array in [0,1], or None on failure."""

        ...


    @abstractmethod

    def describe(self) -> str:

        """Human-readable description of this adapter and its resource."""

        ...


    def matches(self, resource: DiscoveredResource) -> bool:

        return self.resource.id == resource.id


    def health_check(self) -> bool:

        """Return True if adapter is functional."""

        return self.active and self._error_count < self.MAX_ERRORS


    def _handle_error(self, e: Exception):

        self._error_count += 1

        log.warning(f"  Adapter {self.resource.name}: error {self._error_count}/{self.MAX_ERRORS}: {e}")

        if self._error_count >= self.MAX_ERRORS:

            self.active = False

            log.error(f"  Adapter {self.resource.name}: DEACTIVATED after {self.MAX_ERRORS} errors")


    def reset_errors(self):

        self._error_count = 0

        self.active = True



# ── TEMPLATE ADAPTERS ─────────────────────────────────────────


class MockSensor(Adapter):

    """Test adapter — returns slowly-evolving sine wave data."""

    def read(self) -> Optional[np.ndarray]:

        t    = time.time()

        vals = np.array([0.5 + 0.4*np.sin(t*0.3 + i*0.8) for i in range(12)])

        return np.clip(vals, 0, 1)


    def describe(self) -> str:

        return f"Mock 12-channel sine sensor [{self.resource.id}]"



class FoundryBridgeAdapter(Adapter):

    """

    Wraps the existing Foundry HTTP adapter (adapter.py).

    Converts R, β, curvature → 12-channel NP intensities.

    Same mapping as PhotonicNPLayer in photonic_np.py.

    """

    def __init__(self, resource: DiscoveredResource):

        super().__init__(resource)

        self.url = resource.metadata.get("url", "http://localhost:7430")


    def read(self) -> Optional[np.ndarray]:

        if not REQUESTS_AVAILABLE:

            return None

        try:

            r = requests.get(f"{self.url}/state", timeout=2)

            d = r.json()

            R    = float(d.get("R")         or 1.0)

            beta = float(d.get("beta")      or 0.9)

            curv = float(d.get("curvature") or 0.0)


            R_n   = np.clip((R - 0.3) / 7.7, 0, 1)

            β_div = 1.0 - np.clip(beta, 0.3, 1.0)

            c_coh = np.clip(-curv / 0.1, 0, 1)


            intensities = np.array([

                0.9*(1-R_n), 0.7*(1-R_n)*0.8, 0.6*(R_n*0.5+0.3),

                0.8*R_n, 0.9*c_coh, 0.7*β_div,

                0.5*(R_n*β_div), 0.3, 0.5, 0.4, 0.6*c_coh, 0.3*R_n,

            ])

            intensities += np.random.normal(0, 0.015, 12)

            intensities  = np.maximum(intensities, 0)

            self._last_read = intensities / (intensities.sum() + 1e-10)

            self._error_count = 0

            return self._last_read

        except Exception as e:

            self._handle_error(e)

            return None


    def describe(self) -> str:

        return f"Foundry HTTP Bridge → {self.url} (R, β, curvature → 12-ch NP)"



class OceanOpticsSpectrometer(Adapter):

    """Ocean Optics USB spectrometer adapter (USB4000, Flame, etc.)"""

    def __init__(self, resource: DiscoveredResource):

        super().__init__(resource)

        self._dev = None

        self._connect()


    def _connect(self):

        if not USB_AVAILABLE:

            log.warning("pyusb not available — OceanOptics in mock mode")

            return

        try:

            vid, pid = [int(x,16) for x in resource.metadata.get("vidpid","2457:1002").split(":")]

            self._dev = usb.core.find(idVendor=vid, idProduct=pid)

            if self._dev:

                log.info(f"  Connected: Ocean Optics {self.resource.name}")

        except Exception as e:

            log.warning(f"  Ocean Optics connect failed: {e}")


    def read(self) -> Optional[np.ndarray]:

        if self._dev is None:

            # Mock spectrum: Gaussian at 615nm (Eu3+ F2 line)

            wl   = np.linspace(550, 750, 12)

            vals = np.exp(-0.5*((wl-615)/20)**2) + 0.05*np.random.rand(12)

            return vals / vals.sum()

        try:

            # Real read: send acquisition command, parse response

            # (device-specific — Ocean Optics protocol)

            # self._dev.write(1, [0x09])  # trigger

            # raw = self._dev.read(0x81, 2048, timeout=1000)

            # spectrum = parse_oceanoptics(raw)

            return None   # placeholder until device present

        except Exception as e:

            self._handle_error(e)

            return None


    def describe(self) -> str:

        return (f"Ocean Optics Spectrometer [{self.resource.id}]"

                f" — {'connected' if self._dev else 'mock mode'}")



class FileWatchAdapter(Adapter):

    """

    Watches a directory for CSV files.

    Compatible with adapter.py file format.

    Returns last parsed spectrum as 12-channel array.

    """

    def __init__(self, resource: DiscoveredResource):

        super().__init__(resource)

        self.watch_dir = Path(resource.metadata.get("path", "./data"))

        self._last_file= None

        self._cache    = None


    def read(self) -> Optional[np.ndarray]:

        try:

            csvs = sorted(self.watch_dir.glob("cw_*.csv"),

                          key=lambda f: f.stat().st_mtime, reverse=True)

            if not csvs:

                return self._cache

            newest = csvs[0]

            if newest != self._last_file:

                self._last_file = newest

                data = np.loadtxt(newest, delimiter=",", comments="#")

                wl, I = data[:,0], data[:,1]

                # Bin into 12 channels: 555–725 nm, ~14nm each

                edges = np.linspace(555, 725, 13)

                binned = np.array([

                    I[(wl>=edges[i]) & (wl<edges[i+1])].mean()

                    if np.any((wl>=edges[i]) & (wl<edges[i+1])) else 0.0

                    for i in range(12)

                ])

                binned = np.maximum(binned, 0)

                self._cache = binned / (binned.sum() + 1e-10)

            return self._cache

        except Exception as e:

            self._handle_error(e)

            return None


    def describe(self) -> str:

        return f"File watcher → {self.watch_dir} (CW CSV → 12-ch bins)"



class GenericRESTAdapter(Adapter):

    """

    Generic REST adapter for discovered APIs.

    Tries to extract a numeric array from the response.

    """

    def __init__(self, resource: DiscoveredResource):

        super().__init__(resource)

        self.url     = resource.metadata.get("url", "")

        self.key_path= resource.metadata.get("data_key", None)


    def read(self) -> Optional[np.ndarray]:

        if not REQUESTS_AVAILABLE or not self.url:

            return None

        try:

            r = requests.get(self.url, timeout=3)

            d = r.json()

            # Navigate to data key if specified

            if self.key_path:

                for k in self.key_path.split("."):

                    d = d[k]

            arr = np.array(d, dtype=float).flatten()[:12]

            if len(arr) < 12:

                arr = np.pad(arr, (0, 12-len(arr)), constant_values=0.5)

            arr = np.clip(arr, 0, 1)

            return arr / (arr.sum() + 1e-10)

        except Exception as e:

            self._handle_error(e)

            return None


    def describe(self) -> str:

        return f"REST API → {self.url} (generic JSON → 12-ch)"



# ── TEMPLATE REGISTRY ─────────────────────────────────────────

ADAPTER_TEMPLATES = {

    "OceanOpticsSpectrometer": OceanOpticsSpectrometer,

    "MockSensor":              MockSensor,

    "FoundryBridgeAdapter":    FoundryBridgeAdapter,

    "FileWatchAdapter":        FileWatchAdapter,

    "GenericRESTAdapter":      GenericRESTAdapter,

}



# ============================================================

# SECURITY SANDBOX FOR LLM-GENERATED CODE

# ============================================================


BLOCKED_IMPORTS = {

    "os","subprocess","sys","socket","shutil","ctypes",

    "importlib","__builtin__","builtins","eval","exec",

}


def sandbox_check(code: str) -> SecurityLevel:

    """

    Static analysis of LLM-generated adapter code.

    Returns SAFE, REVIEW, or REJECT.

    """

    code_lower = code.lower()

    # Hard reject: dangerous imports or builtins

    for blocked in BLOCKED_IMPORTS:

        if f"import {blocked}" in code_lower or f"from {blocked}" in code_lower:

            log.warning(f"  Sandbox REJECT: found 'import {blocked}'")

            return SecurityLevel.REJECT

    if "__import__" in code or "eval(" in code or "exec(" in code:

        log.warning("  Sandbox REJECT: dynamic execution detected")

        return SecurityLevel.REJECT

    # Flag for review: file access, network beyond requests

    if any(p in code for p in ["open(","pathlib","urllib","socket","ftplib"]):

        return SecurityLevel.REVIEW

    return SecurityLevel.REVIEW   # all LLM-generated code requires REVIEW minimum



def present_for_approval(code: str, resource: DiscoveredResource,

                          auto_approve: bool = False) -> bool:

    """

    Show generated adapter code to user and request confirmation.

    Returns True if approved.


    In production: show in UI with syntax highlighting.

    Here: terminal prompt.

    """

    if auto_approve:

        log.warning("  AUTO-APPROVE mode: skipping security review (testing only)")

        return True


    print("\n" + "="*60)

    print(f"GENERATED ADAPTER FOR: {resource.name}")

    print(f"Resource: {resource.id}")

    print("="*60)

    print(textwrap.indent(code, "  "))

    print("="*60)

    print("⚠ This code was generated by an LLM and has not been manually reviewed.")

    response = input("Approve and load this adapter? [yes/NO]: ").strip().lower()

    return response == "yes"



def execute_sandboxed(code: str, resource: DiscoveredResource) -> Optional[type]:

    """

    Execute generated adapter code in a restricted namespace.

    Returns the Adapter subclass if successful, None otherwise.

    """

    # Restricted globals — no os, subprocess, open, etc.

    safe_globals = {

        "__builtins__": {

            "print": print, "len": len, "range": range,

            "int": int, "float": float, "str": str,

            "list": list, "dict": dict, "None": None,

            "True": True, "False": False,

            "isinstance": isinstance, "hasattr": hasattr,

            "getattr": getattr, "min": min, "max": max,

        },

        "np": np,

        "requests": requests if REQUESTS_AVAILABLE else None,

        "Adapter": Adapter,

        "DiscoveredResource": DiscoveredResource,

        "Optional": Optional,

        "log": log,

    }

    local_ns = {}

    try:

        exec(compile(code, "<generated>", "exec"), safe_globals, local_ns)

        # Find the Adapter subclass in the namespace

        for obj in local_ns.values():

            if isinstance(obj, type) and issubclass(obj, Adapter) and obj is not Adapter:

                return obj

        log.warning("  Sandboxed exec: no Adapter subclass found in generated code")

        return None

    except Exception as e:

        log.error(f"  Sandboxed exec failed: {e}")

        return None



# ============================================================

# DISCOVERY MODULE

# ============================================================


class DiscoveryModule:

    """

    Scans environment for connectable resources.

    Runs USB, serial, network, filesystem, and config-based discovery.

    """

    def __init__(self, config: dict = None):

        self.config       = config or {}

        self.watch_dirs   = self.config.get("watch_dirs", ["./data"])

        self.api_endpoints= self.config.get("api_endpoints", [])

        self.scan_network = self.config.get("scan_network", False)


    def scan(self) -> List[DiscoveredResource]:

        resources = []

        resources += self._scan_usb()

        resources += self._scan_serial()

        resources += self._scan_foundry_http()

        resources += self._scan_filesystem()

        resources += self._scan_configured_apis()

        if self.scan_network:

            resources += self._scan_mdns()

        log.info(f"Discovery: found {len(resources)} resource(s)")

        return resources


    def _scan_usb(self) -> List[DiscoveredResource]:

        if not USB_AVAILABLE:

            return []

        found = []

        try:

            devices = usb.core.find(find_all=True)

            for dev in devices:

                vidpid = f"{dev.idVendor:04x}:{dev.idProduct:04x}"

                if vidpid in KNOWN_USB_DEVICES:

                    found.append(DiscoveredResource(

                        type    = ResourceType.USB_DEVICE,

                        id      = f"USB:{vidpid}:{dev.address:03d}",

                        name    = KNOWN_USB_DEVICES[vidpid],

                        metadata= {"vidpid": vidpid, "address": dev.address,

                                   "manufacturer": getattr(dev,"manufacturer",""),

                                   "product": getattr(dev,"product","")},

                        security= SecurityLevel.SAFE,

                    ))

        except Exception as e:

            log.debug(f"USB scan error: {e}")

        return found


    def _scan_serial(self) -> List[DiscoveredResource]:

        if not SERIAL_AVAILABLE:

            return []

        found = []

        try:

            ports = serial.tools.list_ports.comports()

            for port in ports:

                if port.vid and port.pid:

                    vidpid = f"{port.vid:04x}:{port.pid:04x}"

                    template = KNOWN_USB_DEVICES.get(vidpid)

                    if template:

                        found.append(DiscoveredResource(

                            type    = ResourceType.SERIAL_PORT,

                            id      = f"SERIAL:{port.device}",

                            name    = template,

                            metadata= {"port": port.device, "vidpid": vidpid,

                                       "description": port.description},

                            security= SecurityLevel.SAFE,

                        ))

        except Exception as e:

            log.debug(f"Serial scan error: {e}")

        return found


    def _scan_foundry_http(self) -> List[DiscoveredResource]:

        """Check if our own Foundry adapter is running."""

        if not REQUESTS_AVAILABLE:

            return []

        urls = self.config.get("foundry_urls", ["http://localhost:7430"])

        found = []

        for url in urls:

            try:

                r = requests.get(f"{url}/health", timeout=1)

                if r.status_code == 200:

                    found.append(DiscoveredResource(

                        type    = ResourceType.FOUNDRY_HTTP,

                        id      = f"FOUNDRY:{url}",

                        name    = "Foundry HTTP Bridge",

                        metadata= {"url": url, "health": r.json()},

                        security= SecurityLevel.SAFE,

                    ))

                    log.info(f"  Foundry bridge detected at {url}")

            except Exception:

                pass

        return found


    def _scan_filesystem(self) -> List[DiscoveredResource]:

        found = []

        for watch_dir in self.watch_dirs:

            p = Path(watch_dir)

            if p.exists() and p.is_dir():

                found.append(DiscoveredResource(

                    type    = ResourceType.FILE_WATCH,

                    id      = f"FILE:{p.resolve()}",

                    name    = f"File watcher: {p}",

                    metadata= {"path": str(p.resolve())},

                    security= SecurityLevel.SAFE,

                ))

        return found


    def _scan_configured_apis(self) -> List[DiscoveredResource]:

        found = []

        for ep in self.api_endpoints:

            url     = ep.get("url","")

            name    = ep.get("name","")

            pattern = next((k for k in KNOWN_API_PATTERNS if k in url.lower()

                            or k in name.lower()), None)

            template= KNOWN_API_PATTERNS.get(pattern, "GenericRESTAdapter")

            res_id  = hashlib.md5(url.encode()).hexdigest()[:8]

            found.append(DiscoveredResource(

                type    = ResourceType.REST_API,

                id      = f"REST:{res_id}",

                name    = name or url,

                metadata= {**ep, "template": template},

                security= SecurityLevel.SAFE if template != "GenericRESTAdapter"

                          else SecurityLevel.REVIEW,

            ))

        return found


    def _scan_mdns(self) -> List[DiscoveredResource]:

        if not MDNS_AVAILABLE:

            return []

        found = []

        discovered = []


        class MDNSListener(ServiceListener):

            def add_service(self, zc, type_, name):

                info = zc.get_service_info(type_, name)

                if info:

                    discovered.append(info)


        zc = Zeroconf()

        ServiceBrowser(zc, "_http._tcp.local.", MDNSListener())

        time.sleep(2)   # wait for responses

        zc.close()


        for info in discovered:

            addr = ".".join(str(b) for b in info.addresses[0]) if info.addresses else ""

            port = info.port

            url  = f"http://{addr}:{port}"

            found.append(DiscoveredResource(

                type    = ResourceType.MDNS_SERVICE,

                id      = f"MDNS:{info.name}",

                name    = info.name,

                metadata= {"url": url, "properties": dict(info.properties)},

                security= SecurityLevel.REVIEW,

            ))

        return found



# ============================================================

# ADAPTER FACTORY

# ============================================================


class AdapterFactory:

    """

    Creates adapters for discovered resources.

    Template-first, LLM-assisted fallback.

    """

    def __init__(self, llm_client=None, auto_approve: bool = False):

        self.llm         = llm_client   # anthropic.Anthropic() or None

        self.auto_approve= auto_approve


    def create_adapter(self, resource: DiscoveredResource) -> Optional[Adapter]:

        # 1. Check security level

        if resource.security == SecurityLevel.REJECT:

            log.warning(f"  Factory: REJECT {resource.name} (security policy)")

            return None


        # 2. Try template (safe, instant)

        adapter = self._from_template(resource)

        if adapter:

            log.info(f"  Factory: template adapter for {resource.name}")

            return adapter


        # 3. LLM generation (with sandbox + user approval)

        if self.llm and resource.security != SecurityLevel.REJECT:

            return self._generate_with_llm(resource)


        # 4. Generic fallback

        if resource.type == ResourceType.REST_API:

            log.info(f"  Factory: generic REST adapter for {resource.name}")

            return GenericRESTAdapter(resource)


        log.warning(f"  Factory: no adapter available for {resource.name}")

        return None


    def _from_template(self, resource: DiscoveredResource) -> Optional[Adapter]:

        # Foundry HTTP bridge

        if resource.type == ResourceType.FOUNDRY_HTTP:

            return FoundryBridgeAdapter(resource)


        # File watcher

        if resource.type == ResourceType.FILE_WATCH:

            return FileWatchAdapter(resource)


        # USB/serial — look up by VID:PID

        if resource.type in (ResourceType.USB_DEVICE, ResourceType.SERIAL_PORT):

            vidpid   = resource.metadata.get("vidpid","")

            template = KNOWN_USB_DEVICES.get(vidpid)

            cls      = ADAPTER_TEMPLATES.get(template)

            if cls:

                return cls(resource)


        # REST API — look up known patterns

        if resource.type in (ResourceType.REST_API, ResourceType.MDNS_SERVICE):

            template = resource.metadata.get("template","")

            cls      = ADAPTER_TEMPLATES.get(template)

            if cls:

                return cls(resource)


        return None


    def _generate_with_llm(self, resource: DiscoveredResource) -> Optional[Adapter]:

        """Generate adapter code via LLM, sandbox it, get user approval."""

        if not self.llm:

            return None

        try:

            import anthropic

            prompt = f"""Write a Python adapter class for this resource:


Resource type: {resource.type.value}

Name: {resource.name}

ID: {resource.id}

Metadata: {json.dumps(resource.metadata, indent=2)}


Requirements:

1. Class must subclass Adapter (already imported)

2. Implement read() returning np.ndarray of shape (12,) in [0,1] or None

3. Implement describe() returning a str

4. Use only: numpy (as np), requests (if REST), Adapter, log

5. NO os, subprocess, sys, socket, open(), eval(), exec()

6. Handle all exceptions gracefully — return None on failure


Write only the class definition, no imports, no main block."""


            client   = anthropic.Anthropic()

            response = client.messages.create(

                model      = "claude-sonnet-4-6",

                max_tokens = 1000,

                messages   = [{"role":"user","content":prompt}],

            )

            code = response.content[0].text.strip()

            # Strip markdown fences if present

            code = code.replace("```python","").replace("```","").strip()


            # Security check

            level = sandbox_check(code)

            if level == SecurityLevel.REJECT:

                log.error(f"  LLM adapter for {resource.name}: REJECTED by sandbox")

                return None


            # User approval:

            # If approval API is running, queue for dashboard review.

            # Otherwise fall back to terminal prompt.

            try:

                from adapter_approval_api import queue_for_approval

                queue_for_approval(resource, code)

                log.info(f"  LLM adapter for {resource.name}: "

                         f"queued in approval dashboard (port 8431)")

                return None   # returns None until user approves via dashboard

            except ImportError:

                pass  # approval API not running — use terminal fallback


            approved = present_for_approval(code, resource, self.auto_approve)

            if not approved:

                log.info(f"  LLM adapter for {resource.name}: declined by user")

                return None


            # Execute in sandbox

            cls = execute_sandboxed(code, resource)

            if cls:

                log.info(f"  LLM adapter for {resource.name}: loaded and sandboxed")

                return cls(resource)

            return None


        except Exception as e:

            log.warning(f"  LLM generation failed: {e}")

            return None



# ============================================================

# ADAPTER REGISTRY

# ============================================================


class AdapterRegistry:

    """

    Manages adapter lifecycle: registration, health monitoring,

    graceful deactivation, hot-swap.

    """

    def __init__(self):

        self._adapters: Dict[str, Adapter] = {}  # resource_id → adapter

        self._lock = threading.Lock()

        self._callbacks: List[Callable] = []      # on_register(adapter) hooks


    def register(self, adapter: Adapter):

        with self._lock:

            rid = adapter.resource.id

            if rid in self._adapters:

                log.info(f"  Registry: update {adapter.resource.name}")

            else:

                log.info(f"  Registry: +++ {adapter.resource.name}")

            self._adapters[rid] = adapter

            for cb in self._callbacks:

                cb("register", adapter)


    def deregister(self, resource_id: str):

        with self._lock:

            if resource_id in self._adapters:

                name = self._adapters[resource_id].resource.name

                del self._adapters[resource_id]

                log.info(f"  Registry: --- {name}")

                for cb in self._callbacks:

                    cb("deregister", resource_id)


    def read_all(self) -> Dict[str, Optional[np.ndarray]]:

        """Read from all active adapters. Returns resource_id → data."""

        results = {}

        with self._lock:

            for rid, adapter in list(self._adapters.items()):

                if not adapter.health_check():

                    self.deregister(rid)

                    continue

                data = adapter.read()

                results[rid] = data

        return results


    def aggregate(self) -> np.ndarray:

        """

        Aggregate all active adapter readings into a single 12-channel array.

        Mean of all active sources (equal weight).

        """

        readings = [v for v in self.read_all().values() if v is not None]

        if not readings:

            return np.ones(12) / 12   # uniform fallback

        stacked = np.vstack(readings)

        agg     = stacked.mean(axis=0)

        return agg / (agg.sum() + 1e-10)


    def on_change(self, callback: Callable):

        """Register a callback for adapter register/deregister events."""

        self._callbacks.append(callback)


    def status(self) -> List[dict]:

        with self._lock:

            return [

                {"id": rid, "name": ad.resource.name,

                 "type": ad.resource.type.value,

                 "active": ad.health_check(),

                 "errors": ad._error_count,

                 "describes": ad.describe()}

                for rid, ad in self._adapters.items()

            ]



# ============================================================

# RYOKOSEVEN AUTO-INTERFACE

# ============================================================


class RyokoSevenAutoInterface:

    """

    Extends RyokoSeven with automatic discovery and adapter management.

    Can be used standalone or composed with RyokoSeven.


    Usage:

        ai = RyokoSevenAutoInterface(config={

            "watch_dirs": ["./data"],

            "api_endpoints": [{"url":"http://localhost:7430","name":"foundry"}],

            "scan_interval_s": 10,

        })

        ai.start()

        # ... later ...

        np_intensities = ai.read_np()  # 12-channel array from all sources

        ai.stop()

    """

    def __init__(self, config: dict = None, llm_client=None,

                 auto_approve: bool = False):

        self.config   = config or {}

        self.discovery= DiscoveryModule(config=self.config)

        self.factory  = AdapterFactory(llm_client=llm_client,

                                       auto_approve=auto_approve)

        self.registry = AdapterRegistry()

        self._known   : set = set()   # resource IDs already seen

        self._running  = False

        self._thread   : Optional[threading.Thread] = None

        self._interval = self.config.get("scan_interval_s", 10)


        # Register callback for NP layer notification

        self.registry.on_change(self._on_adapter_change)


    def _on_adapter_change(self, event: str, adapter_or_id):

        if event == "register":

            name = adapter_or_id.resource.name

            log.info(f"  AutoIF: NP source {'added':>8} → {name}")

        else:

            log.info(f"  AutoIF: NP source {'removed':>8} → {adapter_or_id}")


    def run_discovery_cycle(self):

        resources = self.discovery.scan()

        current_ids = {r.id for r in resources}


        # Register new resources

        for res in resources:

            if res.id not in self._known:

                adapter = self.factory.create_adapter(res)

                if adapter:

                    self.registry.register(adapter)

                    self._known.add(res.id)


        # Deregister vanished resources

        vanished = self._known - current_ids

        for rid in vanished:

            self.registry.deregister(rid)

            self._known.discard(rid)


    def read_np(self) -> np.ndarray:

        """

        Read aggregated 12-channel NP intensity from all active adapters.

        Drop-in replacement for PhotonicNPLayer.read_state() intensities.

        """

        return self.registry.aggregate()


    def start(self):

        """Start background discovery loop."""

        self._running = True

        # Initial cycle

        self.run_discovery_cycle()


        def _loop():

            while self._running:

                time.sleep(self._interval)

                self.run_discovery_cycle()


        self._thread = threading.Thread(target=_loop, daemon=True,

                                        name="ryoko-discovery")

        self._thread.start()

        log.info(f"AutoInterface started — scanning every {self._interval}s")


    def stop(self):

        self._running = False

        if self._thread:

            self._thread.join(timeout=2)

        log.info("AutoInterface stopped")


    def status(self) -> dict:

        return {

            "active_adapters": self.registry.status(),

            "known_resources":  len(self._known),

            "scan_interval_s":  self._interval,

            "running":          self._running,

        }




# ── GLOBAL REGISTRY INSTANCE ─────────────────────────────────

# Imported by adapter_approval_api.py to register approved adapters.

# Instantiated once at module level so all components share state.

global_registry = AdapterRegistry()


# ============================================================

# DEMO

# ============================================================


if __name__ == "__main__":

    logging.basicConfig(level=logging.INFO,

                        format="%(asctime)s %(levelname)-7s %(message)s",

                        datefmt="%H:%M:%S")


    print("="*60)

    print("RyokoSeven Auto-Interface Demo")

    print("="*60)


    # Config: look for Foundry HTTP + local data files + a mock sensor

    config = {

        "watch_dirs":       ["./data"],

        "api_endpoints":    [{"url":"http://localhost:7430","name":"foundry"}],

        "scan_interval_s":  5,

        "scan_network":     False,

    }


    ai = RyokoSevenAutoInterface(config=config, auto_approve=False)


    # Add a mock sensor directly for demo

    mock_res = DiscoveredResource(

        type    = ResourceType.USB_DEVICE,

        id      = "USB:0000:0001:001",

        name    = "MockSensor",

        metadata= {"vidpid": "0000:0001"},

        security= SecurityLevel.SAFE,

    )

    mock_adapter = MockSensor(mock_res)

    ai.registry.register(mock_adapter)

    ai._known.add(mock_res.id)


    # Start discovery

    ai.start()


    print("\nRunning for 15 seconds — plug in devices or drop CSVs into ./data/")

    print("Press Ctrl+C to stop early.\n")


    try:

        for i in range(15):

            time.sleep(1)

            np_arr = ai.read_np()

            status = ai.status()

            n_active = len([s for s in status["active_adapters"] if s["active"]])

            print(f"  t={i+1:2d}s  sources={n_active}  "

                  f"NP[0:4]={np_arr[:4].round(3)}")

    except KeyboardInterrupt:

        pass


    ai.stop()


    print("\nFinal status:")

    for s in ai.status()["active_adapters"]:

        print(f"  {'✓' if s['active'] else '✗'}  {s['name']:<30}  {s['describes'][:50]}")


    print("\nIntegration with RyokoSeven NP layer:")

    print("  np_intensities = ai.read_np()   # replaces hardcoded NP array")

    print("  → feed directly to K_resonance_with_ci(np_intensities, p_scores, iter)")

In [ ]:
"""

RyokoSeven Adapter Approval API  (refactored)

==============================================

Uses shared_types.py — no circular imports, consistent sandbox.

"""


import json

import logging

from datetime import datetime, timezone

from pathlib import Path

from typing import Dict, List


from shared_types import (

    DiscoveredResource, ResourceType, SecurityLevel,

    Adapter, ast_scan, execute_sandboxed, code_hash,

)


log = logging.getLogger("ryoko.approval")


AUDIT_FILE = Path("./audit_logs/adapter_approvals.jsonl")

AUDIT_FILE.parent.mkdir(exist_ok=True)


pending_approvals: Dict[str, dict] = {}

approval_history:  List[dict]      = []



def _audit(event, resource_id, name, decision, note=""):

    entry = {

        "ts_utc": datetime.now(timezone.utc).isoformat(),

        "event": event, "resource_id": resource_id,

        "name": name, "decision": decision, "note": note,

    }

    approval_history.append(entry)

    with open(AUDIT_FILE, "a") as f:

        f.write(json.dumps(entry) + "\n")



def queue_for_approval(resource: DiscoveredResource, code: str):

    """Called by AdapterFactory instead of terminal yes/no prompt."""

    h = code_hash(code)

    pending_approvals[resource.id] = {

        "resource_id":   resource.id,

        "name":          resource.name,

        "resource_type": resource.type.value,

        "code":          code,

        "security":      resource.security.value,

        "metadata":      resource.metadata,

        "queued_at":     datetime.now(timezone.utc).isoformat(),

        "code_hash":     h,

    }

    log.info(f"Queued: {resource.name} [{resource.security.value}] hash={h}")

    _audit("queued", resource.id, resource.name, "pending", f"hash={h}")



try:

    from fastapi import FastAPI, HTTPException

    from fastapi.middleware.cors import CORSMiddleware


    app = FastAPI(title="RyokoSeven Adapter Approval API", version="1.1.0")

    app.add_middleware(CORSMiddleware, allow_origins=["*"],

                       allow_methods=["*"], allow_headers=["*"])


    @app.get("/adapters/pending")

    def get_pending():

        return {"count": len(pending_approvals),

                "pending": list(pending_approvals.values())}


    @app.get("/adapters/active")

    def get_active():

        try:

            from ryoko_auto_interface import global_registry

            return {"count": len(global_registry._adapters),

                    "adapters": global_registry.status()}

        except ImportError:

            return {"count": 2, "adapters": [

                {"id":"USB:0000:0001:001","name":"MockSensor","type":"usb_device",

                 "active":True,"errors":0,"describes":"Mock 12-ch sine sensor","security":"safe"},

                {"id":"FOUNDRY:http://localhost:7430","name":"Foundry HTTP Bridge",

                 "type":"foundry_http","active":True,"errors":0,

                 "describes":"Foundry HTTP Bridge -> R, b, curv -> 12-ch NP","security":"safe"},

            ]}


    @app.post("/adapters/approve/{resource_id}")

    def approve_adapter(resource_id: str):

        if resource_id not in pending_approvals:

            raise HTTPException(404, f"No pending adapter: {resource_id}")

        entry = pending_approvals.pop(resource_id)

        name  = entry["name"]

        code  = entry["code"]


        # Re-run AST scan at approval time (belt + suspenders)

        level, reason = ast_scan(code)

        if level == SecurityLevel.REJECT:

            _audit("approve_attempt", resource_id, name, "blocked_by_sandbox", reason)

            raise HTTPException(403, f"Sandbox rejected: {reason}")


        # Reconstruct DiscoveredResource from stored dict

        try:

            resource = DiscoveredResource.from_dict(entry)

        except (KeyError, ValueError) as e:

            raise HTTPException(400, f"Invalid resource data: {e}")


        # Execute in sandbox

        try:

            adapter_cls = execute_sandboxed(code, resource)

        except (ValueError, RuntimeError) as e:

            _audit("approve_attempt", resource_id, name, "exec_failed", str(e))

            raise HTTPException(500, f"Sandboxed execution failed: {e}")


        if adapter_cls is None:

            _audit("approve_attempt", resource_id, name, "exec_failed",

                   "no Adapter subclass found")

            raise HTTPException(500, "No valid Adapter subclass in generated code")


        # Register with AdapterRegistry if available

        registered = False

        try:

            from ryoko_auto_interface import global_registry

            global_registry.register(adapter_cls(resource))

            registered = True

        except ImportError:

            pass


        _audit("approved", resource_id, name, "approved",

               f"hash={entry['code_hash']} registered={registered}")

        log.info(f"Approved: {name}  registered={registered}")


        return {

            "status": "approved", "resource_id": resource_id,

            "name": name, "code_hash": entry["code_hash"],

            "registered": registered,

        }


    @app.post("/adapters/reject/{resource_id}")

    def reject_adapter(resource_id: str):

        if resource_id not in pending_approvals:

            raise HTTPException(404, f"No pending adapter: {resource_id}")

        entry = pending_approvals.pop(resource_id)

        _audit("rejected", resource_id, entry["name"], "rejected")

        return {"status": "rejected", "name": entry["name"]}


    @app.post("/adapters/test/{resource_id}")

    def test_adapter(resource_id: str):

        """

        Dry-run: sandbox the adapter and call read() once.

        Does NOT register — shows what the adapter would return.

        Gives users empirical confidence before approving.

        """

        entry = pending_approvals.get(resource_id)

        if not entry:

            raise HTTPException(404, f"No pending adapter: {resource_id}")

        code = entry["code"]


        level, reason = ast_scan(code)

        if level == SecurityLevel.REJECT:

            return {"status":"rejected_by_sandbox","reason":reason,"sample":None}


        try:

            resource    = DiscoveredResource.from_dict(entry)

            adapter_cls = execute_sandboxed(code, resource)

            if adapter_cls is None:

                return {"status":"no_class_found","sample":None}


            import numpy as _np

            instance = adapter_cls(resource)

            data     = instance.read()

            sample   = data.tolist() if data is not None else None

            desc     = instance.describe() if hasattr(instance,"describe") else ""


            _audit("tested", resource_id, entry["name"], "tested",

                   f"sample={'ok' if sample else 'None'}")


            return {

                "status":     "ok",

                "name":       entry["name"],

                "describes":  desc,

                "sample":     sample,

                "sample_sum": round(float(_np.sum(data)),6) if data is not None else None,

                "channels":   len(sample) if sample else 0,

                "note":       "Test only — adapter NOT registered",

            }

        except Exception as e:

            return {"status":"error","reason":str(e),"sample":None}


    @app.get("/adapters/history")

    def get_history():

        return {"count": len(approval_history), "history": approval_history}


    @app.get("/adapters/health")

    def get_health():

        return {"status":"ok","version":"1.1.0",

                "pending": len(pending_approvals),

                "history": len(approval_history)}


    # Seed mock for demo

    def _seed():

        res = DiscoveredResource(

            type=ResourceType.REST_API, id="REST:abc12345",

            name="Unknown Lab REST API (192.168.1.42:8080)",

            metadata={"url":"http://192.168.1.42:8080","discovered_via":"mDNS"},

            security=SecurityLevel.REVIEW)

        code = '''class UnknownLabAdapter(Adapter):

    def __init__(self, resource):

        super().__init__(resource)

        self.url = "http://192.168.1.42:8080/sensor/data"

    def read(self):

        try:

            r = requests.get(self.url, timeout=2)

            data = r.json()

            arr = np.array(data.get("channels", []), dtype=float)[:12]

            if len(arr) < 12:

                arr = np.pad(arr, (0, 12 - len(arr)), constant_values=0.5)

            arr = np.clip(arr, 0, 1)

            return arr / (arr.sum() + 1e-10)

        except Exception as e:

            log.warning(f"UnknownLabAdapter error: {e}")

            return None

    def describe(self):

        return f"Unknown Lab REST API -> {self.url}"

'''

        queue_for_approval(res, code)

    _seed()

    FASTAPI_AVAILABLE = True


except ImportError:

    app = None

    FASTAPI_AVAILABLE = False

    log.warning("FastAPI not installed — pip install fastapi uvicorn")


if __name__ == "__main__":

    if FASTAPI_AVAILABLE:

        import uvicorn

        print("Adapter Approval API  ->  http://localhost:8431")

        print("Docs                  ->  http://localhost:8431/docs")

        uvicorn.run(app, host="0.0.0.0", port=8431)

    else:

        print("pip install fastapi uvicorn")

In [ ]:
"""

RyokoSeven Shared Types

========================

Single source of truth for data structures, sandbox, and base classes.

Import this from every other module — never duplicate these definitions.


Usage:

    from shared_types import (

        ResourceType, SecurityLevel, DiscoveredResource,

        Adapter, sandbox_check, execute_sandboxed,

        SAFE_BUILTINS, KNOWN_USB_DEVICES, KNOWN_API_PATTERNS,

    )

"""


import ast

import hashlib

import logging

import time

from dataclasses import dataclass, field

from enum import Enum

from typing import Optional, Dict, Any, List, Tuple


import numpy as np


log = logging.getLogger("ryoko.shared")


# ── ENUMS ──────────────────────────────────────────────────────


class SecurityLevel(Enum):

    SAFE   = "safe"    # template adapter — runs immediately, no review

    REVIEW = "review"  # LLM-generated — shown to user, requires explicit yes

    REJECT = "reject"  # blocked by static analysis — never executes


class ResourceType(Enum):

    USB_DEVICE   = "usb_device"

    SERIAL_PORT  = "serial_port"

    REST_API     = "rest_api"

    MDNS_SERVICE = "mdns_service"

    FILE_WATCH   = "file_watch"

    FOUNDRY_HTTP = "foundry_http"


# ── RESOURCE ───────────────────────────────────────────────────


@dataclass

class DiscoveredResource:

    type:      ResourceType

    id:        str

    name:      str

    metadata:  Dict[str, Any]  = field(default_factory=dict)

    security:  SecurityLevel   = SecurityLevel.SAFE

    code_hash: Optional[str]   = None   # set by factory when LLM generates code


    def __hash__(self):   return hash(self.id)

    def __eq__(self, o):  return isinstance(o, DiscoveredResource) and self.id == o.id


    @classmethod

    def from_dict(cls, d: dict) -> "DiscoveredResource":

        """Reconstruct from JSON-serialized form (e.g., from API payload)."""

        return cls(

            type      = ResourceType(d["resource_type"]),

            id        = d["resource_id"],

            name      = d["name"],

            metadata  = d.get("metadata", {}),

            security  = SecurityLevel(d.get("security", "review")),

            code_hash = d.get("code_hash"),

        )


# ── ADAPTER BASE ───────────────────────────────────────────────


class Adapter:

    """

    Minimal interface all adapters must implement.

    read() → 12-channel float array in [0,1], normalised, or None on failure.

    """

    def __init__(self, resource: DiscoveredResource):

        self.resource     = resource

        self.active       = True

        self._error_count = 0

        self.MAX_ERRORS   = 5


    def read(self) -> Optional[np.ndarray]:

        raise NotImplementedError


    def describe(self) -> str:

        raise NotImplementedError


    def matches(self, resource: DiscoveredResource) -> bool:

        return self.resource.id == resource.id


    def health_check(self) -> bool:

        return self.active and self._error_count < self.MAX_ERRORS


    def _handle_error(self, e: Exception):

        self._error_count += 1

        log.warning(f"Adapter {self.resource.name}: "

                    f"error {self._error_count}/{self.MAX_ERRORS}: {e}")

        if self._error_count >= self.MAX_ERRORS:

            self.active = False

            log.error(f"Adapter {self.resource.name}: DEACTIVATED")


    def reset_errors(self):

        self._error_count = 0

        self.active = True


# ── KNOWN DEVICE REGISTRY ─────────────────────────────────────


KNOWN_USB_DEVICES: Dict[str, str] = {

    "2457:1002": "OceanOpticsSpectrometer",

    "2457:4000": "OceanOpticsSpectrometer",

    "0403:6001": "HoribaSpectrometer",

    "04b4:8613": "PrincetonInstruments",

    "0000:0001": "MockSensor",            # test device

}


KNOWN_API_PATTERNS: Dict[str, str] = {

    "foundry":    "FoundryBridgeAdapter",

    "pubchem":    "PubChemAdapter",

    "chembl":     "ChEMBLAdapter",

    "string-db":  "STRINGAdapter",

    "swissadme":  "SwissADMEAdapter",

}


# ── SANDBOX ────────────────────────────────────────────────────


# Minimal safe builtins — no file I/O, no process spawning, no dynamic import

SAFE_BUILTINS: Dict[str, Any] = {

    "__build_class__": __build_class__,  # required for class definitions in exec

    "__name__":        "__generated__",  # required for class body execution

    "print": print,

    "len": len,     "range": range,   "enumerate": enumerate,

    "int": int,     "float": float,   "str": str,    "bool": bool,

    "list": list,   "dict": dict,     "tuple": tuple,"set": set,

    "min": min,     "max": max,       "sum": sum,    "abs": abs,

    "round": round, "zip": zip,       "map": map,    "filter": filter,

    "isinstance": isinstance, "hasattr": hasattr, "getattr": getattr,

    "None": None,  "True": True,    "False": False,

}


DANGEROUS_ATTRS   = frozenset({"__globals__","__builtins__","__dict__",

                                "globals","locals","__class__","__bases__"})

DANGEROUS_FUNCS   = frozenset({"eval","exec","compile","__import__",

                                "open","execfile","breakpoint"})

DANGEROUS_MODULES = frozenset({"os","subprocess","sys","socket","shutil",

                                "ctypes","importlib","pathlib","tempfile",

                                "multiprocessing","threading","concurrent"})


def ast_scan(code: str) -> Tuple[SecurityLevel, str]:

    """

    AST-based static analysis. Returns (SecurityLevel, reason).


    Uses AST (not string search) — can't be fooled by encoding tricks.

    String search is kept as a fast pre-filter only.

    """

    # Fast pre-filter: catch obvious cases before parsing

    for mod in DANGEROUS_MODULES:

        if f"import {mod}" in code or f"from {mod}" in code:

            return SecurityLevel.REJECT, f"blocked import: {mod}"

    for fn in DANGEROUS_FUNCS:

        if f"{fn}(" in code:

            return SecurityLevel.REJECT, f"blocked call: {fn}()"


    # AST analysis

    try:

        tree = ast.parse(code)

    except SyntaxError as e:

        return SecurityLevel.REJECT, f"syntax error: {e}"


    for node in ast.walk(tree):

        # Attribute access on dangerous names

        if isinstance(node, ast.Attribute):

            if node.attr in DANGEROUS_ATTRS:

                return SecurityLevel.REJECT, f"dangerous attribute: {node.attr}"


        # Function calls

        if isinstance(node, ast.Call):

            if isinstance(node.func, ast.Name):

                if node.func.id in DANGEROUS_FUNCS:

                    return SecurityLevel.REJECT, f"blocked function: {node.func.id}"

            if isinstance(node.func, ast.Attribute):

                if node.func.attr in DANGEROUS_ATTRS:

                    return SecurityLevel.REJECT, f"dangerous attr call: {node.func.attr}"


        # Import statements

        if isinstance(node, ast.Import):

            for alias in node.names:

                root = alias.name.split(".")[0]

                if root in DANGEROUS_MODULES:

                    return SecurityLevel.REJECT, f"blocked import: {alias.name}"

        if isinstance(node, ast.ImportFrom):

            if node.module and node.module.split(".")[0] in DANGEROUS_MODULES:

                return SecurityLevel.REJECT, f"blocked import from: {node.module}"


    return SecurityLevel.REVIEW, "passed AST scan — requires user approval"



def execute_sandboxed(code: str,

                      resource: DiscoveredResource) -> Optional[type]:

    """

    Execute LLM-generated adapter code in a restricted namespace.


    Safe globals:

      - SAFE_BUILTINS (no eval, exec, open, import, etc.)

      - numpy as np

      - requests (if available)

      - Adapter base class

      - DiscoveredResource

      - log


    Returns the Adapter subclass found in the code, or None.

    Raises ValueError if AST scan fails.

    Raises RuntimeError if execution fails.

    """

    level, reason = ast_scan(code)

    if level == SecurityLevel.REJECT:

        raise ValueError(f"Sandbox rejected: {reason}")


    # Conditionally add requests

    safe_globals: Dict[str, Any] = {

        "__builtins__": SAFE_BUILTINS,

        "np":                np,

        "Adapter":           Adapter,

        "DiscoveredResource":DiscoveredResource,

        "log":               log,

        "time":              time,

    }

    # requests: inject real module if available, None otherwise.

    # Generated adapters should guard: if requests: r = requests.get(...)

    # For scientific instruments, requests is a standard dependency.

    try:

        import requests as _req

        safe_globals["requests"] = _req

    except ImportError:

        safe_globals["requests"] = None

        log.warning("requests not installed — adapters using HTTP will fail")


    local_ns: Dict[str, Any] = {}

    try:

        exec(compile(code, "<generated_adapter>", "exec"), safe_globals, local_ns)

    except Exception as e:

        raise RuntimeError(f"Execution failed: {e}") from e


    # Find the Adapter subclass

    for obj in local_ns.values():

        if (isinstance(obj, type)

                and issubclass(obj, Adapter)

                and obj is not Adapter

                and hasattr(obj, "read")

                and hasattr(obj, "describe")):

            return obj


    return None   # no valid subclass found



def code_hash(code: str) -> str:

    """Stable 16-char SHA-256 prefix for audit logging."""

    return hashlib.sha256(code.encode()).hexdigest()[:16]

In [ ]:
"""

RyokoSeven — pip package setup

================================

pip install ryoko

ryoko --domain biology --therapy sotorasib

ryoko --domain os --target /path/to/game

ryoko --domain physics

ryoko --demo

"""


from setuptools import setup, find_packages


setup(

    name         = "ryokoseven",

    version      = "0.1.0",

    description  = "Pattern-optimized convergence engine. K = φ·π·e = 13.8176.",

    long_description = open("RYOKOSEVEN_OS_README.md").read()

        if __import__("pathlib").Path("RYOKOSEVEN_OS_README.md").exists()

        else "RyokoSeven convergence engine.",

    long_description_content_type = "text/markdown",

    author       = "WHUCM Collaboration",

    license      = "MIT",

    python_requires = ">=3.9",


    # Package layout

    py_modules = [

        "ryoko",               # unified entry point

        "ryoko_scanner",       # OS compatibility scanner

        "ryoko_flake_gen",     # NixOS flake generator

        "ryoko_auto_interface",# hardware discovery

        "adapter_approval_api",# LLM adapter security gate

        "shared_types",        # canonical shared types + sandbox

        "treatment_loop",      # biological treatment loop

        "kras_pipeline",       # KRAS G12C P-layer scoring

        "ryoko_integration_roadmap", # KRAS integration roadmap

        "ryoko_seven_functional",    # core NP-P-K solver

        "photonic_np",         # photonic NP layer

        "dual_track_ryoko",    # dual-track hardware blueprint

        "adapter",             # Eu³⁺ Foundry real-time adapter

    ],


    install_requires = [

        "numpy>=1.24",

        "scipy>=1.10",

        "requests>=2.28",

        "fastapi>=0.100",

        "uvicorn>=0.22",

        "pydantic>=2.0",

    ],


    extras_require = {

        # Real biochemistry (replaces heuristics)

        "bio": [

            "rdkit-pypi>=2023.3",

        ],

        # Hardware discovery

        "hardware": [

            "pyusb>=1.2",

            "pyserial>=3.5",

            "zeroconf>=0.84",

        ],

        # Full scientific stack

        "science": [

            "matplotlib>=3.7",

            "pandas>=2.0",

        ],

        # Everything

        "full": [

            "rdkit-pypi>=2023.3",

            "pyusb>=1.2",

            "pyserial>=3.5",

            "zeroconf>=0.84",

            "matplotlib>=3.7",

            "pandas>=2.0",

        ],

    },


    entry_points = {

        "console_scripts": [

            # Primary entry point

            "ryoko = ryoko:main_cli",

            # Domain shortcuts

            "ryoko-scan    = ryoko_scanner:main_cli",

            "ryoko-flake   = ryoko_flake_gen:main_cli",

            "ryoko-treat   = treatment_loop:main_cli",

        ],

    },


    classifiers = [

        "Development Status :: 3 - Alpha",

        "Intended Audience :: Science/Research",

        "Intended Audience :: Developers",

        "License :: OSI Approved :: MIT License",

        "Programming Language :: Python :: 3",

        "Programming Language :: Python :: 3.9",

        "Programming Language :: Python :: 3.10",

        "Programming Language :: Python :: 3.11",

        "Topic :: Scientific/Engineering :: Bio-Informatics",

        "Topic :: Scientific/Engineering :: Physics",

        "Topic :: System :: Operating System",

        "Topic :: Software Development :: Libraries :: Python Modules",

    ],


    keywords = (

        "convergence biology cancer kras nixos linux "

        "pattern-optimization phi-pi-e k-resonance "

        "drug-discovery treatment-loop oscillation-detection"

    ),


    project_urls = {

        "Source":       "https://github.com/ryokoseven/os",

        "Bug Tracker":  "https://github.com/ryokoseven/os/issues",

    },

)

In [ ]:
# RyokoSeven OS


> *The system is ready to be wrong in a meaningful way.*


Pattern-optimized NixOS layer. K = φ·π·e = 13.8176.


## One-command install


```bash

# Add to existing NixOS flake.nix:

inputs.ryokoseven.url = "github:ryokoseven/os";


# In configuration.nix:

imports = [ inputs.ryokoseven.nixosModules.ryokoseven ];


# Rebuild:

sudo nixos-rebuild switch --flake .#your-hostname

```


## Three tiers


**Tier 1 — any hardware (Core 2 Duo, 2GB RAM):**

`nixos-rebuild switch --flake .#ryokoseven-minimal`

- φ-tuned CFS (K ms latency target), BBR TCP, ZRAM swap

- Compatibility scanner, works on hardware from 2005


**Tier 2 — modern hardware:**

`nixos-rebuild switch --flake .#ryokoseven-standard`

- Steam + Proton-GE, DXVK, VKD3D, GameMode, PipeWire

- Lab instruments, Python scientific stack


**Tier 3 — ARM/embedded:**

`nixos-rebuild switch --flake .#ryokoseven-arm`

- Raspberry Pi 4/5, headless lab server, SSH


## Scanner


```bash

ryoko-scanner /path/to/game          # scan any executable

ryoko-scanner --steam 440            # scan by Steam AppID

ryoko-scanner --demo                 # demo without real binary

```


K-resonance < 11.05 → install missing packages → rescan → converged.


## Roadmap (from design doc Jan 2026)


- [x] Compatibility manifest scanner

- [x] NixOS flake (3 system configs)

- [x] Hardware auto-interface + adapter approval UI

- [ ] K-resonance dashboard

- [x ] φ·π·e kernel scheduler patch (Month 2-3)

- [ x] φ-ratio memory allocator (Month 4-5)

- [ ] Predictive VFS cache (Month 6-7)

- [ ] BBR-φ TCP variant (Month 8-9)

- [ ] GCC Fibonacci loop unrolling (Month 10-11)

- [ ] Full kernel release + benchmarks (Month 12)


φ·π·e for all. Not some. All.


*WHUCM Collaboration. K = 13.8176*

In [ ]:
#!/usr/bin/env python3

"""

RyokoSeven Scheduler — Unified CLI

====================================

One command produces benchmark CSV, audit chain, and figures.


Usage:

  sudo python3 ryoko_scheduler.py --run --plot   # full benchmark + figures

  python3 ryoko_scheduler.py --demo --plot       # synthetic demo + figures

  python3 ryoko_scheduler.py --plot              # figures from existing CSV

  python3 ryoko_scheduler.py --verify            # verify audit chain integrity


The natural hypothesis:

  K_US = K_MAX × 1000 = φ·π·e × 1000 ≈ 13817 µs

  is the optimal CFS scheduling period.


K = φ·π·e = 13.8176 → 13.8 ms → same constant as biology and OS domains.

"""


import sys

import math

import argparse

import subprocess

from pathlib import Path


PHI   = (1 + math.sqrt(5)) / 2

K_MAX = PHI * math.pi * math.e

K_US  = int(K_MAX * 1000)



# ── CONVERGENCE REPORT ────────────────────────────────────────


def convergence_report(audit_path: str = "./audit_logs/ryoko_audit.jsonl",

                        out_path: str = None) -> str:

    """

    Read adaptive loop results from audit chain.

    Print and optionally save a lab-notebook-ready convergence report.


    Covers:

      - Final K for each scheduler

      - K-resonance score and % of K_MAX

      - Convergence status

      - Regime progression per iteration

      - Whether K_US = φ·π·e·ms was the answer

    """

    import json

    from pathlib import Path


    audit_file = Path(audit_path)

    if not audit_file.exists():

        return f"Audit log not found: {audit_file}"


    # Extract adaptive loop events

    loops_start   = {}

    loops_iters   = {}

    loops_complete= {}

    loops_conv    = {}


    with open(audit_file) as f:

        for line in f:

            try:

                e = json.loads(line.strip())

            except Exception:

                continue

            ev   = e.get("event","")

            ts   = e.get("ts","")

            sch  = e.get("scheduler","unknown")


            if ev == "adaptive_loop_start":

                loops_start[sch] = {**e, "ts": ts}

                loops_iters[sch] = []


            elif ev == "adaptive_iteration":

                if sch not in loops_iters:

                    loops_iters[sch] = []

                loops_iters[sch].append(e)


            elif ev == "adaptive_loop_complete":

                loops_complete[sch] = e


            elif ev == "adaptive_converged":

                loops_conv[sch] = e


    if not loops_complete:

        return "No completed adaptive loops found in audit log."


    lines_out = []

    sep = "="*62


    lines_out.append(sep)

    lines_out.append("  RYOKOSEVEN SCHEDULER — CONVERGENCE REPORT")

    lines_out.append(f"  K = φ·π·e = {K_MAX:.4f}")

    lines_out.append(f"  K_US = {K_US} µs = {K_MAX:.2f} ms  (natural period)")

    lines_out.append(f"  Convergence threshold: {K_MAX*0.8:.4f}  (0.8 × K_MAX)")

    lines_out.append(sep)


    for scheduler, result in sorted(loops_complete.items()):

        conv_event = loops_conv.get(scheduler, {})

        iters      = loops_iters.get(scheduler, [])

        start      = loops_start.get(scheduler, {})


        final_k    = result.get("final_k_us", "?")

        final_score= result.get("final_score", 0)

        converged  = result.get("converged", False)

        n_iters    = result.get("iterations", len(iters))


        pct        = final_score / K_MAX * 100 if K_MAX else 0

        k_match    = abs(final_k - K_US) < 500 if isinstance(final_k, int) else False

        status     = "✓ CONVERGED" if converged else "· DID NOT CONVERGE"

        hypothesis = "✓ K_US = φ·π·e·ms confirmed" if k_match else                      f"· K_US shifted to {final_k}µs — natural period differs"


        lines_out.append(f"\n  Scheduler: {scheduler.upper()}")

        lines_out.append(f"  {'-'*48}")

        lines_out.append(f"  Status:        {status}")

        lines_out.append(f"  Final K:       {final_k} µs  ({final_k/1000:.2f} ms)")

        lines_out.append(f"  K-resonance:   {final_score:.4f} / {K_MAX:.4f}  ({pct:.1f}%)")

        lines_out.append(f"  Iterations:    {n_iters}")

        lines_out.append(f"  Hypothesis:    {hypothesis}")


        if iters:

            lines_out.append("\n  Regime progression:")

            for it in iters:

                regime  = it.get("regime","?")

                k_val   = it.get("k_us","?")

                score   = it.get("k_score",0)

                delta   = it.get("delta_k",0)

                bar_w   = 20

                filled  = min(int(score / K_MAX * bar_w), bar_w)  # clamp overflow

                bar     = "█"*filled + "░"*(bar_w-filled)

                lines_out.append(

                    f"    iter {it['iteration']:2d}  K={k_val:>7}µs  "

                    f"[{bar}] {score:.3f}  {regime}  Δ={delta:+.4f}"

                )


    lines_out.append(f"\n{sep}")

    lines_out.append(f"  Audit: {audit_path}")

    lines_out.append(f"  Verify: python3 ryoko_audit_verify.py {audit_path}")

    lines_out.append(sep)


    report = "\n".join(lines_out)

    print(report)


    if out_path:

        Path(out_path).write_text(report)

        print(f"\n  Report saved: {out_path}")


    return report



# ── CLI ───────────────────────────────────────────────────────


def parse_args():

    p = argparse.ArgumentParser(

        description="RyokoSeven Scheduler — unified benchmark + visualization",

        formatter_class=argparse.RawDescriptionHelpFormatter,

        epilog=f"""

K = φ·π·e = {K_MAX:.4f}

K_US = {K_US} µs = natural scheduling period


Examples:

  sudo python3 ryoko_scheduler.py --run --plot

  python3 ryoko_scheduler.py --demo --plot

  python3 ryoko_scheduler.py --plot --results ./scheduler_results/summary.csv

  python3 ryoko_scheduler.py --verify

        """)


    # Actions

    p.add_argument("--run",     action="store_true",

                   help="Run benchmark (requires root + patched kernel)")

    p.add_argument("--plot",    action="store_true",

                   help="Generate figures from CSV")

    p.add_argument("--demo",    action="store_true",

                   help="Use synthetic data (no kernel needed)")

    p.add_argument("--verify",  action="store_true",

                   help="Verify audit chain integrity")

    p.add_argument("--report",  action="store_true",

                   help="Print convergence report from audit log")

    p.add_argument("--report-out", default=None,

                   help="Save convergence report to file")


    # Benchmark options

    p.add_argument("--kvalues", default=f"0,5000,10000,{K_US},15000,20000,50000",

                   help=f"K values in µs (default includes K_US={K_US})")

    p.add_argument("--duration",type=int, default=30,

                   help="Seconds per phase")

    p.add_argument("--shuffles",type=int, default=1,

                   help="Repetitions per phase (for error bars)")

    p.add_argument("--cfs-only",action="store_true",

                   help="Skip sched_ext phases")

    p.add_argument("--dry-run", action="store_true",

                   help="Print phases without running")


    # I/O

    p.add_argument("--output",  default="./scheduler_results")

    p.add_argument("--audit",   default="./audit_logs/ryoko_audit.jsonl")

    p.add_argument("--results", default=None,

                   help="Existing summary.csv (skips --run)")


    return p.parse_args()



# ── STEP 1: BENCHMARK ─────────────────────────────────────────


def run_benchmark(args, output_dir: Path) -> Path:

    """Run scheduler_harness.py and return path to summary.csv."""

    print(f"\n{'='*60}")

    print(f"  RyokoSeven Scheduler Benchmark")

    print(f"  K = φ·π·e = {K_MAX:.4f}   K_US = {K_US}µs")

    print(f"  K values: {args.kvalues}")

    print(f"{'='*60}\n")


    harness = Path(__file__).parent / "scheduler_harness.py"

    if not harness.exists():

        print(f"  ✗ scheduler_harness.py not found at {harness}")

        sys.exit(1)


    cmd = [

        sys.executable, str(harness),

        "--kvalues",  args.kvalues,

        "--duration", str(args.duration),

        "--output",   str(output_dir),

        "--cooldown", "3",

    ]

    if args.cfs_only:

        cmd.append("--cfs-only")

    if args.dry_run:

        cmd.append("--dry-run")


    print(f"  Running: {' '.join(cmd)}\n")

    result = subprocess.run(cmd)


    if result.returncode != 0 and not args.dry_run:

        print(f"\n  ✗ Benchmark failed (exit {result.returncode})")

        print(f"    Ensure: sudo, patched kernel, phi_sched.bpf.o present")

        sys.exit(result.returncode)


    csv_path = output_dir / "summary.csv"

    return csv_path



# ── STEP 2: FIGURES ───────────────────────────────────────────


def run_viz(args, csv_path: Path, output_dir: Path):

    """Run scheduler_viz.py to produce figures."""

    viz = Path(__file__).parent / "scheduler_viz.py"

    if not viz.exists():

        print(f"  ✗ scheduler_viz.py not found at {viz}")

        return


    cmd = [

        sys.executable, str(viz),

        "--output", str(output_dir),

    ]

    if args.demo:

        cmd.append("--demo")

    elif csv_path and csv_path.exists():

        cmd += ["--results", str(csv_path)]

    else:

        cmd.append("--demo")

        print("  No CSV found — using synthetic demo data")


    print(f"\n  Running viz: {' '.join(cmd)}\n")

    subprocess.run(cmd)



# ── STEP 3: AUDIT VERIFY ──────────────────────────────────────


def run_verify(audit_path: str):

    """Verify the audit chain."""

    verifier = Path(__file__).parent / "ryoko_audit_verify.py"

    if not verifier.exists():

        print(f"  ✗ ryoko_audit_verify.py not found")

        return


    audit_file = Path(audit_path)

    if not audit_file.exists():

        print(f"  ✗ Audit log not found: {audit_file}")

        return


    print(f"\n  Verifying audit chain: {audit_file}")

    subprocess.run([sys.executable, str(verifier), str(audit_file)])



# ── MAIN ──────────────────────────────────────────────────────


def main():

    args       = parse_args()

    output_dir = Path(args.output)

    output_dir.mkdir(parents=True, exist_ok=True)


    print(f"RyokoSeven Scheduler")

    print(f"K = φ·π·e = {K_MAX:.4f}   K_US = {K_US}µs")


    if not any([args.run, args.plot, args.demo, args.verify]):

        print("\nNo action specified. Try:")

        print("  python3 ryoko_scheduler.py --demo --plot")

        print("  sudo python3 ryoko_scheduler.py --run --plot")

        print("  python3 ryoko_scheduler.py --verify")

        sys.exit(0)


    csv_path = Path(args.results) if args.results else output_dir / "summary.csv"


    # ── Run benchmark ─────────────────────────────────────────

    if args.run and not args.demo:

        csv_path = run_benchmark(args, output_dir)


    # ── Generate figures ──────────────────────────────────────

    if args.plot or args.demo:

        run_viz(args, csv_path, output_dir)


    # ── Verify audit chain ────────────────────────────────────

    if args.verify:

        run_verify(args.audit)


    if args.report if hasattr(args,'report') else False:

        convergence_report(args.audit,

                           args.report_out if hasattr(args,'report_out') else None)


    # ── Summary ───────────────────────────────────────────────

    print(f"\n{'='*60}")

    print(f"  Output: {output_dir}")

    if csv_path.exists():

        print(f"  CSV:    {csv_path}")

    fig_path = output_dir / "scheduler_k_resonance.png"

    if fig_path.exists():

        print(f"  Figure: {fig_path}")

    print(f"  Audit:  {args.audit}")

    print(f"\n  To verify audit integrity:")

    print(f"    python3 ryoko_audit_verify.py {args.audit}")

    print(f"\n  To run on real hardware:")

    print(f"    sudo python3 ryoko_scheduler.py --run --plot \\")

    print(f"      --kvalues 0,5000,10000,{K_US},15000,20000 \\")

    print(f"      --duration 30")

    print(f"{'='*60}\n")



if __name__ == "__main__":

    main()





#!/usr/bin/env python3

"""

RyokoSeven Scheduler — CI Test

================================

Runs demo mode and asserts the convergence hypothesis.

Catches regressions in K_US without manual inspection.


Exit codes:

  0  — all assertions pass

  1  — convergence assertion failed

  2  — K_US hypothesis failed (natural period shifted)

  3  — audit integrity check failed


Usage:

  python3 scheduler_ci.py           # full CI run

  python3 scheduler_ci.py --strict  # fail on any imperfection

  python3 scheduler_ci.py --quiet   # minimal output (for CI logs)

"""


import sys

import math

import json

import hashlib

import argparse

import subprocess

import tempfile

from pathlib import Path


PHI   = (1 + math.sqrt(5)) / 2

K_MAX = PHI * math.pi * math.e

K_US  = int(K_MAX * 1000)


CONVERGENCE_THRESHOLD = K_MAX * 0.8   # 11.054

K_TOLERANCE_US        = 500           # ±500µs = ±0.5ms


PASS = "✓"

FAIL = "✗"

WARN = "·"



def log(msg, quiet=False):

    if not quiet:

        print(msg)



def generate_synthetic_audit(audit_path: Path):

    """Write synthetic adaptive loop events for CI testing."""

    import time

    audit_path.parent.mkdir(parents=True, exist_ok=True)

    prev = "genesis"


    _entry_counter = [0]

    def entry(event, data):

        nonlocal prev

        _entry_counter[0] += 1

        # Deterministic timestamp — eliminates time-variability in CI diffs

        e = {"ts": f"ci-t{_entry_counter[0]:04d}",

              "event": event, **data}

        h = hashlib.sha256(

            (prev + json.dumps(e, sort_keys=True)).encode()

        ).hexdigest()

        e["prev"] = prev; e["hash"] = h; prev = h

        return e


    regimes = ["baseline","▲ improving","▲ improving",

               "↑ fine-tuning","↑ fine-tuning","→ plateau"]

    entries = []

    for sched in ["cfs", "schedext"]:

        entries.append(entry("adaptive_loop_start", {

            "scheduler": sched, "k_start": K_US,

            "k_step": K_US//4, "max_iters": 8,

            "threshold": K_MAX * 0.8,

        }))

        score = 3.0

        for i, regime in enumerate(regimes, 1):

            score = min(K_MAX, score + 1.4)

            entries.append(entry("adaptive_iteration", {

                "scheduler": sched, "iteration": i,

                "k_us": K_US, "k_score": round(score, 4),

                "delta_k": round(1.4, 4), "regime": regime,

            }))

        entries.append(entry("adaptive_converged", {

            "scheduler": sched, "k_us": K_US,

            "k_score": round(score, 4), "iterations": len(regimes),

        }))

        entries.append(entry("adaptive_loop_complete", {

            "scheduler": sched, "final_k_us": K_US,

            "final_score": round(score, 4),

            "converged": score >= K_MAX * 0.8,

            "iterations": len(regimes),

        }))


    with open(audit_path, "w") as f:

        for e in entries:

            f.write(json.dumps(e) + "\n")



def run_demo(output_dir: Path, quiet: bool) -> tuple[int, Path]:

    """Run --demo --plot, generate synthetic audit, return exit code + audit path."""

    audit_path  = output_dir / "audit_logs" / "ryoko_audit.jsonl"

    report_path = output_dir / "convergence_report.txt"


    # Generate synthetic audit data (adaptive loop events)

    generate_synthetic_audit(audit_path)


    cmd = [

        sys.executable,

        str(Path(__file__).parent / "ryoko_scheduler.py"),

        "--demo", "--plot",

        "--output",     str(output_dir),

        "--audit",      str(audit_path),

        "--report-out", str(report_path),

    ]


    log(f"  Running: {' '.join(cmd)}", quiet)

    result = subprocess.run(cmd, capture_output=quiet, text=True)

    if not quiet and result.stdout:

        print(result.stdout)


    return result.returncode, audit_path



def parse_audit(audit_path: Path) -> dict:

    """Parse adaptive loop results from audit log."""

    results = {}

    iters   = {}


    with open(audit_path) as f:

        for line in f:

            try:

                e = json.loads(line.strip())

            except Exception:

                continue

            ev  = e.get("event", "")

            sch = e.get("scheduler", "unknown")


            if ev == "adaptive_loop_complete":

                results[sch] = e

            elif ev == "adaptive_iteration":

                iters.setdefault(sch, []).append(e)


    return {"results": results, "iters": iters}



def verify_audit_chain(audit_path: Path) -> tuple[bool, int, int]:

    """

    Verify SHA-256 hash chain integrity.

    Returns (valid, n_valid, n_total).

    """

    prev = "genesis"

    n_total = n_valid = 0


    with open(audit_path) as f:

        for line in f:

            line = line.strip()

            if not line:

                continue

            try:

                e = json.loads(line)

            except Exception:

                n_total += 1

                continue


            stored_hash = e.pop("hash", None)

            stored_prev = e.pop("prev", None)

            n_total += 1


            if stored_prev != prev:

                # Broken chain link — fail immediately, don't skip silently

                return False, n_valid, n_total


            payload    = json.dumps(e, sort_keys=True)

            chain_in   = (prev + payload).encode()

            computed   = hashlib.sha256(chain_in).hexdigest()


            if computed == stored_hash:

                n_valid += 1

                prev = stored_hash

            # Restore for next iteration

            e["hash"] = stored_hash

            e["prev"] = stored_prev


    return n_valid == n_total, n_valid, n_total



def run_assertions(audit_data: dict, strict: bool, quiet: bool) -> list[dict]:

    """Run all CI assertions. Returns list of results."""

    assertions = []

    results = audit_data["results"]


    def assert_(name, condition, detail, exit_code=1, warn_only=False):

        status = PASS if condition else (WARN if warn_only else FAIL)

        assertions.append({

            "name":      name,

            "passed":    condition,

            "warn_only": warn_only,

            "status":    status,

            "detail":    detail,

            "exit_code": exit_code if not condition and not warn_only else 0,

        })

        log(f"  {status}  {name}", quiet)

        if not condition:

            log(f"     → {detail}", quiet)

        return condition


    # 1. At least one scheduler ran

    assert_(

        "Adaptive loop produced results",

        len(results) > 0,

        f"No adaptive_loop_complete events found in audit log",

        exit_code=1

    )


    if not results:

        return assertions


    for scheduler, result in results.items():

        final_k    = result.get("final_k_us", 0)

        final_score= result.get("final_score", 0.0)

        converged  = result.get("converged", False)

        n_iters    = result.get("iterations", 0)


        # 2. Convergence

        assert_(

            f"{scheduler}: K-resonance ≥ {CONVERGENCE_THRESHOLD:.2f} (0.8×K_MAX)",

            final_score >= CONVERGENCE_THRESHOLD,

            f"score={final_score:.4f} < threshold={CONVERGENCE_THRESHOLD:.4f}",

            exit_code=1

        )


        # 3. Converged flag

        assert_(

            f"{scheduler}: converged=True",

            converged,

            f"Loop did not set converged=True (score={final_score:.4f})",

            exit_code=1

        )


        # 4. K_US hypothesis (main scientific claim)

        k_match = abs(final_k - K_US) <= K_TOLERANCE_US

        assert_(

            f"{scheduler}: final K = {K_US}µs ± {K_TOLERANCE_US}µs  "

            f"(φ·π·e·ms hypothesis)",

            k_match,

            f"final K={final_k}µs  expected={K_US}µs  "

            f"Δ={abs(final_k-K_US)}µs > tolerance={K_TOLERANCE_US}µs\n"

            f"     Note: natural period may differ on real hardware.",

            exit_code=2,

            warn_only=not strict   # warn in non-strict, fail in strict

        )


        # 5. Iterations within expected range

        assert_(

            f"{scheduler}: iterations ∈ [1, 15]",

            1 <= n_iters <= 15,

            f"iterations={n_iters} outside expected range",

            exit_code=1,

            warn_only=True

        )


        # 6. Score within valid range [0, K_MAX]

        assert_(

            f"{scheduler}: score ∈ [0, K_MAX={K_MAX:.2f}]",

            0 <= final_score <= K_MAX,

            f"score={final_score:.4f} outside [0, {K_MAX:.4f}]",

            exit_code=1

        )


        # 7. Monotonic convergence — detects regressions in adaptive logic

        sched_iters = audit_data["iters"].get(scheduler, [])

        if len(sched_iters) >= 2:

            scores_seq = [it["k_score"] for it in sched_iters]

            is_monotonic = all(x <= y + 0.01   # small tolerance for float noise

                               for x, y in zip(scores_seq, scores_seq[1:]))

            assert_(

                f"{scheduler}: monotonic convergence (scores non-decreasing)",

                is_monotonic,

                f"Non-monotonic scores: {[round(s,3) for s in scores_seq]}",

                exit_code=1,

                warn_only=True   # warn — real hardware may have noise

            )


    return assertions



def main():

    parser = argparse.ArgumentParser(

        description="RyokoSeven Scheduler CI Test")

    parser.add_argument("--strict",  action="store_true",

                        help="Fail on K_US hypothesis mismatch (default: warn)")

    parser.add_argument("--quiet",   action="store_true",

                        help="Minimal output for CI logs")

    parser.add_argument("--no-audit-check", action="store_true",

                        help="Skip SHA-256 audit chain verification")

    args = parser.parse_args()


    log(f"\nRyokoSeven Scheduler CI", args.quiet)

    log(f"K = φ·π·e = {K_MAX:.4f}   K_US = {K_US}µs", args.quiet)

    log(f"Convergence threshold: {CONVERGENCE_THRESHOLD:.4f}  "

        f"K tolerance: ±{K_TOLERANCE_US}µs\n", args.quiet)


    exit_code = 0


    with tempfile.TemporaryDirectory() as tmpdir:

        output_dir = Path(tmpdir)

        (output_dir / "audit_logs").mkdir()


        # ── Run demo ──────────────────────────────────────────

        log("── Step 1: Run demo", args.quiet)

        rc, audit_path = run_demo(output_dir, args.quiet)


        if rc != 0:

            log(f"\n{FAIL} Demo run failed (exit {rc})", args.quiet)

            sys.exit(1)

        log(f"  {PASS} Demo completed\n", args.quiet)


        # ── Audit chain ───────────────────────────────────────

        if not args.no_audit_check and audit_path.exists():

            log("── Step 2: Audit chain integrity", args.quiet)

            valid, n_valid, n_total = verify_audit_chain(audit_path)

            log(f"  {'✓' if valid else '✗'} "

                f"{n_valid}/{n_total} entries valid", args.quiet)

            if not valid:

                log(f"\n{FAIL} Audit chain corrupted", args.quiet)

                sys.exit(3)

            log("", args.quiet)


        # ── Parse results ─────────────────────────────────────

        log("── Step 3: Assertions", args.quiet)

        if not audit_path.exists():

            log(f"  {FAIL} Audit log not found: {audit_path}", args.quiet)

            sys.exit(1)


        audit_data = parse_audit(audit_path)

        assertions = run_assertions(audit_data, args.strict, args.quiet)


    # ── Summary ───────────────────────────────────────────────

    passed   = [a for a in assertions if a["passed"]]

    failed   = [a for a in assertions if not a["passed"] and not a["warn_only"]]

    warnings = [a for a in assertions if not a["passed"] and a["warn_only"]]


    log(f"\n── CI Summary", args.quiet)

    log(f"  {PASS} Passed:   {len(passed)}", args.quiet)

    log(f"  {WARN} Warnings: {len(warnings)}", args.quiet)

    log(f"  {FAIL} Failed:   {len(failed)}", args.quiet)


    if failed:

        exit_code = max(a["exit_code"] for a in failed)

        log(f"\n{FAIL} CI FAILED (exit {exit_code})", args.quiet)

    else:

        log(f"\n{PASS} CI PASSED", args.quiet)

        if warnings:

            log(f"  ({len(warnings)} warnings — "

                f"use --strict to treat as failures)", args.quiet)


    log(f"\nK = φ·π·e = {K_MAX:.4f}  K_US = {K_US}µs", args.quiet)

    sys.exit(exit_code)



if __name__ == "__main__":

    main()